# Evaluación Sumativa 3 — Fase 4
## Modelamiento integrado de `RainTomorrow` en *Weather Australia*

**Curso:** MCDI501 — Estadística Computacional para la Toma de Decisiones  
**Grupo 6:** Eduardo Contreras, Gonzalo Bouldres y Luis Díaz Giral  
**Docente:** Dr. Jean Paul Maidana González  
**Semilla reproducible:** `5012026`

El notebook integra los resultados de S1 y S2 en un flujo reproducible de imputación, clasificación, diagnóstico y evaluación de estabilidad. Antes del modelamiento se cierran computacionalmente las observaciones formuladas en la retroalimentación de S2 mediante: profundización bootstrap con `Rainfall`, regeneración de la sensibilidad Monte Carlo, incorporación de un estimador agregado, reproducción del diagnóstico IQR y recálculo de las correlaciones de multicolinealidad sobre la base completa.

La metodología comprende:

- delimitación del conjunto analítico a las variables presentes en la matriz de correlaciones de S1;
- regresiones explícitas para las variables numéricas con faltantes retenidas;
- comparación simétrica de casos completos, imputación simple e imputación por regresión;
- división estratificada 70/30 antes de ajustar transformaciones o modelos de S3;
- tres modelos logísticos con criterios de selección diferenciados;
- evaluación predictiva con métricas sensibles a la clase minoritaria;
- selección del punto de operación mediante predicciones fuera de muestra internas;
- diagnóstico de multicolinealidad, calibración, dependencia por localidad y sensibilidad temporal;
- bootstrap de 10.000 réplicas para la estabilidad de los parámetros del modelo final.

`RainTomorrow` es la variable objetivo. Sus registros faltantes se excluyen y no se imputan. El conjunto de prueba permanece reservado durante la preparación, selección y ajuste desarrollados en S3. Debido a que S1 y S2 analizaron previamente la base completa, esta partición no se interpreta como un holdout completamente independiente de las fases exploratorias anteriores; su finalidad es evaluar el flujo de modelamiento de S3 sin reutilizar el test en sus decisiones internas.

In [1]:
from pathlib import Path
from itertools import combinations
import hashlib
import math
import platform
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import scipy
from scipy import stats
from scipy.stats import chi2, chi2_contingency, wasserstein_distance

import sklearn
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, balanced_accuracy_score,
    brier_score_loss, confusion_matrix, f1_score, mean_absolute_error,
    mean_squared_error, precision_score, recall_score, roc_auc_score,
    roc_curve
)
from sklearn.model_selection import KFold, StratifiedKFold, train_test_split
from sklearn.exceptions import ConvergenceWarning
from sklearn.preprocessing import StandardScaler, OneHotEncoder

import statsmodels
import statsmodels.api as sm
from statsmodels.stats.diagnostic import het_breuschpagan, linear_reset
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import jarque_bera, durbin_watson
from IPython.display import display, Markdown

SEED = 5012026
TEST_SIZE = 0.30
ALPHA = 0.05
N_CV = 5
N_NESTED_CV = 5
N_IMPUTATION_CYCLES = 5
N_BOOT = 10_000
N_BOOT_RAINFALL = 10_000
N_MC_S2_CLOSURE = 100_000
MC_PORTFOLIO_SIZE = 1_000
N_MC_PORTFOLIO_REPLICATES = 10_000
MIN_BOOTSTRAP_SUCCESS = 0.95
BOOTSTRAP_SEED = SEED + 101
RAINFALL_BOOTSTRAP_SEED = SEED + 211
MC_S2_CLOSURE_SEED = SEED + 777

pd.set_option('display.max_columns', 150)
pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda value: f'{value:,.6f}')


def find_repo_root(start=None):
    current = Path.cwd().resolve() if start is None else Path(start).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'semana1').is_dir() and (candidate / 'semana2').is_dir():
            return candidate
    raise FileNotFoundError(
        'No se encontró la raíz del repositorio. Abra el notebook desde '
        'semana3/notebooks o defina ROOT manualmente.'
    )


ROOT = find_repo_root()
S3 = ROOT / 'semana3'
TABLES = S3 / 'docs' / 'tables'
FIGURES = S3 / 'docs' / 'figures'
PROCESSED = S3 / 'data' / 'processed'
for folder in (TABLES, FIGURES, PROCESSED):
    folder.mkdir(parents=True, exist_ok=True)

# La ejecución debe producir un conjunto de salidas completamente nuevo.
# Solo se eliminan artefactos computacionales de S3; no se modifican informes,
# presentaciones ni insumos de S1/S2.
for old_output in [
    *TABLES.glob('*.csv'),
    *FIGURES.glob('*.png'),
    *PROCESSED.glob('*.csv'),
]:
    old_output.unlink()

ENVIRONMENT = pd.DataFrame([
    ['Python', platform.python_version()],
    ['numpy', np.__version__],
    ['pandas', pd.__version__],
    ['scipy', scipy.__version__],
    ['scikit-learn', sklearn.__version__],
    ['statsmodels', statsmodels.__version__],
    ['matplotlib', matplotlib.__version__],
], columns=['componente', 'version'])
EXPECTED_VERSIONS = {
    'Python': '3.12.4',
    'numpy': '1.26.4',
    'pandas': '2.2.2',
    'scipy': '1.13.1',
    'scikit-learn': '1.4.2',
    'statsmodels': '0.14.2',
    'matplotlib': '3.8.4',
}
ENVIRONMENT['version_esperada'] = ENVIRONMENT['componente'].map(EXPECTED_VERSIONS)
ENVIRONMENT['coincide_entorno_validado'] = (
    ENVIRONMENT['version'].astype(str) == ENVIRONMENT['version_esperada'].astype(str)
)
ENVIRONMENT.to_csv(TABLES / '00_entorno_ejecucion.csv', index=False)

if not ENVIRONMENT['coincide_entorno_validado'].all():
    mismatches = ENVIRONMENT.loc[
        ~ENVIRONMENT['coincide_entorno_validado'],
        ['componente', 'version', 'version_esperada']
    ]
    display(Markdown(
        '**Advertencia de reproducibilidad:** el entorno activo difiere del entorno '
        'validado. Revise `semana3/requirements.txt` antes de interpretar los resultados.'
    ))
    display(mismatches)

display(Markdown(
    f'**Configuración reproducible:** semilla `{SEED}`, prueba `{TEST_SIZE:.0%}`, '
    f'validación cruzada `{N_CV}` pliegues, validación anidada '
    f'`{N_NESTED_CV}` pliegues, bootstrap del modelo `{N_BOOT:,}` réplicas, '
    f'bootstrap de Rainfall `{N_BOOT_RAINFALL:,}` réplicas, Monte Carlo '
    f'`{N_MC_S2_CLOSURE:,}` iteraciones y '
    f'`{N_MC_PORTFOLIO_REPLICATES:,}` carteras simuladas de tamaño '
    f'`{MC_PORTFOLIO_SIZE:,}`. '
    'La selección e inferencia se realizan sin ponderación. La evaluación '
    'predictiva compara configuraciones con y sin ponderación y selecciona '
    'la alternativa mediante predicciones out-of-fold.'
))


**Configuración reproducible:** semilla `5012026`, prueba `30%`, validación cruzada `5` pliegues, validación anidada `5` pliegues, bootstrap del modelo `10,000` réplicas, bootstrap de Rainfall `10,000` réplicas, Monte Carlo `100,000` iteraciones y `10,000` carteras simuladas de tamaño `1,000`. La selección e inferencia se realizan sin ponderación. La evaluación predictiva compara configuraciones con y sin ponderación y selecciona la alternativa mediante predicciones out-of-fold.

## 1. Insumos y trazabilidad S1 → S2 → S3

La base original se utiliza porque las regresiones de imputación requieren variables auxiliares que no están presentes en la base reducida de S2. Las tablas de S1 y S2 se cargan como insumos verificables y cada decisión de S3 queda vinculada a esos resultados.

Antes de iniciar el tratamiento de faltantes y el modelamiento, se incorporan las observaciones formuladas en la retroalimentación de la Sumativa 2. El cierre se realiza mediante código reproducible y genera nuevas evidencias para: profundización bootstrap con una variable asimétrica, regeneración de la sensibilidad Monte Carlo, reproducción del diagnóstico IQR y recálculo de correlaciones de multicolinealidad sobre la base completa. Estas verificaciones tienen finalidad de trazabilidad retrospectiva y no intervienen en la selección predictiva realizada exclusivamente con entrenamiento.

In [2]:
raw_path = ROOT / 'semana1' / 'data' / 'raw' / 'weatherAUS.csv'
s1_missing_path = ROOT / 'semana1' / 'docs' / 'tables' / '03_auditoria_datos_faltantes.csv'
s1_corr_path = ROOT / 'semana1' / 'docs' / 'tables' / '06_matriz_correlacion_pearson.csv'
s1_descriptive_path = ROOT / 'semana1' / 'docs' / 'tables' / '05_estadistica_descriptiva_variables_clave.csv'
s1_boxplot_path = ROOT / 'semana1' / 'docs' / 'figures' / 'fig_04_boxplots_por_rain_tomorrow.png'
s2_stability_path = ROOT / 'semana2' / 'docs' / 'tables' / '04b_diagnostico_estabilidad_correlaciones.csv'
s2_collinearity_path = ROOT / 'semana2' / 'docs' / 'tables' / '04c_multicolinealidad_predictoras.csv'
s2_montecarlo_sensitivity_path = ROOT / 'semana2' / 'docs' / 'tables' / '05b_sensibilidad_montecarlo.csv'
s2_outliers_path = ROOT / 'semana2' / 'docs' / 'tables' / '08b_diagnostico_outliers_iqr.csv'
s2_robustness_path = ROOT / 'semana2' / 'docs' / 'tables' / '09b_sintesis_robustez.csv'
s2_inputs_path = ROOT / 'semana2' / 'docs' / 'tables' / '10_resultados_validados_para_s3.csv'

required_files = [
    raw_path, s1_missing_path, s1_corr_path, s1_descriptive_path,
    s1_boxplot_path, s2_stability_path,
    s2_collinearity_path, s2_montecarlo_sensitivity_path,
    s2_outliers_path, s2_robustness_path, s2_inputs_path
]
missing_files = [str(path) for path in required_files if not path.exists()]
if missing_files:
    raise FileNotFoundError(f'Faltan insumos obligatorios: {missing_files}')

raw = pd.read_csv(raw_path)
s1_missing = pd.read_csv(s1_missing_path)
s1_corr = pd.read_csv(s1_corr_path).set_index('variable')
s2_stability = pd.read_csv(s2_stability_path)
s2_collinearity = pd.read_csv(s2_collinearity_path)
s2_montecarlo_sensitivity = pd.read_csv(s2_montecarlo_sensitivity_path)
s2_outliers = pd.read_csv(s2_outliers_path)
s2_robustness = pd.read_csv(s2_robustness_path)
s2_inputs = pd.read_csv(s2_inputs_path)

raw['Date'] = pd.to_datetime(raw['Date'], errors='coerce')
raw['Year'] = raw['Date'].dt.year
raw['Month'] = raw['Date'].dt.month
raw['RainToday_bin'] = raw['RainToday'].map({'No': 0.0, 'Yes': 1.0})
raw['RainTomorrow_bin'] = raw['RainTomorrow'].map({'No': 0.0, 'Yes': 1.0})
raw = raw.reset_index(drop=False).rename(columns={'index': 'row_id'})

if raw[['Date', 'Location']].duplicated().any():
    raise ValueError('La combinación Date–Location no es única.')
if not raw['RainTomorrow_bin'].dropna().isin([0, 1]).all():
    raise ValueError('RainTomorrow contiene categorías no reconocidas.')

traceability = s2_inputs.copy()
traceability['fase_origen'] = 'S2'
traceability.to_csv(TABLES / '01_trazabilidad_S1_S2_hacia_S3.csv', index=False)

stable_predictors = s2_stability.loc[
    s2_stability['estable_95'].astype(bool),
    ['variable', 'correlacion_pearson_s1', 'ic_bootstrap_95_li',
     'ic_bootstrap_95_ls', 'diagnostico_correlacion']
].copy()
stable_predictors.to_csv(TABLES / '02_correlaciones_estables_S2.csv', index=False)

display(traceability)
display(stable_predictors)
display(s2_collinearity)
display(s2_robustness)

,tipo_insumo_s3,resultado_validado,evidencia_sumativa2,decision_para_s3,nivel_confianza,fase_origen
0,Parámetro robustamente estimado,Separación de Humidity3pm entre RainTomorrow Y...,"Diferencia bootstrap BCa 95 % [22.0584, 22.5317]",Mantener Humidity3pm como predictor prioritario,Alto,S2
1,Parámetro robustamente estimado,Media de Pressure3pm,"IC bootstrap BCa 95 % [1015.2181, 1015.2947]","Incluir como variable candidata, controlando r...",Medio-alto,S2
2,Parámetro robustamente estimado,Media de MaxTemp,"IC bootstrap BCa 95 % [23.1845, 23.2580]","Incluir como variable candidata, controlando r...",Medio,S2
3,Proporción validada,Proporción RainTomorrow = Yes,"Wilson S1 95 % [0.2220, 0.2264]; BCa 95 % [0.2...",Usar como prevalencia base y considerar desbal...,Alto,S2
4,Prueba de hipótesis validada,Prueba de hipótesis sobre Humidity3pm,"p permutación unilateral = 0.000100; p < 0,001",Usar la diferencia de humedad como relación es...,Alto,S2
5,Prueba complementaria documentada,Chi-cuadrado RainToday vs RainTomorrow,"chi2 = 13801.29; V de Cramer = 0.3131; p < 0,001",Incluir RainToday_bin como predictor categóric...,Alto,S2
6,Correlación estable,Correlación Humidity3pm con RainTomorrow_bin,r = 0.4462,Priorizar en modelos explicativos y predictivos,Alto,S2
7,Correlación estable,RainToday_bin como señal antecedente,r = 0.3131,Incluir como variable categórica binaria,Medio-alto,S2
8,Correlación estable,Rainfall como señal meteorológica antecedente,r = 0.2390,"Incluir como variable candidata, considerando ...",Medio,S2
9,Correlación estable,Pressure3pm con asociación inversa,r = -0.2260,"Incluir por interpretación meteorológica, revi...",Medio,S2


,variable,correlacion_pearson_s1,ic_bootstrap_95_li,ic_bootstrap_95_ls,diagnostico_correlacion
0,Humidity3pm,0.446160,0.441734,0.450599,Estable: el IC bootstrap 95 % no incluye cero ...
1,RainToday_bin,0.313097,0.307230,0.318769,Estable: el IC bootstrap 95 % no incluye cero ...
2,Rainfall,0.239032,0.231840,0.246107,Estable: el IC bootstrap 95 % no incluye cero ...
3,Pressure3pm,-0.226031,-0.231396,-0.220614,Estable: el IC bootstrap 95 % no incluye cero ...
4,MaxTemp,-0.159237,-0.164258,-0.154150,Estable: el IC bootstrap 95 % no incluye cero ...


,variable_1,variable_2,correlacion_pearson,abs_correlacion,n_pares_validos,fuente,lectura_s3
0,Pressure9am,Pressure3pm,0.960000,0.960000,NaN,Evidencia reportada en retroalimentación de Su...,Alta redundancia potencial; evaluar selección ...
1,Temp9am,Temp3pm,0.860000,0.860000,NaN,Evidencia reportada en retroalimentación de Su...,Alta redundancia potencial; evaluar selección ...


,dimension_evaluada,metodo,resultado,interpretacion
0,Valores extremos,Filtro IQR por grupo,Diferencia = 22.6907,El resultado se mantiene positivo y de magnitu...
1,Valores extremos,Winsorización 1 %-99 %,Diferencia = 22.3699,El resultado se mantiene positivo después de l...
2,Métrica del centro,Diferencia de medianas con bootstrap,Diferencia de medianas = 23.0000; IC 95 % [22....,La separación entre grupos persiste al usar un...
3,Subconjuntos influyentes,Exclusión de una localidad a la vez,Mayor variación: AliceSprings = -0.5548,La conclusión principal no depende de una únic...


### 1.1 Profundización bootstrap para una variable asimétrica

La comparación de intervalos de S2 se amplía con `Rainfall`, variable continua con asimetría positiva pronunciada. Se estiman 10.000 remuestras no paramétricas de tamaño completo y se comparan el intervalo clásico, el intervalo percentil y el intervalo BCa. Para la media, el remuestreo se implementa mediante conteos multinomiales sobre la distribución empírica observada; esta formulación es probabilísticamente equivalente al remuestreo individual con reemplazo y reduce el costo computacional sin modificar el estimador. El cálculo BCa incorpora el sesgo de la distribución bootstrap y la aceleración obtenida mediante valores jackknife. Debido al gran tamaño muestral, la asimetría de la variable no implica necesariamente una separación amplia entre los intervalos de la media; la comparación permite distinguir la forma de la variable original de la distribución muestral del estimador.

In [3]:
def bootstrap_mean_empirical(values, n_boot, rng, batch_size=1_000):
    values = np.asarray(values, dtype=float)
    if values.ndim != 1 or len(values) < 2:
        raise ValueError(
            'El bootstrap requiere un vector unidimensional con al menos dos observaciones.'
        )

    support, observed_counts = np.unique(values, return_counts=True)
    sample_size = len(values)
    empirical_probabilities = observed_counts / sample_size
    estimates = np.empty(n_boot, dtype=float)

    for start in range(0, n_boot, batch_size):
        stop = min(start + batch_size, n_boot)
        bootstrap_counts = rng.multinomial(
            sample_size,
            empirical_probabilities,
            size=stop - start
        )
        estimates[start:stop] = (
            bootstrap_counts @ support
        ) / sample_size

    return estimates


def bca_interval_from_jackknife(theta_hat, bootstrap_values, jackknife_values, alpha=0.05):
    bootstrap_values = np.asarray(bootstrap_values, dtype=float)
    jackknife_values = np.asarray(jackknife_values, dtype=float)
    proportion_less = (
        np.sum(bootstrap_values < theta_hat)
        + 0.5 * np.sum(bootstrap_values == theta_hat)
    ) / len(bootstrap_values)
    epsilon = 1 / (2 * len(bootstrap_values))
    proportion_less = np.clip(proportion_less, epsilon, 1 - epsilon)
    z0 = stats.norm.ppf(proportion_less)

    jackknife_mean = jackknife_values.mean()
    centered = jackknife_mean - jackknife_values
    denominator = 6 * np.power(np.sum(centered ** 2), 1.5)
    acceleration = (
        np.sum(centered ** 3) / denominator
        if denominator > 0 else 0.0
    )

    z_low = stats.norm.ppf(alpha / 2)
    z_high = stats.norm.ppf(1 - alpha / 2)
    adjusted = []
    for z_alpha in (z_low, z_high):
        numerator = z0 + z_alpha
        denominator_adjustment = 1 - acceleration * numerator
        adjusted_probability = stats.norm.cdf(
            z0 + numerator / denominator_adjustment
        )
        adjusted.append(float(np.clip(adjusted_probability, 0, 1)))

    return (
        float(np.quantile(bootstrap_values, adjusted[0])),
        float(np.quantile(bootstrap_values, adjusted[1])),
        float(z0),
        float(acceleration)
    )


rainfall_values = raw['Rainfall'].dropna().to_numpy(dtype=float)
rainfall_mean = float(rainfall_values.mean())
rainfall_sd = float(rainfall_values.std(ddof=1))
rainfall_skewness = float(stats.skew(rainfall_values, bias=False))
rainfall_rng = np.random.default_rng(RAINFALL_BOOTSTRAP_SEED)
rainfall_bootstrap = bootstrap_mean_empirical(
    rainfall_values,
    N_BOOT_RAINFALL,
    rainfall_rng,
    batch_size=1_000
)

rainfall_jackknife = (
    rainfall_values.sum() - rainfall_values
) / (len(rainfall_values) - 1)
rainfall_bca_li, rainfall_bca_ls, rainfall_z0, rainfall_acceleration = (
    bca_interval_from_jackknife(
        rainfall_mean,
        rainfall_bootstrap,
        rainfall_jackknife,
        alpha=ALPHA
    )
)
rainfall_percentile_li, rainfall_percentile_ls = np.quantile(
    rainfall_bootstrap,
    [ALPHA / 2, 1 - ALPHA / 2]
)
rainfall_standard_error = rainfall_sd / np.sqrt(len(rainfall_values))
rainfall_t_critical = stats.t.ppf(
    1 - ALPHA / 2,
    df=len(rainfall_values) - 1
)
rainfall_classic_li = rainfall_mean - rainfall_t_critical * rainfall_standard_error
rainfall_classic_ls = rainfall_mean + rainfall_t_critical * rainfall_standard_error

bootstrap_rainfall_summary = pd.DataFrame([{
    'parametro': 'Media de Rainfall',
    'n_valido': len(rainfall_values),
    'estimacion': rainfall_mean,
    'desviacion_estandar': rainfall_sd,
    'asimetria_Fisher_Pearson': rainfall_skewness,
    'n_remuestras': N_BOOT_RAINFALL,
    'semilla': RAINFALL_BOOTSTRAP_SEED,
    'ic_clasico_95_li': rainfall_classic_li,
    'ic_clasico_95_ls': rainfall_classic_ls,
    'ic_percentil_95_li': rainfall_percentile_li,
    'ic_percentil_95_ls': rainfall_percentile_ls,
    'ic_BCa_95_li': rainfall_bca_li,
    'ic_BCa_95_ls': rainfall_bca_ls,
    'ancho_ic_clasico': rainfall_classic_ls - rainfall_classic_li,
    'ancho_ic_percentil': rainfall_percentile_ls - rainfall_percentile_li,
    'ancho_ic_BCa': rainfall_bca_ls - rainfall_bca_li,
    'sesgo_z0_BCa': rainfall_z0,
    'aceleracion_BCa': rainfall_acceleration,
    'diferencia_centro_BCa_vs_clasico': (
        (rainfall_bca_li + rainfall_bca_ls) / 2
        - (rainfall_classic_li + rainfall_classic_ls) / 2
    ),
    'interpretacion': (
        'Rainfall presenta asimetría positiva pronunciada. La proximidad entre los '
        'intervalos de la media se interpreta por el gran tamaño muestral y no como '
        'evidencia de simetría de la variable original.'
    )
}])
bootstrap_rainfall_summary.to_csv(
    TABLES / '02a_bootstrap_Rainfall_asimetria.csv',
    index=False
)

fig, ax = plt.subplots(figsize=(8, 4.8))
ax.hist(rainfall_bootstrap, bins=50, edgecolor='black', alpha=0.75)
ax.axvline(rainfall_mean, linestyle='-', linewidth=1.8, label='Media observada')
ax.axvline(rainfall_bca_li, linestyle='--', linewidth=1.4, label='Límites BCa 95 %')
ax.axvline(rainfall_bca_ls, linestyle='--', linewidth=1.4)
ax.set_xlabel('Media bootstrap de Rainfall')
ax.set_ylabel('Frecuencia')
ax.set_title('Distribución bootstrap de la media de Rainfall')
ax.legend()
fig.tight_layout()
fig.savefig(
    FIGURES / 'fig_00a_bootstrap_media_Rainfall.png',
    dpi=220
)
plt.close(fig)

display(bootstrap_rainfall_summary)

,parametro,n_valido,estimacion,desviacion_estandar,asimetria_Fisher_Pearson,n_remuestras,semilla,ic_clasico_95_li,ic_clasico_95_ls,ic_percentil_95_li,ic_percentil_95_ls,ic_BCa_95_li,ic_BCa_95_ls,ancho_ic_clasico,ancho_ic_percentil,ancho_ic_BCa,sesgo_z0_BCa,aceleracion_BCa,diferencia_centro_BCa_vs_clasico,interpretacion
0,Media de Rainfall,142199,2.360918,8.478060,9.836225,10000,5012237,2.316853,2.404984,2.316793,2.404124,2.317683,2.404878,0.088131,0.087331,0.087194,0.014038,0.004347,0.000362,Rainfall presenta asimetría positiva pronuncia...


### 1.2 Regeneración de sensibilidad Monte Carlo y estimador agregado

La simulación se reconstruye dentro del notebook maestro con los parámetros validados en S1. Las medias y desviaciones estándar de `Humidity3pm` se estiman mediante casos observados por grupo, mientras que la prevalencia de `RainTomorrow = Yes` se calcula sobre la totalidad de observaciones con variable objetivo válida, sin condicionarla a la disponibilidad de `Humidity3pm`. Esta separación reproduce el parámetro de prevalencia validado en S1 y evita un sesgo por casos completos.

Se evalúa el escenario base y ocho perturbaciones relativas de ±5 % mediante 100.000 iteraciones y números aleatorios comunes. El delta individual se conserva para mantener la trazabilidad con S2, pero la interpretación operacional se basa además en sensibilidad, especificidad, valores predictivos, tasas esperadas por cada 1.000 observaciones y una distribución muestral de 10.000 carteras de tamaño 1.000. El umbral `Humidity3pm ≥ 60 %` es meteorológico y no debe confundirse con el umbral probabilístico del clasificador desarrollado posteriormente.

In [4]:
mc_humidity_source = raw.loc[
    raw['RainTomorrow_bin'].notna() & raw['Humidity3pm'].notna(),
    ['Humidity3pm', 'RainTomorrow_bin']
].copy()
mc_target_source = raw.loc[
    raw['RainTomorrow_bin'].notna(),
    'RainTomorrow_bin'
].astype(float)

mc_yes = mc_humidity_source.loc[
    mc_humidity_source['RainTomorrow_bin'].eq(1),
    'Humidity3pm'
].to_numpy(dtype=float)
mc_no = mc_humidity_source.loc[
    mc_humidity_source['RainTomorrow_bin'].eq(0),
    'Humidity3pm'
].to_numpy(dtype=float)

s2_mc_base_reference = s2_montecarlo_sensitivity.loc[
    s2_montecarlo_sensitivity['escenario'].eq('Base')
].copy()
if len(s2_mc_base_reference) != 1:
    raise ValueError(
        'La evidencia de S2 debe contener exactamente un escenario Monte Carlo base.'
    )
s2_mc_base_reference = s2_mc_base_reference.iloc[0]

mc_parameters_base = {
    'media_yes': float(mc_yes.mean()),
    'media_no': float(mc_no.mean()),
    'sd_yes': float(mc_yes.std(ddof=1)),
    'sd_no': float(mc_no.std(ddof=1)),
    'proporcion_yes': float(mc_target_source.mean())
}

mc_parameter_reference_columns = {
    'media_yes': 'media_yes_usada',
    'media_no': 'media_no_usada',
    'sd_yes': 'sd_yes_usada',
    'sd_no': 'sd_no_usada',
    'proporcion_yes': 'proporcion_yes_usada'
}
mc_parameter_sample_sizes = {
    'media_yes': len(mc_yes),
    'media_no': len(mc_no),
    'sd_yes': len(mc_yes),
    'sd_no': len(mc_no),
    'proporcion_yes': len(mc_target_source)
}
mc_parameter_sources = {
    'media_yes': 'S1: Humidity3pm observada en RainTomorrow = Yes',
    'media_no': 'S1: Humidity3pm observada en RainTomorrow = No',
    'sd_yes': 'S1: Humidity3pm observada en RainTomorrow = Yes',
    'sd_no': 'S1: Humidity3pm observada en RainTomorrow = No',
    'proporcion_yes': 'S1: todas las observaciones con RainTomorrow válido'
}

mc_parameter_traceability_rows = []
for parameter, estimated_value in mc_parameters_base.items():
    reference_column = mc_parameter_reference_columns[parameter]
    reference_value = float(s2_mc_base_reference[reference_column])
    absolute_difference = abs(estimated_value - reference_value)
    mc_parameter_traceability_rows.append({
        'parametro': parameter,
        'valor_recalculado_S3_desde_S1': estimated_value,
        'valor_referencia_S2': reference_value,
        'diferencia_absoluta': absolute_difference,
        'n_observaciones_fuente': mc_parameter_sample_sizes[parameter],
        'fuente_calculo': mc_parameter_sources[parameter],
        'coincide_referencia_S2': absolute_difference <= 1e-10
    })

mc_parameter_traceability = pd.DataFrame(mc_parameter_traceability_rows)
mc_parameter_traceability.to_csv(
    TABLES / '02b0_parametros_base_montecarlo_S1.csv',
    index=False
)
if not mc_parameter_traceability['coincide_referencia_S2'].all():
    inconsistent_parameters = mc_parameter_traceability.loc[
        ~mc_parameter_traceability['coincide_referencia_S2'],
        'parametro'
    ].tolist()
    raise AssertionError(
        'Los parámetros Monte Carlo recalculados desde S1 no reproducen la '
        f'evidencia validada de S2: {inconsistent_parameters}'
    )
if not np.isclose(
    mc_parameters_base['proporcion_yes'],
    0.22418121848473554,
    rtol=0,
    atol=1e-12
):
    raise AssertionError(
        'La prevalencia Monte Carlo debe calcularse sobre las 142.193 '
        'observaciones con RainTomorrow válido y reproducir el parámetro S1.'
    )

mc_scenarios = [
    ('Base', 'Sin modificación', 0.00, {}),
    ('Media Yes +5%', 'media_humidity3pm_yes', 0.05, {'media_yes': 1.05}),
    ('Media Yes -5%', 'media_humidity3pm_yes', -0.05, {'media_yes': 0.95}),
    ('Media No +5%', 'media_humidity3pm_no', 0.05, {'media_no': 1.05}),
    ('Media No -5%', 'media_humidity3pm_no', -0.05, {'media_no': 0.95}),
    ('SD Yes +5%', 'sd_humidity3pm_yes', 0.05, {'sd_yes': 1.05}),
    ('SD Yes -5%', 'sd_humidity3pm_yes', -0.05, {'sd_yes': 0.95}),
    ('Proporción Yes +5%', 'proporcion_raintomorrow_yes', 0.05, {'proporcion_yes': 1.05}),
    ('Proporción Yes -5%', 'proporcion_raintomorrow_yes', -0.05, {'proporcion_yes': 0.95})
]

mc_rng = np.random.default_rng(MC_S2_CLOSURE_SEED)
mc_uniform_event = mc_rng.random(N_MC_S2_CLOSURE)
mc_z_yes = mc_rng.standard_normal(N_MC_S2_CLOSURE)
mc_z_no = mc_rng.standard_normal(N_MC_S2_CLOSURE)
mc_z_delta_yes = mc_rng.standard_normal(N_MC_S2_CLOSURE)
mc_z_delta_no = mc_rng.standard_normal(N_MC_S2_CLOSURE)

if N_MC_PORTFOLIO_REPLICATES < 10_000:
    raise ValueError(
        'La distribución agregada requiere al menos 10.000 carteras simuladas.'
    )
N_MC_PORTFOLIOS = N_MC_PORTFOLIO_REPLICATES


def safe_ratio(numerator, denominator):
    return numerator / denominator if denominator else np.nan


def simulate_mc_scenario(parameter_multipliers, portfolio_seed):
    parameters = mc_parameters_base.copy()
    for parameter, multiplier in parameter_multipliers.items():
        parameters[parameter] *= multiplier
    parameters['proporcion_yes'] = float(np.clip(
        parameters['proporcion_yes'],
        1e-6,
        1 - 1e-6
    ))

    rain_event = mc_uniform_event < parameters['proporcion_yes']
    humidity_yes = np.clip(
        parameters['media_yes'] + parameters['sd_yes'] * mc_z_yes,
        0,
        100
    )
    humidity_no = np.clip(
        parameters['media_no'] + parameters['sd_no'] * mc_z_no,
        0,
        100
    )
    humidity_simulated = np.where(rain_event, humidity_yes, humidity_no)
    alert = humidity_simulated >= 60

    delta_individual = (
        np.clip(
            parameters['media_yes'] + parameters['sd_yes'] * mc_z_delta_yes,
            0,
            100
        )
        - np.clip(
            parameters['media_no'] + parameters['sd_no'] * mc_z_delta_no,
            0,
            100
        )
    )

    tp = int(np.sum(alert & rain_event))
    fp = int(np.sum(alert & ~rain_event))
    tn = int(np.sum(~alert & ~rain_event))
    fn = int(np.sum(~alert & rain_event))

    # Distribución muestral de carteras independientes de tamaño fijo.
    # Las probabilidades de las cuatro celdas de la matriz de confusión se
    # estiman con las 100.000 iteraciones del escenario y se propagan mediante
    # 10.000 realizaciones multinomiales de cartera.
    confusion_probabilities = np.asarray(
        [tp, fp, tn, fn],
        dtype=float
    ) / N_MC_S2_CLOSURE
    portfolio_rng = np.random.default_rng(portfolio_seed)
    portfolio_counts = portfolio_rng.multinomial(
        MC_PORTFOLIO_SIZE,
        confusion_probabilities,
        size=N_MC_PORTFOLIOS
    )
    portfolio_tp = portfolio_counts[:, 0]
    portfolio_fp = portfolio_counts[:, 1]
    portfolio_fn = portfolio_counts[:, 3]
    portfolio_vpp = np.divide(
        portfolio_tp,
        portfolio_tp + portfolio_fp,
        out=np.full(N_MC_PORTFOLIOS, np.nan, dtype=float),
        where=(portfolio_tp + portfolio_fp) > 0
    )
    portfolio_sensitivity = np.divide(
        portfolio_tp,
        portfolio_tp + portfolio_fn,
        out=np.full(N_MC_PORTFOLIOS, np.nan, dtype=float),
        where=(portfolio_tp + portfolio_fn) > 0
    )

    return {
        'parameters': parameters,
        'rain_event': rain_event,
        'alert': alert,
        'delta_individual': delta_individual,
        'tp': tp,
        'fp': fp,
        'tn': tn,
        'fn': fn,
        'portfolio_vpp': portfolio_vpp,
        'portfolio_sensitivity': portfolio_sensitivity
    }


mc_records = []
mc_base_arrays = None
for scenario_index, (
    scenario,
    modified_parameter,
    relative_change,
    multipliers
) in enumerate(mc_scenarios):
    simulation = simulate_mc_scenario(
        multipliers,
        portfolio_seed=MC_S2_CLOSURE_SEED + 10_000 + scenario_index
    )
    parameters = simulation['parameters']
    rain_event = simulation['rain_event']
    alert = simulation['alert']
    delta_individual = simulation['delta_individual']
    tp, fp, tn, fn = (
        simulation['tp'], simulation['fp'], simulation['tn'], simulation['fn']
    )
    if scenario == 'Base':
        mc_base_arrays = simulation

    mc_records.append({
        'escenario': scenario,
        'parametro_modificado': modified_parameter,
        'cambio_relativo': relative_change,
        'n_iteraciones': N_MC_S2_CLOSURE,
        'semilla': MC_S2_CLOSURE_SEED,
        'media_yes_usada': parameters['media_yes'],
        'media_no_usada': parameters['media_no'],
        'sd_yes_usada': parameters['sd_yes'],
        'sd_no_usada': parameters['sd_no'],
        'proporcion_yes_usada': parameters['proporcion_yes'],
        'n_objetivo_valido_para_prevalencia': len(mc_target_source),
        'n_humidity_observada_yes': len(mc_yes),
        'n_humidity_observada_no': len(mc_no),
        'fuente_prevalencia': (
            'S1: todas las observaciones con RainTomorrow válido'
        ),
        'media_delta_individual_yes_no': float(delta_individual.mean()),
        'probabilidad_delta_individual_mayor_0': float(np.mean(delta_individual > 0)),
        'sensibilidad_umbral_60': safe_ratio(tp, tp + fn),
        'especificidad_umbral_60': safe_ratio(tn, tn + fp),
        'vpp_umbral_60': safe_ratio(tp, tp + fp),
        'vpn_umbral_60': safe_ratio(tn, tn + fn),
        'proporcion_alertas_umbral_60': float(alert.mean()),
        'eventos_lluvia_esperados_por_1000': 1000 * float(rain_event.mean()),
        'alertas_esperadas_por_1000': 1000 * float(alert.mean()),
        'verdaderos_positivos_esperados_por_1000': 1000 * tp / N_MC_S2_CLOSURE,
        'falsos_positivos_esperados_por_1000': 1000 * fp / N_MC_S2_CLOSURE,
        'eventos_no_detectados_esperados_por_1000': 1000 * fn / N_MC_S2_CLOSURE,
        'n_carteras_simuladas': N_MC_PORTFOLIOS,
        'tamano_cartera': MC_PORTFOLIO_SIZE,
        'vpp_cartera_media': float(np.nanmean(simulation['portfolio_vpp'])),
        'vpp_cartera_ic_empirico_95_li': float(np.nanquantile(simulation['portfolio_vpp'], 0.025)),
        'vpp_cartera_ic_empirico_95_ls': float(np.nanquantile(simulation['portfolio_vpp'], 0.975)),
        'sensibilidad_cartera_media': float(np.nanmean(simulation['portfolio_sensitivity'])),
        'sensibilidad_cartera_ic_empirico_95_li': float(np.nanquantile(simulation['portfolio_sensitivity'], 0.025)),
        'sensibilidad_cartera_ic_empirico_95_ls': float(np.nanquantile(simulation['portfolio_sensitivity'], 0.975))
    })

montecarlo_sensitivity_regenerated = pd.DataFrame(mc_records)
base_mc_row = montecarlo_sensitivity_regenerated.loc[
    montecarlo_sensitivity_regenerated['escenario'].eq('Base')
].iloc[0]
montecarlo_sensitivity_regenerated['variacion_media_delta_vs_base'] = (
    montecarlo_sensitivity_regenerated['media_delta_individual_yes_no']
    - base_mc_row['media_delta_individual_yes_no']
)
montecarlo_sensitivity_regenerated['variacion_vpp_umbral_60_vs_base'] = (
    montecarlo_sensitivity_regenerated['vpp_umbral_60']
    - base_mc_row['vpp_umbral_60']
)
montecarlo_sensitivity_regenerated.to_csv(
    TABLES / '02b_sensibilidad_montecarlo_regenerada.csv',
    index=False
)

parameter_comparison_columns = [
    'media_yes_usada',
    'media_no_usada',
    'sd_yes_usada',
    'sd_no_usada',
    'proporcion_yes_usada'
]
result_comparison_columns = [
    'media_delta_individual_yes_no',
    'sensibilidad_umbral_60',
    'especificidad_umbral_60',
    'vpp_umbral_60',
    'proporcion_alertas_umbral_60'
]
comparison_columns = parameter_comparison_columns + result_comparison_columns
mc_consistency = montecarlo_sensitivity_regenerated.merge(
    s2_montecarlo_sensitivity[['escenario', *comparison_columns]],
    on='escenario',
    how='left',
    suffixes=('_regenerado_S3', '_archivo_S2')
)
for variable in comparison_columns:
    mc_consistency[f'diferencia_absoluta_{variable}'] = (
        mc_consistency[f'{variable}_regenerado_S3']
        - mc_consistency[f'{variable}_archivo_S2']
    ).abs()
mc_consistency['consistencia_parametros_S1_S2'] = (
    mc_consistency[
        [f'diferencia_absoluta_{variable}' for variable in parameter_comparison_columns]
    ].le(1e-10).all(axis=1)
)
mc_consistency['consistencia_resultados_simulados'] = (
    mc_consistency['diferencia_absoluta_media_delta_individual_yes_no'].le(0.35)
    & mc_consistency['diferencia_absoluta_sensibilidad_umbral_60'].le(0.015)
    & mc_consistency['diferencia_absoluta_especificidad_umbral_60'].le(0.015)
    & mc_consistency['diferencia_absoluta_vpp_umbral_60'].le(0.015)
    & mc_consistency['diferencia_absoluta_proporcion_alertas_umbral_60'].le(0.015)
)
mc_consistency['consistencia_numerica'] = (
    mc_consistency['consistencia_parametros_S1_S2']
    & mc_consistency['consistencia_resultados_simulados']
)
mc_consistency.to_csv(
    TABLES / '02c_consistencia_sensibilidad_montecarlo_S2_S3.csv',
    index=False
)

base_delta = mc_base_arrays['delta_individual']
base_event = mc_base_arrays['rain_event']
base_alert = mc_base_arrays['alert']
base_tp_cumulative = np.cumsum(base_event & base_alert)
base_alert_cumulative = np.cumsum(base_alert)
checkpoints = np.unique(np.r_[
    np.arange(1_000, 10_001, 1_000),
    np.arange(20_000, N_MC_S2_CLOSURE + 1, 10_000)
])
mc_convergence = pd.DataFrame({
    'iteracion': checkpoints,
    'media_acumulada_delta_individual': [
        float(base_delta[:checkpoint].mean()) for checkpoint in checkpoints
    ],
    'vpp_acumulado_umbral_60': [
        safe_ratio(
            int(base_tp_cumulative[checkpoint - 1]),
            int(base_alert_cumulative[checkpoint - 1])
        ) for checkpoint in checkpoints
    ]
})
mc_convergence['desviacion_media_delta_respecto_final'] = (
    mc_convergence['media_acumulada_delta_individual']
    - float(base_delta.mean())
)
mc_convergence['desviacion_vpp_respecto_final'] = (
    mc_convergence['vpp_acumulado_umbral_60']
    - float(base_mc_row['vpp_umbral_60'])
)
mc_convergence.to_csv(
    TABLES / '02d_convergencia_montecarlo_regenerada.csv',
    index=False
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
plot_data = montecarlo_sensitivity_regenerated.loc[
    ~montecarlo_sensitivity_regenerated['escenario'].eq('Base')
].copy()
axes[0].barh(
    plot_data['escenario'],
    plot_data['variacion_media_delta_vs_base']
)
axes[0].axvline(0, linestyle='--', linewidth=1.2)
axes[0].set_xlabel('Variación de la media del delta frente al escenario base')
axes[0].set_title('Sensibilidad del delta individual')
axes[1].barh(
    plot_data['escenario'],
    plot_data['variacion_vpp_umbral_60_vs_base']
)
axes[1].axvline(0, linestyle='--', linewidth=1.2)
axes[1].set_xlabel('Variación del VPP frente al escenario base')
axes[1].set_title('Sensibilidad del valor predictivo positivo')
fig.tight_layout()
fig.savefig(
    FIGURES / 'fig_00b_sensibilidad_montecarlo_regenerada.png',
    dpi=220
)
plt.close(fig)

display(mc_parameter_traceability)
display(montecarlo_sensitivity_regenerated)
display(mc_consistency)

,parametro,valor_recalculado_S3_desde_S1,valor_referencia_S2,diferencia_absoluta,n_observaciones_fuente,fuente_calculo,coincide_referencia_S2
0,media_yes,68.800019,68.800019,0.000000,30913,S1: Humidity3pm observada en RainTomorrow = Yes,True
1,media_no,46.510625,46.510625,0.000000,107670,S1: Humidity3pm observada en RainTomorrow = No,True
2,sd_yes,19.037409,19.037409,0.000000,30913,S1: Humidity3pm observada en RainTomorrow = Yes,True
3,sd_no,18.489476,18.489476,0.000000,107670,S1: Humidity3pm observada en RainTomorrow = No,True
4,proporcion_yes,0.224181,0.224181,0.000000,142193,S1: todas las observaciones con RainTomorrow v...,True


,escenario,parametro_modificado,cambio_relativo,n_iteraciones,semilla,media_yes_usada,media_no_usada,sd_yes_usada,sd_no_usada,proporcion_yes_usada,n_objetivo_valido_para_prevalencia,n_humidity_observada_yes,n_humidity_observada_no,fuente_prevalencia,media_delta_individual_yes_no,probabilidad_delta_individual_mayor_0,sensibilidad_umbral_60,especificidad_umbral_60,vpp_umbral_60,vpn_umbral_60,proporcion_alertas_umbral_60,eventos_lluvia_esperados_por_1000,alertas_esperadas_por_1000,verdaderos_positivos_esperados_por_1000,falsos_positivos_esperados_por_1000,eventos_no_detectados_esperados_por_1000,n_carteras_simuladas,tamano_cartera,vpp_cartera_media,vpp_cartera_ic_empirico_95_li,vpp_cartera_ic_empirico_95_ls,sensibilidad_cartera_media,sensibilidad_cartera_ic_empirico_95_li,sensibilidad_cartera_ic_empirico_95_ls,variacion_media_delta_vs_base,variacion_vpp_umbral_60_vs_base
0,Base,Sin modificación,0.000000,100000,5012803,68.800019,46.510625,19.037409,18.489476,0.224181,142193,30913,107670,S1: todas las observaciones con RainTomorrow v...,21.783047,0.799000,0.676657,0.765859,0.456531,0.890693,0.333800,225.210000,333.800000,152.390000,181.410000,72.820000,10000,1000,0.456708,0.403846,0.509092,0.676589,0.615385,0.736842,0.000000,0.000000
1,Media Yes +5%,media_humidity3pm_yes,0.050000,100000,5012803,72.240020,46.510625,19.037409,18.489476,0.224181,142193,30913,107670,S1: todas las observaciones con RainTomorrow v...,25.014025,0.833320,0.740065,0.765859,0.478827,0.910204,0.348080,225.210000,348.080000,166.670000,181.410000,58.540000,10000,1000,0.478523,0.425146,0.530615,0.740243,0.680849,0.797875,3.230978,0.022296
2,Media Yes -5%,media_humidity3pm_yes,-0.050000,100000,5012803,65.360018,46.510625,19.037409,18.489476,0.224181,142193,30913,107670,S1: todas las observaciones con RainTomorrow v...,18.487787,0.761300,0.612051,0.765859,0.431762,0.871656,0.319250,225.210000,319.250000,137.840000,181.410000,87.370000,10000,1000,0.431567,0.375394,0.486239,0.612171,0.547738,0.676190,-3.295260,-0.024769
3,Media No +5%,media_humidity3pm_no,0.050000,100000,5012803,68.800019,48.836156,19.037409,18.489476,0.224181,142193,30913,107670,S1: todas las observaciones con RainTomorrow v...,19.474798,0.773760,0.676657,0.725990,0.417862,0.885379,0.364690,225.210000,364.690000,152.390000,212.300000,72.820000,10000,1000,0.417883,0.368269,0.468495,0.676432,0.615385,0.737971,-2.308248,-0.038669
4,Media No -5%,media_humidity3pm_no,-0.050000,100000,5012803,68.800019,44.185094,19.037409,18.489476,0.224181,142193,30913,107670,S1: todas las observaciones con RainTomorrow v...,24.087659,0.823040,0.676657,0.802179,0.498560,0.895123,0.305660,225.210000,305.660000,152.390000,153.270000,72.820000,10000,1000,0.498729,0.441859,0.555556,0.676868,0.614106,0.735294,2.304612,0.042030
5,SD Yes +5%,sd_humidity3pm_yes,0.050000,100000,5012803,68.800019,46.510625,19.989279,18.489476,0.224181,142193,30913,107670,S1: todas las observaciones con RainTomorrow v...,21.675909,0.792830,0.668443,0.765859,0.453502,0.888227,0.331950,225.210000,331.950000,150.540000,181.410000,74.670000,10000,1000,0.453768,0.400611,0.507042,0.668790,0.606635,0.730088,-0.107138,-0.003029
6,SD Yes -5%,sd_humidity3pm_yes,-0.050000,100000,5012803,68.800019,46.510625,18.085538,18.489476,0.224181,142193,30913,107670,S1: todas las observaciones con RainTomorrow v...,21.877275,0.804990,0.686160,0.765859,0.459993,0.893564,0.335940,225.210000,335.940000,154.530000,181.410000,70.680000,10000,1000,0.460414,0.406666,0.513433,0.686471,0.623430,0.748793,0.094228,0.003462
7,Proporción Yes +5%,proporcion_raintomorrow_yes,0.050000,100000,5012803,68.800019,46.510625,19.037409,18.489476,0.235390,142193,30913,107670,S1: todas las observaciones con RainTomorrow v...,21.783047,0.799000,0.676842,0.765800,0.472835,0.884199,0.339040,236.850000,339.040000,160.310000,178.730000,76.540000,10000,1000,0.473161,0.420588,0.525641,0.676952,0.616327,0.735294,0.000000,0.016304
8,Proporción Yes -5%,proporcion_raintomorrow_yes,-0.050000,100000,5012803

,escenario,parametro_modificado,cambio_relativo,n_iteraciones,semilla,media_yes_usada_regenerado_S3,media_no_usada_regenerado_S3,sd_yes_usada_regenerado_S3,sd_no_usada_regenerado_S3,proporcion_yes_usada_regenerado_S3,n_objetivo_valido_para_prevalencia,n_humidity_observada_yes,n_humidity_observada_no,fuente_prevalencia,media_delta_individual_yes_no_regenerado_S3,probabilidad_delta_individual_mayor_0,sensibilidad_umbral_60_regenerado_S3,especificidad_umbral_60_regenerado_S3,vpp_umbral_60_regenerado_S3,vpn_umbral_60,proporcion_alertas_umbral_60_regenerado_S3,eventos_lluvia_esperados_por_1000,alertas_esperadas_por_1000,verdaderos_positivos_esperados_por_1000,falsos_positivos_esperados_por_1000,eventos_no_detectados_esperados_por_1000,n_carteras_simuladas,tamano_cartera,vpp_cartera_media,vpp_cartera_ic_empirico_95_li,vpp_cartera_ic_empirico_95_ls,sensibilidad_cartera_media,sensibilidad_cartera_ic_empirico_95_li,sensibilidad_cartera_ic_empirico_95_ls,variacion_media_delta_vs_base,variacion_vpp_umbral_60_vs_base,media_yes_usada_archivo_S2,media_no_usada_archivo_S2,sd_yes_usada_archivo_S2,sd_no_usada_archivo_S2,proporcion_yes_usada_archivo_S2,media_delta_individual_yes_no_archivo_S2,sensibilidad_umbral_60_archivo_S2,especificidad_umbral_60_archivo_S2,vpp_umbral_60_archivo_S2,proporcion_alertas_umbral_60_archivo_S2,diferencia_absoluta_media_yes_usada,diferencia_absoluta_media_no_usada,diferencia_absoluta_sd_yes_usada,diferencia_absoluta_sd_no_usada,diferencia_absoluta_proporcion_yes_usada,diferencia_absoluta_media_delta_individual_yes_no,diferencia_absoluta_sensibilidad_umbral_60,diferencia_absoluta_especificidad_umbral_60,diferencia_absoluta_vpp_umbral_60,diferencia_absoluta_proporcion_alertas_umbral_60,consistencia_parametros_S1_S2,consistencia_resultados_simulados,consistencia_numerica
0,Base,Sin modificación,0.000000,100000,5012803,68.800019,46.510625,19.037409,18.489476,0.224181,142193,30913,107670,S1: todas las observaciones con RainTomorrow v...,21.783047,0.799000,0.676657,0.765859,0.456531,0.890693,0.333800,225.210000,333.800000,152.390000,181.410000,72.820000,10000,1000,0.456708,0.403846,0.509092,0.676589,0.615385,0.736842,0.000000,0.000000,68.800019,46.510625,19.037409,18.489476,0.224181,21.860308,0.679948,0.765429,0.456084,0.334500,0.000000,0.000000,0.000000,0.000000,0.000000,0.077262,0.003291,0.000430,0.000447,0.000700,True,True,True
1,Media Yes +5%,media_humidity3pm_yes,0.050000,100000,5012803,72.240020,46.510625,19.037409,18.489476,0.224181,142193,30913,107670,S1: todas las observaciones con RainTomorrow v...,25.014025,0.833320,0.740065,0.765859,0.478827,0.910204,0.348080,225.210000,348.080000,166.670000,181.410000,58.540000,10000,1000,0.478523,0.425146,0.530615,0.740243,0.680849,0.797875,3.230978,0.022296,72.240020,46.510625,19.037409,18.489476,0.224181,25.016057,0.745099,0.762883,0.474801,0.350610,0.000000,0.000000,0.000000,0.000000,0.000000,0.002032,0.005034,0.002976,0.004026,0.002530,True,True,True
2,Media Yes -5%,media_humidity3pm_yes,-0.050000,100000,5012803,65.360018,46.510625,19.037409,18.489476,0.224181,142193,30913,107670,S1: todas las observaciones con RainTomorrow v...,18.487787,0.761300,0.612051,0.765859,0.431762,0.871656,0.319250,225.210000,319.250000,137.840000,181.410000,87.370000,10000,1000,0.431567,0.375394,0.486239,0.612171,0.547738,0.676190,-3.295260,-0.024769,65.360018,46.510625,19.037409,18.489476,0.224181,18.601474,0.603761,0.766905,0.427645,0.316080,0.000000,0.000000,0.000000,0.000000,0.000000,0.113687,0.008290,0.001045,0.004117,0.003170,True,True,True
3,Media No +5%,media_humidity3pm_no,0.050000,100000,5012803,68.800019,48.836156,19.037409,18.489476,0.224181,142193,30913,107670,S1: todas las observaciones con RainTomorrow v...,19.474798,0.773760,0.676657,0.725990,0.417862,0.885379,0.364690,225.210000,364.690000,152.390000,212.300000,72.820000,10000,1000,0.417883,0.368269,0.468495,0.676432,0.615385,0.737971,-2.308248,-0.038669,68.800019,48.836156,19.037409,18.489476,0.224181,19.427173,0.674397,0.72

### 1.3 Reproducción del diagnóstico IQR y recálculo de multicolinealidad

El diagnóstico IQR de S2 se reproduce sobre los mismos casos válidos de `Humidity3pm` y `RainTomorrow`, con límites calculados separadamente por grupo. Las correlaciones de redundancia se recalculan mediante casos completos por par sobre la base original completa. Estos coeficientes actualizan la trazabilidad de los pares `Pressure9am`–`Pressure3pm` y `Temp9am`–`Temp3pm`; la selección definitiva del modelo continúa respaldándose con VIF y diagnósticos calculados exclusivamente sobre entrenamiento.

In [5]:
def iqr_group_diagnostic(values):
    values = np.asarray(values, dtype=float)
    q1, q3 = np.quantile(values, [0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    retained = values[(values >= lower) & (values <= upper)]
    return {
        'q1': float(q1),
        'q3': float(q3),
        'IQR': float(iqr),
        'limite_inferior': float(lower),
        'limite_superior': float(upper),
        'n_original': len(values),
        'n_filtrado_iqr': len(retained),
        'datos_eliminados': len(values) - len(retained),
        'porcentaje_eliminado': 100 * (len(values) - len(retained)) / len(values)
    }


iqr_records = []
for group_value, group_label in [(1, 'RainTomorrow = Yes'), (0, 'RainTomorrow = No')]:
    group_values = raw.loc[
        raw['RainTomorrow_bin'].eq(group_value)
        & raw['Humidity3pm'].notna(),
        'Humidity3pm'
    ].to_numpy(dtype=float)
    iqr_records.append({
        'grupo': group_label,
        **iqr_group_diagnostic(group_values)
    })

iqr_diagnostic_regenerated = pd.DataFrame(iqr_records)
iqr_diagnostic_regenerated.to_csv(
    TABLES / '02e_diagnostico_outliers_IQR_regenerado.csv',
    index=False
)

iqr_comparison = iqr_diagnostic_regenerated.merge(
    s2_outliers,
    on='grupo',
    how='left',
    suffixes=('_regenerado_S3', '_archivo_S2')
)
for variable in [
    'n_original', 'n_filtrado_iqr', 'datos_eliminados',
    'porcentaje_eliminado'
]:
    iqr_comparison[f'diferencia_{variable}'] = (
        iqr_comparison[f'{variable}_regenerado_S3']
        - iqr_comparison[f'{variable}_archivo_S2']
    )
iqr_comparison['reproduccion_exacta_conteos'] = (
    iqr_comparison['diferencia_n_original'].eq(0)
    & iqr_comparison['diferencia_n_filtrado_iqr'].eq(0)
    & iqr_comparison['diferencia_datos_eliminados'].eq(0)
)
iqr_comparison['reproduccion_porcentaje_tolerancia'] = (
    iqr_comparison['diferencia_porcentaje_eliminado'].abs().le(1e-5)
)
iqr_comparison.to_csv(
    TABLES / '02f_consistencia_diagnostico_IQR_S2_S3.csv',
    index=False
)

multicollinearity_pairs = [
    ('Pressure9am', 'Pressure3pm', 'Par expresamente observado en la retroalimentación de S2'),
    ('Temp9am', 'Temp3pm', 'Par expresamente observado en la retroalimentación de S2'),
    ('Temp3pm', 'MaxTemp', 'Redundancia relevante para delimitar el universo logístico'),
    ('Temp9am', 'MinTemp', 'Redundancia relevante para delimitar el universo logístico'),
    ('Temp9am', 'MaxTemp', 'Redundancia relevante para delimitar el universo logístico')
]

multicollinearity_rows = []
for variable_1, variable_2, purpose in multicollinearity_pairs:
    pair = raw[[variable_1, variable_2]].dropna()
    recalculated = float(pair[variable_1].corr(pair[variable_2]))
    historical_s1 = (
        float(s1_corr.loc[variable_1, variable_2])
        if variable_1 in s1_corr.index and variable_2 in s1_corr.columns
        else np.nan
    )
    historical_s2_row = s2_collinearity.loc[
        (
            s2_collinearity['variable_1'].eq(variable_1)
            & s2_collinearity['variable_2'].eq(variable_2)
        )
        | (
            s2_collinearity['variable_1'].eq(variable_2)
            & s2_collinearity['variable_2'].eq(variable_1)
        )
    ]
    historical_s2 = (
        float(historical_s2_row['correlacion_pearson'].iloc[0])
        if not historical_s2_row.empty else np.nan
    )
    multicollinearity_rows.append({
        'variable_1': variable_1,
        'variable_2': variable_2,
        'n_pares_validos_base_completa': len(pair),
        'correlacion_pearson_recalculada_S3': recalculated,
        'abs_correlacion_recalculada_S3': abs(recalculated),
        'correlacion_matriz_S1': historical_s1,
        'correlacion_referencia_S2': historical_s2,
        'diferencia_absoluta_vs_matriz_S1': (
            abs(recalculated - historical_s1)
            if np.isfinite(historical_s1) else np.nan
        ),
        'diferencia_absoluta_vs_referencia_S2': (
            abs(recalculated - historical_s2)
            if np.isfinite(historical_s2) else np.nan
        ),
        'proposito': purpose,
        'uso_metodologico': (
            'Trazabilidad retrospectiva sobre la base completa. La decisión '
            'predictiva se confirma posteriormente mediante VIF en entrenamiento.'
        )
    })

multicollinearity_recalculated = pd.DataFrame(multicollinearity_rows)
multicollinearity_recalculated.to_csv(
    TABLES / '02g_recalculo_multicolinealidad_base_completa.csv',
    index=False
)


def recalculated_pair_correlation(variable_1, variable_2):
    row = multicollinearity_recalculated.loc[
        (
            multicollinearity_recalculated['variable_1'].eq(variable_1)
            & multicollinearity_recalculated['variable_2'].eq(variable_2)
        )
        | (
            multicollinearity_recalculated['variable_1'].eq(variable_2)
            & multicollinearity_recalculated['variable_2'].eq(variable_1)
        )
    ]
    if row.empty:
        raise KeyError(
            f'No existe correlación recalculada para {variable_1}–{variable_2}.'
        )
    return float(row['correlacion_pearson_recalculada_S3'].iloc[0])


correction_control = pd.DataFrame([
    {
        'observacion_retroalimentacion_S2': 'Profundizar bootstrap con un parámetro asimétrico',
        'evidencia_generada_S3': '02a_bootstrap_Rainfall_asimetria.csv',
        'criterio_verificacion': '10.000 remuestras; IC clásico, percentil y BCa finitos',
        'resultado': (
            f'n_boot={N_BOOT_RAINFALL}; asimetría={rainfall_skewness:.4f}; '
            f'BCa=[{rainfall_bca_li:.4f}, {rainfall_bca_ls:.4f}]'
        ),
        'cumple': bool(
            N_BOOT_RAINFALL >= 10_000
            and np.isfinite([
                rainfall_classic_li, rainfall_classic_ls,
                rainfall_percentile_li, rainfall_percentile_ls,
                rainfall_bca_li, rainfall_bca_ls
            ]).all()
        )
    },
    {
        'observacion_retroalimentacion_S2': 'Incorporar el código de sensibilidad Monte Carlo al notebook maestro',
        'evidencia_generada_S3': (
            '02b0_parametros_base_montecarlo_S1.csv; '
            '02b_sensibilidad_montecarlo_regenerada.csv'
        ),
        'criterio_verificacion': '100.000 iteraciones; escenario base y ocho escenarios de sensibilidad',
        'resultado': (
            f'n_iteraciones={N_MC_S2_CLOSURE}; '
            f'n_escenarios={len(montecarlo_sensitivity_regenerated)}; '
            f'prevalencia_S1={mc_parameters_base["proporcion_yes"]:.6f}'
        ),
        'cumple': bool(
            N_MC_S2_CLOSURE >= 10_000
            and len(montecarlo_sensitivity_regenerated) == 9
            and mc_parameter_traceability['coincide_referencia_S2'].all()
            and np.isclose(
                mc_parameters_base['proporcion_yes'],
                0.22418121848473554,
                rtol=0,
                atol=1e-12
            )
            and mc_consistency['consistencia_numerica'].all()
        )
    },
    {
        'observacion_retroalimentacion_S2': 'Agregar un estimador agregado conectado con la decisión',
        'evidencia_generada_S3': '02b_sensibilidad_montecarlo_regenerada.csv',
        'criterio_verificacion': 'Tasas y resultados esperados por 1.000 observaciones; distribución por carteras',
        'resultado': (
            f'tamaño_cartera={MC_PORTFOLIO_SIZE}; '
            f'n_carteras={N_MC_PORTFOLIOS}; '
            f'VPP_base={base_mc_row["vpp_umbral_60"]:.4f}'
        ),
        'cumple': bool(
            N_MC_PORTFOLIOS >= 10_000
            and montecarlo_sensitivity_regenerated[
                [
                    'alertas_esperadas_por_1000',
                    'verdaderos_positivos_esperados_por_1000',
                    'falsos_positivos_esperados_por_1000',
                    'eventos_no_detectados_esperados_por_1000',
                    'vpp_cartera_media',
                    'vpp_cartera_ic_empirico_95_li',
                    'vpp_cartera_ic_empirico_95_ls',
                    'sensibilidad_cartera_media',
                    'sensibilidad_cartera_ic_empirico_95_li',
                    'sensibilidad_cartera_ic_empirico_95_ls'
                ]
            ].notna().all().all()
        )
    },
    {
        'observacion_retroalimentacion_S2': 'Regenerar el diagnóstico IQR citado en la trazabilidad',
        'evidencia_generada_S3': '02e_diagnostico_outliers_IQR_regenerado.csv',
        'criterio_verificacion': 'Reproducción exacta de conteos y porcentaje dentro de tolerancia numérica',
        'resultado': (
            f'grupos_reproducidos={len(iqr_diagnostic_regenerated)}; '
            f'eliminados_total={int(iqr_diagnostic_regenerated["datos_eliminados"].sum())}'
        ),
        'cumple': bool(
            iqr_comparison['reproduccion_exacta_conteos'].all()
            and iqr_comparison['reproduccion_porcentaje_tolerancia'].all()
        )
    },
    {
        'observacion_retroalimentacion_S2': 'Recalcular correlaciones redundantes sobre la base completa',
        'evidencia_generada_S3': '02g_recalculo_multicolinealidad_base_completa.csv',
        'criterio_verificacion': 'Pares observados recalculados con n válido y concordancia con referencias',
        'resultado': (
            'Pressure9am–Pressure3pm='
            f'{recalculated_pair_correlation("Pressure9am", "Pressure3pm"):.4f}; '
            'Temp9am–Temp3pm='
            f'{recalculated_pair_correlation("Temp9am", "Temp3pm"):.4f}'
        ),
        'cumple': bool(
            multicollinearity_recalculated['n_pares_validos_base_completa'].gt(0).all()
            and multicollinearity_recalculated.loc[
                multicollinearity_recalculated['correlacion_referencia_S2'].notna(),
                'diferencia_absoluta_vs_referencia_S2'
            ].le(0.01).all()
        )
    }
])
correction_control.to_csv(
    TABLES / '02h_control_cierre_retroalimentacion_S2.csv',
    index=False
)
if not correction_control['cumple'].all():
    failed_corrections = correction_control.loc[
        ~correction_control['cumple'],
        'observacion_retroalimentacion_S2'
    ].tolist()
    raise AssertionError(
        'No se cerraron todas las observaciones heredadas de S2: '
        f'{failed_corrections}'
    )

traceability_additions = pd.DataFrame([
    {
        'tipo_insumo_s3': 'Profundización bootstrap solicitada en retroalimentación S2',
        'resultado_validado': 'Media de Rainfall evaluada con IC clásico, percentil y BCa',
        'evidencia_sumativa2': (
            f'Rainfall asimétrica; asimetría={rainfall_skewness:.4f}; '
            f'10.000 remuestras generadas en S3'
        ),
        'decision_para_s3': (
            'Mantener transformación log1p en modelamiento y documentar que la '
            'distribución muestral de la media puede ser casi simétrica con gran n.'
        ),
        'nivel_confianza': 'Alto',
        'fase_origen': 'Retroalimentación S2 incorporada en S3'
    },
    {
        'tipo_insumo_s3': 'Sensibilidad Monte Carlo reproducida',
        'resultado_validado': 'Escenario base y ocho perturbaciones relativas de parámetros',
        'evidencia_sumativa2': (
            f'100.000 iteraciones; prevalencia S1='
            f'{mc_parameters_base["proporcion_yes"]:.6f}; VPP base al '
            f'umbral de humedad 60 %={base_mc_row["vpp_umbral_60"]:.4f}'
        ),
        'decision_para_s3': (
            'Conservar el análisis como antecedente operacional y distinguir el '
            'umbral meteorológico de 60 % del umbral probabilístico 0,60.'
        ),
        'nivel_confianza': 'Alto',
        'fase_origen': 'Retroalimentación S2 incorporada en S3'
    },
    {
        'tipo_insumo_s3': 'Diagnóstico IQR reproducido',
        'resultado_validado': 'Conteos por grupo reproducidos desde la base original',
        'evidencia_sumativa2': (
            f'{int(iqr_diagnostic_regenerated["datos_eliminados"].sum())} '
            'observaciones eliminadas por IQR en total'
        ),
        'decision_para_s3': (
            'Mantener outliers en el modelo principal y evaluar sensibilidad '
            'mediante winsorización, sin depender de un archivo no regenerado.'
        ),
        'nivel_confianza': 'Alto',
        'fase_origen': 'Retroalimentación S2 incorporada en S3'
    },
    {
        'tipo_insumo_s3': 'Multicolinealidad recalculada',
        'resultado_validado': (
            'Pressure9am–Pressure3pm y Temp9am–Temp3pm recalculadas '
            'sobre la base completa'
        ),
        'evidencia_sumativa2': (
            f'r={recalculated_pair_correlation("Pressure9am", "Pressure3pm"):.4f} '
            'y '
            f'r={recalculated_pair_correlation("Temp9am", "Temp3pm"):.4f}'
        ),
        'decision_para_s3': (
            'Excluir redundancias antes de la selección automática y confirmar '
            'la especificación definitiva mediante VIF calculado en entrenamiento.'
        ),
        'nivel_confianza': 'Alto',
        'fase_origen': 'Retroalimentación S2 incorporada en S3'
    }
])
traceability = pd.concat(
    [traceability, traceability_additions],
    ignore_index=True
)
traceability.to_csv(
    TABLES / '01_trazabilidad_S1_S2_hacia_S3.csv',
    index=False
)

display(iqr_diagnostic_regenerated)
display(multicollinearity_recalculated)
display(correction_control)

,grupo,q1,q3,IQR,limite_inferior,limite_superior,n_original,n_filtrado_iqr,datos_eliminados,porcentaje_eliminado
0,RainTomorrow = Yes,57.000000,84.000000,27.000000,16.500000,124.500000,30913,30694,219,0.708440
1,RainTomorrow = No,33.000000,60.000000,27.000000,-7.500000,100.500000,107670,107670,0,0.000000


,variable_1,variable_2,n_pares_validos_base_completa,correlacion_pearson_recalculada_S3,abs_correlacion_recalculada_S3,correlacion_matriz_S1,correlacion_referencia_S2,diferencia_absoluta_vs_matriz_S1,diferencia_absoluta_vs_referencia_S2,proposito,uso_metodologico
0,Pressure9am,Pressure3pm,130171,0.961326,0.961326,0.961000,0.960000,0.000326,0.001326,Par expresamente observado en la retroalimenta...,Trazabilidad retrospectiva sobre la base compl...
1,Temp9am,Temp3pm,141213,0.860591,0.860591,0.861000,0.860000,0.000409,0.000591,Par expresamente observado en la retroalimenta...,Trazabilidad retrospectiva sobre la base compl...
2,Temp3pm,MaxTemp,141589,0.984503,0.984503,0.985000,NaN,0.000497,NaN,Redundancia relevante para delimitar el univer...,Trazabilidad retrospectiva sobre la base compl...
3,Temp9am,MinTemp,143382,0.901821,0.901821,0.902000,NaN,0.000179,NaN,Redundancia relevante para delimitar el univer...,Trazabilidad retrospectiva sobre la base compl...
4,Temp9am,MaxTemp,143306,0.887210,0.887210,0.887000,NaN,0.000210,NaN,Redundancia relevante para delimitar el univer...,Trazabilidad retrospectiva sobre la base compl...


,observacion_retroalimentacion_S2,evidencia_generada_S3,criterio_verificacion,resultado,cumple
0,Profundizar bootstrap con un parámetro asimétrico,02a_bootstrap_Rainfall_asimetria.csv,"10.000 remuestras; IC clásico, percentil y BCa...","n_boot=10000; asimetría=9.8362; BCa=[2.3177, 2...",True
1,Incorporar el código de sensibilidad Monte Car...,02b0_parametros_base_montecarlo_S1.csv; 02b_se...,100.000 iteraciones; escenario base y ocho esc...,n_iteraciones=100000; n_escenarios=9; prevalen...,True
2,Agregar un estimador agregado conectado con la...,02b_sensibilidad_montecarlo_regenerada.csv,Tasas y resultados esperados por 1.000 observa...,tamaño_cartera=1000; n_carteras=10000; VPP_bas...,True
3,Regenerar el diagnóstico IQR citado en la traz...,02e_diagnostico_outliers_IQR_regenerado.csv,Reproducción exacta de conteos y porcentaje de...,grupos_reproducidos=2; eliminados_total=219,True
4,Recalcular correlaciones redundantes sobre la ...,02g_recalculo_multicolinealidad_base_completa.csv,Pares observados recalculados con n válido y c...,Pressure9am–Pressure3pm=0.9613; Temp9am–Temp3p...,True


## 2. Manejo de datos faltantes

S1 cuantificó los faltantes por variable, pero no clasificó formalmente su mecanismo. S3 utiliza esa auditoría como punto de partida y evalúa, solo con entrenamiento, la asociación entre los indicadores de ausencia y variables observadas.

La evidencia contra MCAR se interpreta como compatible con MAR. MAR se adopta como supuesto de trabajo, no como certeza; MNAR no puede confirmarse ni descartarse únicamente con los datos observados.


In [6]:
analysis = raw.loc[raw['RainTomorrow_bin'].notna()].copy()
analysis['RainTomorrow_bin'] = analysis['RainTomorrow_bin'].astype(int)

train_positions, test_positions = train_test_split(
    np.arange(len(analysis)),
    test_size=TEST_SIZE,
    random_state=SEED,
    stratify=analysis['RainTomorrow_bin']
)
train_raw = analysis.iloc[train_positions].copy()
test_raw = analysis.iloc[test_positions].copy()

split_summary = pd.DataFrame([
    ['entrenamiento', len(train_raw), train_raw['RainTomorrow_bin'].mean()],
    ['prueba', len(test_raw), test_raw['RainTomorrow_bin'].mean()]
], columns=['conjunto', 'n', 'prevalencia_RainTomorrow_Yes'])
split_summary.to_csv(TABLES / '04_particion_70_30_estratificada.csv', index=False)
display(split_summary)

assert set(train_raw['row_id']).isdisjoint(set(test_raw['row_id']))

,conjunto,n,prevalencia_RainTomorrow_Yes
0,entrenamiento,99535,0.224182
1,prueba,42658,0.224178


### 2.1 Alcance analítico coherente con S1 y S2

La matriz de correlaciones de S1 contiene diez variables numéricas meteorológicas, además de `RainToday_bin` y `RainTomorrow_bin`. Estas diez variables definen el alcance numérico de S3:

`MinTemp`, `MaxTemp`, `Rainfall`, `WindGustSpeed`, `Humidity9am`, `Humidity3pm`, `Pressure9am`, `Pressure3pm`, `Temp9am` y `Temp3pm`.

Cada variable numérica retenida con faltantes recibe una regresión lineal múltiple explícita con tres a cinco predictores de la matriz de S1. `RainToday_bin` se trata mediante regresión logística.

`Evaporation`, `Sunshine`, `WindSpeed9am`, `WindSpeed3pm`, `Cloud9am` y `Cloud3pm` fueron auditadas en S1, pero no forman parte de su matriz de correlaciones ni de la validación de S2. Por ello se excluyen antes de la imputación y del modelamiento. La tabla de alcance comprueba que las dieciséis variables numéricas con faltantes quedan clasificadas como incluidas o excluidas, sin casos sin decisión.


In [7]:
S1_CORE_NUMERIC = [
    'MinTemp', 'MaxTemp', 'Rainfall', 'WindGustSpeed',
    'Humidity9am', 'Humidity3pm', 'Pressure9am', 'Pressure3pm',
    'Temp9am', 'Temp3pm'
]
NUMERIC_EXCLUDED_FOR_TRACEABILITY = [
    'Evaporation', 'Sunshine', 'WindSpeed9am', 'WindSpeed3pm',
    'Cloud9am', 'Cloud3pm'
]
NUMERIC_IMPUTATION_TARGETS = S1_CORE_NUMERIC.copy()
MODEL_NUMERIC = S1_CORE_NUMERIC.copy()
ANALYTIC_VARIABLES = NUMERIC_IMPUTATION_TARGETS + ['RainToday_bin']

# Verificación de coherencia con la matriz efectivamente entregada en S1.
missing_core_in_s1_matrix = [
    variable for variable in S1_CORE_NUMERIC
    if variable not in s1_corr.index or variable not in s1_corr.columns
]
if missing_core_in_s1_matrix:
    raise AssertionError(
        'Las variables analíticas deben existir en la matriz de correlaciones '
        f'de S1. Faltan: {missing_core_in_s1_matrix}.'
    )

excluded_unexpectedly_in_s1_matrix = [
    variable for variable in NUMERIC_EXCLUDED_FOR_TRACEABILITY
    if variable in s1_corr.index or variable in s1_corr.columns
]
if excluded_unexpectedly_in_s1_matrix:
    raise AssertionError(
        'La delimitación debe revisarse porque estas variables sí aparecen en '
        f'la matriz de S1: {excluded_unexpectedly_in_s1_matrix}.'
    )

numeric_missing_original = {
    variable
    for variable in raw.select_dtypes(include=np.number).columns
    if variable not in {'row_id', 'Year', 'Month', 'RainTomorrow_bin'}
    and analysis[variable].isna().any()
}
expected_missing_inventory = set(
    S1_CORE_NUMERIC
    + NUMERIC_EXCLUDED_FOR_TRACEABILITY
    + ['RainToday_bin']
)
if numeric_missing_original != expected_missing_inventory:
    raise AssertionError(
        'El inventario de variables numéricas faltantes cambió respecto del '
        f'alcance documentado. Observado={sorted(numeric_missing_original)}; '
        f'esperado={sorted(expected_missing_inventory)}.'
    )

missing_from_s1 = s1_missing.copy()
missing_from_s1['seleccion_S3'] = np.select(
    [
        missing_from_s1['variable'].isin(S1_CORE_NUMERIC),
        missing_from_s1['variable'].isin(NUMERIC_EXCLUDED_FOR_TRACEABILITY),
        missing_from_s1['variable'].isin(['RainToday', 'RainToday_bin']),
        missing_from_s1['variable'].isin(['RainTomorrow', 'RainTomorrow_bin'])
    ],
    [
        'Incluida: imputación explícita y candidata de modelamiento',
        'Excluida antes de imputación y modelamiento por trazabilidad S1/S2',
        'Incluida: imputación logística del predictor binario',
        'Objetivo: excluir faltantes y no imputar'
    ],
    default='Excluida del modelamiento de S3'
)
missing_from_s1['fundamento_S3'] = np.select(
    [
        missing_from_s1['variable'].isin(S1_CORE_NUMERIC),
        missing_from_s1['variable'].isin(NUMERIC_EXCLUDED_FOR_TRACEABILITY),
        missing_from_s1['variable'].isin(['RainToday', 'RainToday_bin']),
        missing_from_s1['variable'].isin(['RainTomorrow', 'RainTomorrow_bin'])
    ],
    [
        'Variable numérica incluida en la matriz de correlaciones entregada en S1; recibe imputación explícita y puede integrar los modelos logísticos.',
        'Variable auditada en S1, pero ausente de la matriz de correlaciones de S1 y de la validación de S2. Se excluye para no reconstruir evidencia retrospectiva.',
        'Predictor binario respaldado en S1 y S2; se imputa mediante regresión logística sin utilizar RainTomorrow.',
        'Variable objetivo; se eliminan las observaciones sin respuesta y nunca se imputa.'
    ],
    default='Variable no requerida por la especificación predictiva heredada de S1/S2.'
)
missing_from_s1.to_csv(
    TABLES / '03a_faltantes_S1_y_alcance_S3.csv',
    index=False
)
display(missing_from_s1)

numeric_scope = []
for variable in raw.select_dtypes(include=np.number).columns:
    if variable == 'row_id':
        continue
    n_missing = int(analysis[variable].isna().sum())
    if variable in S1_CORE_NUMERIC:
        role = 'Imputación explícita y candidata del modelo logístico'
        correlation_source = 'Matriz de correlaciones entregada en S1'
        decision = (
            'Imputar mediante regresión lineal múltiple explícita'
            if n_missing > 0 else 'Sin imputación requerida'
        )
    elif variable in NUMERIC_EXCLUDED_FOR_TRACEABILITY:
        role = 'Variable documentada en S1, excluida del conjunto analítico S3'
        correlation_source = 'No disponible en la matriz de S1; no se reconstruye en S3'
        decision = 'Excluir antes de comparar estrategias y ajustar modelos'
    elif variable == 'RainToday_bin':
        role = 'Imputación logística del predictor binario'
        correlation_source = 'S1 y validación de S2'
        decision = (
            'Imputar mediante regresión logística explícita'
            if n_missing > 0 else 'Sin imputación requerida'
        )
    elif variable == 'RainTomorrow_bin':
        role = 'Variable objetivo; no se imputa'
        correlation_source = 'No aplica'
        decision = (
            'Excluir observaciones con objetivo faltante'
            if n_missing > 0 else 'Sin faltantes'
        )
    else:
        role = 'Variable derivada sin imputación'
        correlation_source = 'No aplica'
        decision = 'Sin imputación requerida' if n_missing == 0 else 'Revisar alcance'

    numeric_scope.append({
        'variable': variable,
        'n_faltantes_base_modelable': n_missing,
        'porcentaje_faltantes_base_modelable': 100 * analysis[variable].isna().mean(),
        'incluida_en_conjunto_analitico_S3': variable in ANALYTIC_VARIABLES,
        'incluida_en_matriz_correlaciones_S1': (
            variable in s1_corr.index or variable in s1_corr.columns
        ),
        'rol_S3': role,
        'fuente_correlacion_predictores': correlation_source,
        'decision': decision
    })

numeric_scope = pd.DataFrame(numeric_scope)
numeric_scope.to_csv(
    TABLES / '03b_inventario_variables_numericas_y_decision.csv',
    index=False
)
display(numeric_scope)


continuous_missing_original = (
    numeric_missing_original - {'RainToday_bin'}
)
continuous_accounted = (
    set(S1_CORE_NUMERIC)
    | set(NUMERIC_EXCLUDED_FOR_TRACEABILITY)
)
if continuous_missing_original != continuous_accounted:
    raise AssertionError(
        'No todas las variables numéricas continuas con faltantes poseen una '
        'decisión explícita de inclusión o exclusión. '
        f'Observadas={sorted(continuous_missing_original)}; '
        f'contabilizadas={sorted(continuous_accounted)}.'
    )
if set(S1_CORE_NUMERIC) & set(NUMERIC_EXCLUDED_FOR_TRACEABILITY):
    raise AssertionError(
        'Una variable no puede estar simultáneamente incluida y excluida.'
    )

scope_coverage_control = pd.DataFrame([
    {
        'dimension': 'Variables numéricas continuas con faltantes en la base modelable',
        'cantidad': len(continuous_missing_original),
        'variables': ', '.join(sorted(continuous_missing_original)),
        'decision': 'Universo completo auditado antes del tratamiento de faltantes'
    },
    {
        'dimension': 'Variables elegibles e imputadas explícitamente en S3',
        'cantidad': len(S1_CORE_NUMERIC),
        'variables': ', '.join(S1_CORE_NUMERIC),
        'decision': 'Presentes en la matriz de S1; regresión lineal múltiple por variable'
    },
    {
        'dimension': 'Variables excluidas antes de la imputación',
        'cantidad': len(NUMERIC_EXCLUDED_FOR_TRACEABILITY),
        'variables': ', '.join(NUMERIC_EXCLUDED_FOR_TRACEABILITY),
        'decision': 'Ausentes de la matriz de S1 y de la validación de S2; no integran el modelo final'
    },
    {
        'dimension': 'Predictor binario con faltantes',
        'cantidad': 1,
        'variables': 'RainToday_bin',
        'decision': 'Imputación mediante regresión logística explícita'
    },
    {
        'dimension': 'Variables numéricas faltantes sin decisión',
        'cantidad': 0,
        'variables': 'ninguna',
        'decision': 'Cobertura exhaustiva confirmada'
    }
])
scope_coverage_control.to_csv(
    TABLES / '03f_control_cobertura_total_variables_faltantes.csv',
    index=False
)
display(scope_coverage_control)


,variable,n_faltantes,porcentaje_faltantes,seleccion_S3,fundamento_S3
0,Sunshine,69835,48.009762,Excluida antes de imputación y modelamiento po...,"Variable auditada en S1, pero ausente de la ma..."
1,Evaporation,62790,43.166506,Excluida antes de imputación y modelamiento po...,"Variable auditada en S1, pero ausente de la ma..."
2,Cloud3pm,59358,40.807095,Excluida antes de imputación y modelamiento po...,"Variable auditada en S1, pero ausente de la ma..."
3,Cloud9am,55888,38.421559,Excluida antes de imputación y modelamiento po...,"Variable auditada en S1, pero ausente de la ma..."
4,Pressure9am,15065,10.356799,Incluida: imputación explícita y candidata de ...,Variable numérica incluida en la matriz de cor...
5,Pressure3pm,15028,10.331363,Incluida: imputación explícita y candidata de ...,Variable numérica incluida en la matriz de cor...
6,WindDir9am,10566,7.263853,Excluida del modelamiento de S3,Variable no requerida por la especificación pr...
7,WindGustDir,10326,7.098859,Excluida del modelamiento de S3,Variable no requerida por la especificación pr...
8,WindGustSpeed,10263,7.055548,Incluida: imputación explícita y candidata de ...,Variable numérica incluida en la matriz de cor...
9,Humidity3pm,4507,3.098446,Incluida: imputación explícita y candidata de ...,Variable numérica incluida en la matriz de cor...


,variable,n_faltantes_base_modelable,porcentaje_faltantes_base_modelable,incluida_en_conjunto_analitico_S3,incluida_en_matriz_correlaciones_S1,rol_S3,fuente_correlacion_predictores,decision
0,MinTemp,637,0.447983,True,True,Imputación explícita y candidata del modelo lo...,Matriz de correlaciones entregada en S1,Imputar mediante regresión lineal múltiple exp...
1,MaxTemp,322,0.226453,True,True,Imputación explícita y candidata del modelo lo...,Matriz de correlaciones entregada en S1,Imputar mediante regresión lineal múltiple exp...
2,Rainfall,1406,0.988797,True,True,Imputación explícita y candidata del modelo lo...,Matriz de correlaciones entregada en S1,Imputar mediante regresión lineal múltiple exp...
3,Evaporation,60843,42.789026,False,False,"Variable documentada en S1, excluida del conju...",No disponible en la matriz de S1; no se recons...,Excluir antes de comparar estrategias y ajusta...
4,Sunshine,67816,47.692924,False,False,"Variable documentada en S1, excluida del conju...",No disponible en la matriz de S1; no se recons...,Excluir antes de comparar estrategias y ajusta...
5,WindGustSpeed,9270,6.519308,True,True,Imputación explícita y candidata del modelo lo...,Matriz de correlaciones entregada en S1,Imputar mediante regresión lineal múltiple exp...
6,WindSpeed9am,1348,0.948007,False,False,"Variable documentada en S1, excluida del conju...",No disponible en la matriz de S1; no se recons...,Excluir antes de comparar estrategias y ajusta...
7,WindSpeed3pm,2630,1.849599,False,False,"Variable documentada en S1, excluida del conju...",No disponible en la matriz de S1; no se recons...,Excluir antes de comparar estrategias y ajusta...
8,Humidity9am,1774,1.247600,True,True,Imputación explícita y candidata del modelo lo...,Matriz de correlaciones entregada en S1,Imputar mediante regresión lineal múltiple exp...
9,Humidity3pm,3610,2.538803,True,True,Imputación explícita y candidata del modelo lo...,Matriz de correlaciones entregada en S1,Imputar mediante regresión lineal múltiple exp...


,dimension,cantidad,variables,decision
0,Variables numéricas continuas con faltantes en...,16,"Cloud3pm, Cloud9am, Evaporation, Humidity3pm, ...",Universo completo auditado antes del tratamien...
1,Variables elegibles e imputadas explícitamente...,10,"MinTemp, MaxTemp, Rainfall, WindGustSpeed, Hum...",Presentes en la matriz de S1; regresión lineal...
2,Variables excluidas antes de la imputación,6,"Evaporation, Sunshine, WindSpeed9am, WindSpeed...",Ausentes de la matriz de S1 y de la validación...
3,Predictor binario con faltantes,1,RainToday_bin,Imputación mediante regresión logística explícita
4,Variables numéricas faltantes sin decisión,0,ninguna,Cobertura exhaustiva confirmada


### 2.2 Diagnóstico complementario del mecanismo de ausencia

El diagnóstico utiliza únicamente el conjunto de entrenamiento. Para cada variable se evalúa la asociación del indicador de ausencia con `Location`, mes, año y `RainTomorrow_bin`; los valores p se corrigen mediante Holm dentro de cada variable.


In [8]:
def cramers_v(contingency):
    if contingency.shape[0] < 2 or contingency.shape[1] < 2:
        return np.nan
    statistic = chi2_contingency(contingency, correction=False)[0]
    n = contingency.to_numpy().sum()
    rows, columns = contingency.shape
    denominator = min(rows - 1, columns - 1)
    return math.sqrt((statistic / n) / denominator) if denominator > 0 else np.nan

mechanism_rows = []
for variable in ANALYTIC_VARIABLES:
    indicator = train_raw[variable].isna().astype(int)
    tests = []
    for observed in ['Location', 'Month', 'Year', 'RainTomorrow_bin']:
        contingency = pd.crosstab(train_raw[observed], indicator)
        if contingency.shape[1] == 2:
            p_value = chi2_contingency(contingency, correction=False)[1]
            effect = cramers_v(contingency)
        else:
            p_value, effect = np.nan, np.nan
        tests.append({'observed': observed, 'p_raw': p_value, 'effect': effect})

    valid_indices = [index for index, item in enumerate(tests) if pd.notna(item['p_raw'])]
    if valid_indices:
        adjusted = multipletests([tests[index]['p_raw'] for index in valid_indices], method='holm')[1]
        for index, p_adjusted in zip(valid_indices, adjusted):
            tests[index]['p_holm'] = p_adjusted
    for item in tests:
        item.setdefault('p_holm', np.nan)

    evidence_against_mcar = any(
        pd.notna(item['p_holm']) and item['p_holm'] < ALPHA and
        pd.notna(item['effect']) and item['effect'] >= 0.05
        for item in tests
    )
    weak_association = any(
        pd.notna(item['p_holm']) and item['p_holm'] < ALPHA
        for item in tests
    )
    if evidence_against_mcar:
        conclusion = 'Evidencia contra MCAR; mecanismo compatible con MAR'
    elif weak_association:
        conclusion = 'Asociación estadística de magnitud débil; MAR plausible'
    else:
        conclusion = 'MCAR no rechazado por los diagnósticos aplicados'

    row = {
        'variable': variable,
        'n_faltantes_train': int(indicator.sum()),
        'porcentaje_faltantes_train': 100 * indicator.mean(),
        'conclusion_operativa': conclusion,
        'limitacion': 'MNAR no es identificable únicamente con la información observada.'
    }
    for item in tests:
        row[f"p_Holm_{item['observed']}"] = item['p_holm']
        row[f"V_Cramer_{item['observed']}"] = item['effect']
    mechanism_rows.append(row)

missing_mechanism = pd.DataFrame(mechanism_rows)
missing_mechanism.to_csv(TABLES / '03c_diagnostico_complementario_mecanismo_faltantes.csv', index=False)
pattern_summary = missing_mechanism['conclusion_operativa'].value_counts().rename_axis('conclusion').reset_index(name='n_variables')
pattern_summary.to_csv(TABLES / '03d_resumen_mecanismo_faltantes.csv', index=False)
missing_mechanism_summary = missing_mechanism[
    ['variable', 'porcentaje_faltantes_train', 'conclusion_operativa']
].copy()
display(missing_mechanism_summary)
display(pattern_summary)

missingness_statement = pd.DataFrame([{
    'conclusion': (
        'S1 cuantificó la ausencia. El diagnóstico complementario de S3 se realiza solo en entrenamiento. '
        'Para las variables con evidencia contra MCAR se adopta MAR como supuesto de trabajo, respaldado por '
        'asociaciones con variables observadas y no como certeza. MNAR no puede confirmarse ni descartarse sin '
        'información externa sobre el proceso de medición.'
    ),
    'supuesto_operativo': (
        'MAR como supuesto de trabajo únicamente cuando existe evidencia contra MCAR; '
        'MCAR no se rechaza en las variables sin evidencia suficiente.'
    ),
    'limitacion': 'MNAR no es identificable únicamente con los datos observados.'
}])
missingness_statement.to_csv(TABLES / '03e_conclusion_mecanismo_faltantes.csv', index=False)
display(missingness_statement)

,variable,porcentaje_faltantes_train,conclusion_operativa
0,MinTemp,0.460140,Evidencia contra MCAR; mecanismo compatible co...
1,MaxTemp,0.225046,Evidencia contra MCAR; mecanismo compatible co...
2,Rainfall,1.003667,Evidencia contra MCAR; mecanismo compatible co...
3,WindGustSpeed,6.592656,Evidencia contra MCAR; mecanismo compatible co...
4,Humidity9am,1.264882,Evidencia contra MCAR; mecanismo compatible co...
5,Humidity3pm,2.566936,Evidencia contra MCAR; mecanismo compatible co...
6,Pressure9am,9.891998,Evidencia contra MCAR; mecanismo compatible co...
7,Pressure3pm,9.847792,Evidencia contra MCAR; mecanismo compatible co...
8,Temp9am,0.640981,Evidencia contra MCAR; mecanismo compatible co...
9,Temp3pm,1.940021,Evidencia contra MCAR; mecanismo compatible co...


,conclusion,n_variables
0,Evidencia contra MCAR; mecanismo compatible co...,11


,conclusion,supuesto_operativo,limitacion
0,S1 cuantificó la ausencia. El diagnóstico comp...,MAR como supuesto de trabajo únicamente cuando...,MNAR no es identificable únicamente con los da...


### 2.3 Imputación mediante regresión lineal múltiple

Las diez variables numéricas retenidas se imputan mediante modelos explícitos de regresión lineal múltiple con tres a cinco predictores presentes en la matriz de S1. Los modelos se ajustan con observaciones completas de la variable a imputar y de sus predictores.

La imputación es secuencial y determinística. Los faltantes se inicializan con estadísticas de entrenamiento y luego se actualizan mediante predicciones puntuales. Los coeficientes se estiman una sola vez con los casos completos originales; los ciclos posteriores actualizan los valores imputados al cambiar los predictores previamente imputados. El procedimiento no corresponde a MICE.

`Rainfall` se modela en escala `log1p` y se retransmite mediante el factor de *smearing* de Duan calculado con residuos de entrenamiento. `RainToday_bin` se imputa mediante regresión logística. `RainTomorrow` no se utiliza como predictor de imputación.


In [9]:
IMPUTATION_SPECS = {
    'MinTemp': ['Temp9am', 'MaxTemp', 'Pressure3pm', 'Humidity9am', 'WindGustSpeed'],
    'MaxTemp': ['Temp3pm', 'MinTemp', 'Humidity3pm', 'Pressure3pm'],
    'Rainfall': ['Humidity3pm', 'Humidity9am', 'Pressure9am', 'WindGustSpeed', 'MaxTemp'],
    'WindGustSpeed': ['Pressure9am', 'Humidity9am', 'MinTemp', 'Temp9am', 'Rainfall'],
    'Humidity9am': ['Humidity3pm', 'MaxTemp', 'MinTemp', 'Pressure3pm', 'Rainfall'],
    'Humidity3pm': ['Humidity9am', 'Temp3pm', 'Rainfall', 'Pressure3pm'],
    'Pressure9am': ['Pressure3pm', 'WindGustSpeed', 'MinTemp', 'Humidity9am', 'Rainfall'],
    'Pressure3pm': ['Pressure9am', 'WindGustSpeed', 'MinTemp', 'Humidity9am', 'Rainfall'],
    'Temp9am': ['MinTemp', 'MaxTemp', 'Humidity9am', 'Pressure3pm', 'Rainfall'],
    'Temp3pm': ['MaxTemp', 'MinTemp', 'Humidity3pm', 'Pressure3pm'],
}
RAIN_TODAY_PREDICTORS = [
    'Rainfall', 'Humidity3pm', 'Humidity9am', 'MaxTemp', 'Pressure3pm'
]
TRANSFORMED_TARGETS = {
    'Rainfall': 'log1p'
}
DISCRETE_NUMERIC_TARGETS = set()
PHYSICAL_BOUNDS = {
    'Rainfall': (0.0, None),
    'WindGustSpeed': (0.0, None),
    'Humidity9am': (0.0, 100.0),
    'Humidity3pm': (0.0, 100.0),
}

expected_targets = {
    variable
    for variable in NUMERIC_IMPUTATION_TARGETS
    if analysis[variable].isna().any()
}
if set(IMPUTATION_SPECS) != expected_targets:
    raise AssertionError(
        'Cada variable numérica faltante del conjunto analítico debe poseer '
        'una especificación explícita. '
        f'Faltan={sorted(expected_targets - set(IMPUTATION_SPECS))}; '
        f'sobran={sorted(set(IMPUTATION_SPECS) - expected_targets)}.'
    )

stable_s2_variables = set(stable_predictors['variable'])
specification_rows = []
scope_control_rows = []
for target, predictors in IMPUTATION_SPECS.items():
    if not 3 <= len(predictors) <= 5:
        raise ValueError(f'{target} no tiene entre tres y cinco predictores.')
    if len(set(predictors)) != len(predictors):
        raise ValueError(f'{target} contiene predictores duplicados.')
    if target in predictors:
        raise ValueError(f'{target} no puede utilizarse como su propio predictor.')
    if target not in s1_corr.index:
        raise KeyError(
            f'{target} no aparece en la matriz de correlaciones de S1.'
        )

    available_candidates = [
        variable for variable in S1_CORE_NUMERIC
        if variable != target and variable in s1_corr.columns
    ]
    reference_series = s1_corr.loc[target, available_candidates]
    absolute_correlations = reference_series.abs().sort_values(ascending=False)
    correlation_rank = {
        variable: rank
        for rank, variable in enumerate(absolute_correlations.index, start=1)
    }

    missing_predictors_in_s1 = [
        predictor for predictor in predictors
        if predictor not in available_candidates
    ]
    if missing_predictors_in_s1:
        raise KeyError(
            f'Los predictores {missing_predictors_in_s1} de {target} no '
            'están disponibles en la matriz de S1.'
        )

    scope_control_rows.append({
        'variable_imputada': target,
        'variable_en_matriz_S1': True,
        'n_predictores': len(predictors),
        'todos_predictores_en_matriz_S1': True,
        'predictores': ', '.join(predictors),
        'fuente_exclusiva_seleccion': 'Matriz de correlaciones entregada en S1',
        'cumple_trazabilidad_S1': True
    })

    for predictor in predictors:
        correlation = float(reference_series[predictor])
        predictor_stable_s2 = predictor in stable_s2_variables
        specification_rows.append({
            'variable_imputada': target,
            'rol_variable': 'Variable analítica heredada de la matriz de S1',
            'predictor': predictor,
            'fuente_correlacion': 'Matriz de correlaciones entregada en S1',
            'correlacion_S1_par_imputacion': correlation,
            'correlacion_absoluta_S1': abs(correlation),
            'rango_correlacion_absoluta_entre_candidatas': correlation_rank[predictor],
            'predictor_validado_frente_RainTomorrow_en_S2': predictor_stable_s2,
            'fundamento': (
                'Par disponible en la matriz de S1; predictor además validado '
                'frente al objetivo en S2.'
                if predictor_stable_s2
                else 'Par disponible en la matriz de S1.'
            )
        })

imputation_specification = pd.DataFrame(specification_rows)
imputation_specification.to_csv(
    TABLES / '05_predictores_regresiones_imputacion.csv',
    index=False
)

scope_correlation_control = pd.DataFrame(scope_control_rows)
if not scope_correlation_control['cumple_trazabilidad_S1'].all():
    raise AssertionError(
        'Alguna regresión de imputación no está completamente respaldada '
        'por la matriz de correlaciones de S1.'
    )
scope_correlation_control.to_csv(
    TABLES / '05a_control_trazabilidad_matriz_correlacion_S1.csv',
    index=False
)

display(imputation_specification)
display(scope_correlation_control)


,variable_imputada,rol_variable,predictor,fuente_correlacion,correlacion_S1_par_imputacion,correlacion_absoluta_S1,rango_correlacion_absoluta_entre_candidatas,predictor_validado_frente_RainTomorrow_en_S2,fundamento
0,MinTemp,Variable analítica heredada de la matriz de S1,Temp9am,Matriz de correlaciones entregada en S1,0.902000,0.902000,1,False,Par disponible en la matriz de S1.
1,MinTemp,Variable analítica heredada de la matriz de S1,MaxTemp,Matriz de correlaciones entregada en S1,0.737000,0.737000,2,True,Par disponible en la matriz de S1; predictor a...
2,MinTemp,Variable analítica heredada de la matriz de S1,Pressure3pm,Matriz de correlaciones entregada en S1,-0.461000,0.461000,4,True,Par disponible en la matriz de S1; predictor a...
3,MinTemp,Variable analítica heredada de la matriz de S1,Humidity9am,Matriz de correlaciones entregada en S1,-0.233000,0.233000,6,False,Par disponible en la matriz de S1.
4,MinTemp,Variable analítica heredada de la matriz de S1,WindGustSpeed,Matriz de correlaciones entregada en S1,0.177000,0.177000,7,False,Par disponible en la matriz de S1.
5,MaxTemp,Variable analítica heredada de la matriz de S1,Temp3pm,Matriz de correlaciones entregada en S1,0.985000,0.985000,1,False,Par disponible en la matriz de S1.
6,MaxTemp,Variable analítica heredada de la matriz de S1,MinTemp,Matriz de correlaciones entregada en S1,0.737000,0.737000,3,False,Par disponible en la matriz de S1.
7,MaxTemp,Variable analítica heredada de la matriz de S1,Humidity3pm,Matriz de correlaciones entregada en S1,-0.509000,0.509000,4,True,Par disponible en la matriz de S1; predictor a...
8,MaxTemp,Variable analítica heredada de la matriz de S1,Pressure3pm,Matriz de correlaciones entregada en S1,-0.427000,0.427000,6,True,Par disponible en la matriz de S1; predictor a...
9,Rainfall,Variable analítica heredada de la matriz de S1,Humidity3pm,Matriz de correlaciones entregada en S1,0.256000,0.256000,1,True,Par disponible en la matriz de S1; predictor a...


,variable_imputada,variable_en_matriz_S1,n_predictores,todos_predictores_en_matriz_S1,predictores,fuente_exclusiva_seleccion,cumple_trazabilidad_S1
0,MinTemp,True,5,True,"Temp9am, MaxTemp, Pressure3pm, Humidity9am, Wi...",Matriz de correlaciones entregada en S1,True
1,MaxTemp,True,4,True,"Temp3pm, MinTemp, Humidity3pm, Pressure3pm",Matriz de correlaciones entregada en S1,True
2,Rainfall,True,5,True,"Humidity3pm, Humidity9am, Pressure9am, WindGus...",Matriz de correlaciones entregada en S1,True
3,WindGustSpeed,True,5,True,"Pressure9am, Humidity9am, MinTemp, Temp9am, Ra...",Matriz de correlaciones entregada en S1,True
4,Humidity9am,True,5,True,"Humidity3pm, MaxTemp, MinTemp, Pressure3pm, Ra...",Matriz de correlaciones entregada en S1,True
5,Humidity3pm,True,4,True,"Humidity9am, Temp3pm, Rainfall, Pressure3pm",Matriz de correlaciones entregada en S1,True
6,Pressure9am,True,5,True,"Pressure3pm, WindGustSpeed, MinTemp, Humidity9...",Matriz de correlaciones entregada en S1,True
7,Pressure3pm,True,5,True,"Pressure9am, WindGustSpeed, MinTemp, Humidity9...",Matriz de correlaciones entregada en S1,True
8,Temp9am,True,5,True,"MinTemp, MaxTemp, Humidity9am, Pressure3pm, Ra...",Matriz de correlaciones entregada en S1,True
9,Temp3pm,True,4,True,"MaxTemp, MinTemp, Humidity3pm, Pressure3pm",Matriz de correlaciones entregada en S1,True


In [10]:
def transform_target(values, target):
    values = np.asarray(values, dtype=float)
    if TRANSFORMED_TARGETS.get(target) == 'log1p':
        return np.log1p(np.clip(values, 0, None))
    return values


def inverse_target(values, target, smearing_factor=1.0):
    values = np.asarray(values, dtype=float)
    if TRANSFORMED_TARGETS.get(target) == 'log1p':
        return np.exp(values) * float(smearing_factor) - 1
    return values


def postprocess_imputed_values(values, target, observed):
    values = np.asarray(values, dtype=float)
    observed = np.asarray(observed, dtype=float)
    lower = float(np.nanmin(observed))
    upper = float(np.nanmax(observed))
    if target in PHYSICAL_BOUNDS:
        physical_lower, physical_upper = PHYSICAL_BOUNDS[target]
        if physical_lower is not None:
            lower = max(lower, physical_lower)
        if physical_upper is not None:
            upper = min(upper, physical_upper)
    processed = np.clip(values, lower, upper)
    if target in DISCRETE_NUMERIC_TARGETS:
        processed = np.rint(processed)
    return processed


def fit_numeric_imputation_models(training_data, specs, seed=SEED, diagnostics=True):
    models = {}
    diagnostic_rows = []
    coefficient_rows = []
    warning_rows = []
    rng = np.random.default_rng(seed)

    for target, predictors in specs.items():
        complete = (
            training_data[[target] + predictors + ['Location', 'Date']]
            .dropna(subset=[target] + predictors)
            .sort_values(['Location', 'Date'])
            .reset_index(drop=True)
        )
        if len(complete) < 500:
            raise ValueError(
                f'Muestra completa insuficiente para {target}: n={len(complete)}'
            )

        X = complete[predictors].astype(float)
        y_original = complete[target].astype(float).to_numpy()
        y_model = transform_target(y_original, target)
        linear_model = LinearRegression().fit(X, y_model)
        model_prediction = linear_model.predict(X)
        residuals = y_model - model_prediction
        smearing_factor = (
            float(np.mean(np.exp(residuals)))
            if TRANSFORMED_TARGETS.get(target) == 'log1p'
            else 1.0
        )
        ols_model = None

        if diagnostics:
            kfold = KFold(
                n_splits=N_CV,
                shuffle=True,
                random_state=seed
            )
            cv_predictions = np.full(len(complete), np.nan)
            for fit_index, validation_index in kfold.split(X):
                fold_model = LinearRegression().fit(
                    X.iloc[fit_index],
                    y_model[fit_index]
                )
                prediction_model_scale = fold_model.predict(
                    X.iloc[validation_index]
                )
                if TRANSFORMED_TARGETS.get(target) == 'log1p':
                    residual_fit = (
                        y_model[fit_index]
                        - fold_model.predict(X.iloc[fit_index])
                    )
                    fold_smearing = float(np.mean(np.exp(residual_fit)))
                else:
                    fold_smearing = 1.0
                prediction_original = inverse_target(
                    prediction_model_scale,
                    target,
                    smearing_factor=fold_smearing
                )
                cv_predictions[validation_index] = postprocess_imputed_values(
                    prediction_original,
                    target,
                    y_original
                )

            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                ols_model = sm.OLS(
                    y_model,
                    sm.add_constant(X, has_constant='add')
                ).fit()
            for warning in caught:
                warning_rows.append({
                    'modelo': target,
                    'advertencia': str(warning.message)
                })

            confidence = ols_model.conf_int(alpha=ALPHA)
            for term in ols_model.params.index:
                coefficient_rows.append({
                    'variable_imputada': target,
                    'termino': term,
                    'coeficiente': ols_model.params[term],
                    'error_estandar': ols_model.bse[term],
                    'p_value': ols_model.pvalues[term],
                    'IC95_li': confidence.loc[term, 0],
                    'IC95_ls': confidence.loc[term, 1]
                })

            diagnostic_n = min(20_000, len(X))
            diagnostic_index = rng.choice(
                len(X),
                size=diagnostic_n,
                replace=False
            )
            X_diag = X.iloc[diagnostic_index]
            y_diag = y_model[diagnostic_index]
            diag_model = sm.OLS(
                y_diag,
                sm.add_constant(X_diag, has_constant='add')
            ).fit()
            bp_p = het_breuschpagan(
                diag_model.resid,
                diag_model.model.exog
            )[1]
            jb_p = jarque_bera(diag_model.resid)[1]
            try:
                reset_p = float(
                    linear_reset(
                        diag_model,
                        power=2,
                        use_f=True
                    ).pvalue
                )
            except Exception:
                reset_p = np.nan
            X_vif = sm.add_constant(X_diag, has_constant='add')
            vifs = [
                variance_inflation_factor(X_vif.to_numpy(), index)
                for index in range(1, X_vif.shape[1])
            ]
            influence = diag_model.get_influence()
            max_cook = float(
                np.nanmax(influence.cooks_distance[0])
            )

            residual_ordered = np.asarray(
                ols_model.resid,
                dtype=float
            )
            dw_global = float(durbin_watson(residual_ordered))
            residual_frame = complete[['Location', 'Date']].copy()
            residual_frame['residuo_modelo'] = residual_ordered
            dw_by_location = []
            for _, location_group in residual_frame.groupby(
                'Location',
                sort=False
            ):
                if len(location_group) >= 30:
                    dw_by_location.append(
                        float(
                            durbin_watson(
                                location_group
                                .sort_values('Date')['residuo_modelo']
                                .to_numpy()
                            )
                        )
                    )
            if dw_by_location:
                dw_array = np.asarray(dw_by_location, dtype=float)
                dw_median = float(np.nanmedian(dw_array))
                dw_q1, dw_q3 = np.nanpercentile(dw_array, [25, 75])
                dw_outside = float(
                    np.mean((dw_array < 1.5) | (dw_array > 2.5))
                )
                n_locations_dw = int(np.isfinite(dw_array).sum())
            else:
                dw_median = np.nan
                dw_q1 = np.nan
                dw_q3 = np.nan
                dw_outside = np.nan
                n_locations_dw = 0

            diagnostic_rows.append({
                'variable': target,
                'rol_variable': (
                    'Clave S1/S2'
                    if target in S1_CORE_NUMERIC
                    else 'Adicional de cobertura completa'
                ),
                'transformacion_objetivo': TRANSFORMED_TARGETS.get(
                    target,
                    'ninguna'
                ),
                'factor_smearing_Duan': smearing_factor,
                'postprocesamiento_soporte': (
                    'Recorte al rango observado y redondeo entero'
                    if target in DISCRETE_NUMERIC_TARGETS
                    else 'Recorte al rango observado y límites físicos'
                ),
                'predictores': ', '.join(predictors),
                'n_completos_ajuste': len(complete),
                'RMSE_CV_escala_original': (
                    mean_squared_error(y_original, cv_predictions) ** 0.5
                ),
                'MAE_CV_escala_original': mean_absolute_error(
                    y_original,
                    cv_predictions
                ),
                'R2_CV_escala_original': (
                    1
                    - np.sum((y_original - cv_predictions) ** 2)
                    / np.sum((y_original - y_original.mean()) ** 2)
                ),
                'R2_ajustado_escala_modelo': float(
                    ols_model.rsquared_adj
                ),
                'Durbin_Watson_global_orden_Location_Date': dw_global,
                'Durbin_Watson_mediana_por_Location': dw_median,
                'Durbin_Watson_Q1_por_Location': float(dw_q1),
                'Durbin_Watson_Q3_por_Location': float(dw_q3),
                'n_Location_con_DW': n_locations_dw,
                'proporcion_Location_DW_fuera_1_5_2_5': dw_outside,
                'p_Breusch_Pagan': bp_p,
                'p_Jarque_Bera': jb_p,
                'p_RESET': reset_p,
                'VIF_max': float(np.nanmax(vifs)),
                'Cook_max_muestra_diagnostico': max_cook,
                'criterio_interpretacion': (
                    'Las pruebas de supuestos se interpretan junto con '
                    'validación cruzada y gráficos. Su incumplimiento limita '
                    'la inferencia clásica, pero no invalida automáticamente '
                    'la utilidad predictiva para imputación.'
                )
            })

        models[target] = {
            'model': linear_model,
            'ols': ols_model,
            'predictors': predictors,
            'residuals': residuals,
            'smearing_factor': smearing_factor,
            'observed': y_original,
            'complete_data': complete
        }

    warning_table = pd.DataFrame(
        warning_rows,
        columns=['modelo', 'advertencia']
    )
    return (
        models,
        pd.DataFrame(diagnostic_rows),
        pd.DataFrame(coefficient_rows),
        warning_table
    )


def fit_binary_imputation_model(training_data, seed=SEED, diagnostics=True):
    complete = training_data[
        ['RainToday_bin'] + RAIN_TODAY_PREDICTORS
    ].dropna().copy()
    X = complete[RAIN_TODAY_PREDICTORS].astype(float)
    y = complete['RainToday_bin'].astype(int)

    final_scaler = StandardScaler().fit(X)
    X_scaled = final_scaler.transform(X)
    final_model = LogisticRegression(
        penalty=None,
        solver='lbfgs',
        max_iter=2_000
    )
    with warnings.catch_warnings(record=True) as caught_final:
        warnings.simplefilter('always')
        final_model.fit(X_scaled, y)
    convergence_warnings = [
        item for item in caught_final
        if issubclass(item.category, ConvergenceWarning)
    ]
    if convergence_warnings:
        raise RuntimeError(
            'La regresión logística de imputación para RainToday_bin no '
            'convergió: '
            + ' | '.join(str(item.message) for item in convergence_warnings)
        )

    if diagnostics:
        folds = StratifiedKFold(
            n_splits=N_CV,
            shuffle=True,
            random_state=seed
        )
        probability = np.full(len(complete), np.nan)

        for fit_index, validation_index in folds.split(X, y):
            fold_scaler = StandardScaler().fit(X.iloc[fit_index])
            X_fit = fold_scaler.transform(X.iloc[fit_index])
            X_validation = fold_scaler.transform(
                X.iloc[validation_index]
            )
            model = LogisticRegression(
                penalty=None,
                solver='lbfgs',
                max_iter=2_000
            )
            with warnings.catch_warnings(record=True) as caught_fold:
                warnings.simplefilter('always')
                model.fit(X_fit, y.iloc[fit_index])
            convergence_warnings = [
                item for item in caught_fold
                if issubclass(item.category, ConvergenceWarning)
            ]
            if convergence_warnings:
                raise RuntimeError(
                    'La regresión logística de imputación no convergió en '
                    'un fold: '
                    + ' | '.join(
                        str(item.message)
                        for item in convergence_warnings
                    )
                )
            probability[validation_index] = model.predict_proba(
                X_validation
            )[:, 1]

        diagnostic_table = pd.DataFrame([{
            'variable': 'RainToday_bin',
            'modelo': 'regresión logística sin penalización',
            'predictores': ', '.join(RAIN_TODAY_PREDICTORS),
            'n_completos_ajuste': len(complete),
            'ROC_AUC_CV': roc_auc_score(y, probability),
            'PR_AUC_CV': average_precision_score(y, probability),
            'Brier_CV': brier_score_loss(y, probability)
        }])
    else:
        diagnostic_table = pd.DataFrame()

    return {
        'model': final_model,
        'scaler': final_scaler,
        'predictors': RAIN_TODAY_PREDICTORS,
        'complete_data': complete
    }, diagnostic_table


def chained_regression_imputation(
    training_data,
    other_data,
    numeric_specs,
    seed=SEED,
    cycles=N_IMPUTATION_CYCLES,
    diagnostics=True
):
    original_train = training_data.copy()
    working_train = training_data.copy()
    working_other = other_data.copy()

    (
        numeric_models,
        numeric_diagnostics,
        coefficient_table,
        numeric_warnings
    ) = fit_numeric_imputation_models(
        original_train,
        numeric_specs,
        seed=seed,
        diagnostics=diagnostics
    )
    binary_model, binary_diagnostics = fit_binary_imputation_model(
        original_train,
        seed=seed,
        diagnostics=diagnostics
    )

    original_missing_train = {
        variable: original_train[variable].isna()
        for variable in ANALYTIC_VARIABLES
    }
    original_missing_other = {
        variable: other_data[variable].isna()
        for variable in ANALYTIC_VARIABLES
    }

    for variable in NUMERIC_IMPUTATION_TARGETS:
        median = original_train[variable].median()
        working_train[variable] = working_train[variable].fillna(median)
        working_other[variable] = working_other[variable].fillna(median)
    mode = int(original_train['RainToday_bin'].mode().iloc[0])
    working_train['RainToday_bin'] = (
        working_train['RainToday_bin'].fillna(mode)
    )
    working_other['RainToday_bin'] = (
        working_other['RainToday_bin'].fillna(mode)
    )

    convergence_rows = []
    ordered_targets = sorted(
        numeric_specs,
        key=lambda variable: original_train[variable].isna().mean()
    )
    for cycle in range(1, cycles + 1):
        for target in ordered_targets:
            fitted = numeric_models[target]
            predictors = fitted['predictors']
            for dataset_name, data, mask in [
                ('train', working_train, original_missing_train[target]),
                ('otro', working_other, original_missing_other[target])
            ]:
                if not mask.any():
                    continue
                previous = data.loc[mask, target].to_numpy(dtype=float)
                prediction_scale = fitted['model'].predict(
                    data.loc[mask, predictors].astype(float)
                )
                prediction_original_raw = inverse_target(
                    prediction_scale,
                    target,
                    smearing_factor=fitted['smearing_factor']
                )
                prediction_original = postprocess_imputed_values(
                    prediction_original_raw,
                    target,
                    fitted['observed']
                )
                modified_by_support = ~np.isclose(
                    prediction_original_raw,
                    prediction_original,
                    rtol=1e-10,
                    atol=1e-10
                )
                data.loc[mask, target] = prediction_original
                convergence_rows.append({
                    'ciclo': cycle,
                    'conjunto': dataset_name,
                    'variable': target,
                    'n_actualizados': int(mask.sum()),
                    'cambio_absoluto_medio': float(
                        np.mean(np.abs(prediction_original - previous))
                    ),
                    'n_modificados_por_limites_o_redondeo': int(
                        modified_by_support.sum()
                    ),
                    'proporcion_modificada_por_limites_o_redondeo': float(
                        modified_by_support.mean()
                    )
                })

        for dataset_name, data, mask in [
            ('train', working_train, original_missing_train['RainToday_bin']),
            ('otro', working_other, original_missing_other['RainToday_bin'])
        ]:
            if not mask.any():
                continue
            binary_predictors = data.loc[
                mask,
                binary_model['predictors']
            ].astype(float)
            probability = binary_model['model'].predict_proba(
                binary_model['scaler'].transform(binary_predictors)
            )[:, 1]
            previous = data.loc[
                mask,
                'RainToday_bin'
            ].to_numpy(dtype=float)
            updated = (probability >= 0.5).astype(float)
            data.loc[mask, 'RainToday_bin'] = updated
            convergence_rows.append({
                'ciclo': cycle,
                'conjunto': dataset_name,
                'variable': 'RainToday_bin',
                'n_actualizados': int(mask.sum()),
                'cambio_absoluto_medio': float(
                    np.mean(np.abs(updated - previous))
                ),
                'n_modificados_por_limites_o_redondeo': 0,
                'proporcion_modificada_por_limites_o_redondeo': 0.0
            })

    counts = []
    for variable in ANALYTIC_VARIABLES:
        counts.append({
            'variable': variable,
            'metodo_principal': (
                'regresion_lineal_multiple_deterministica'
                if variable != 'RainToday_bin'
                else 'regresion_logistica_deterministica_umbral_0_50'
            ),
            'imputados_train': int(original_missing_train[variable].sum()),
            'imputados_otro': int(original_missing_other[variable].sum()),
            'imputados_total': int(
                original_missing_train[variable].sum()
                + original_missing_other[variable].sum()
            )
        })

    if (
        working_train[ANALYTIC_VARIABLES].isna().any().any()
        or working_other[ANALYTIC_VARIABLES].isna().any().any()
    ):
        raise AssertionError(
            'La imputación secuencial dejó valores faltantes en el conjunto '
            'analítico.'
        )

    return {
        'train': working_train,
        'other': working_other,
        'numeric_models': numeric_models,
        'binary_model': binary_model,
        'numeric_diagnostics': numeric_diagnostics,
        'binary_diagnostics': binary_diagnostics,
        'coefficients': coefficient_table,
        'warnings': numeric_warnings,
        'counts': pd.DataFrame(counts),
        'convergence': pd.DataFrame(convergence_rows)
    }


In [11]:
imputation_result = chained_regression_imputation(
    train_raw,
    test_raw,
    IMPUTATION_SPECS,
    seed=SEED,
    cycles=N_IMPUTATION_CYCLES,
    diagnostics=True
)
regression_train = imputation_result['train']
regression_test = imputation_result['other']
IMPUTATION_REGRESSION_MODE = (
    'secuencial_deterministica_con_smearing_Duan_en_objetivos_log1p'
)

imputation_result['numeric_diagnostics'].to_csv(
    TABLES / '06a_desempeno_y_supuestos_regresiones_imputacion.csv',
    index=False
)
imputation_result['binary_diagnostics'].to_csv(
    TABLES / '06b_desempeno_imputacion_RainToday.csv',
    index=False
)
imputation_result['coefficients'].to_csv(
    TABLES / '07_coeficientes_regresiones_imputacion.csv',
    index=False
)
imputation_result['counts'].to_csv(
    TABLES / '08a_cantidad_valores_imputados.csv',
    index=False
)
imputation_result['convergence'].to_csv(
    TABLES / '08b_convergencia_imputacion_secuencial.csv',
    index=False
)
imputation_result['warnings'].to_csv(
    TABLES / '08c_advertencias_modelos_imputacion.csv',
    index=False
)

numeric_diagnostics = imputation_result['numeric_diagnostics'].copy()
independence_columns = [
    'variable',
    'Durbin_Watson_global_orden_Location_Date',
    'Durbin_Watson_mediana_por_Location',
    'Durbin_Watson_Q1_por_Location',
    'Durbin_Watson_Q3_por_Location',
    'n_Location_con_DW',
    'proporcion_Location_DW_fuera_1_5_2_5'
]
imputation_independence = numeric_diagnostics[
    independence_columns
].copy()
imputation_independence.to_csv(
    TABLES / '06c_independencia_residuos_regresiones_imputacion.csv',
    index=False
)

assumption_evaluation = numeric_diagnostics[
    [
        'variable',
        'R2_CV_escala_original',
        'R2_ajustado_escala_modelo',
        'p_Breusch_Pagan',
        'p_Jarque_Bera',
        'p_RESET',
        'VIF_max',
        'Durbin_Watson_mediana_por_Location',
        'proporcion_Location_DW_fuera_1_5_2_5'
    ]
].copy()
assumption_evaluation['homocedasticidad_no_rechazada_5pct'] = (
    assumption_evaluation['p_Breusch_Pagan'] >= ALPHA
)
assumption_evaluation['normalidad_no_rechazada_5pct'] = (
    assumption_evaluation['p_Jarque_Bera'] >= ALPHA
)
assumption_evaluation['forma_lineal_no_rechazada_RESET_5pct'] = (
    assumption_evaluation['p_RESET'] >= ALPHA
)
assumption_evaluation['VIF_menor_5'] = (
    assumption_evaluation['VIF_max'] < 5
)
assumption_evaluation['independencia_aproximada_por_Location'] = (
    assumption_evaluation[
        'Durbin_Watson_mediana_por_Location'
    ].between(1.5, 2.5, inclusive='both')
    & (
        assumption_evaluation[
            'proporcion_Location_DW_fuera_1_5_2_5'
        ] <= 0.25
    )
)
assumption_evaluation['capacidad_predictiva_R2_CV'] = pd.cut(
    assumption_evaluation['R2_CV_escala_original'],
    bins=[-np.inf, 0.30, 0.50, 0.70, np.inf],
    labels=['baja', 'moderada', 'alta', 'muy_alta'],
    right=False
).astype(str)
assumption_evaluation['conclusion_operativa'] = assumption_evaluation.apply(
    lambda row: (
        'Los supuestos inferenciales no se consideran satisfechos en conjunto; '
        'el modelo se conserva exclusivamente para imputación predictiva, con '
        f'capacidad {row["capacidad_predictiva_R2_CV"]} según R2 CV y con '
        'interpretación cautelosa de sus coeficientes.'
        if not (
            row['homocedasticidad_no_rechazada_5pct']
            and row['normalidad_no_rechazada_5pct']
            and row['forma_lineal_no_rechazada_RESET_5pct']
            and row['independencia_aproximada_por_Location']
        )
        else (
            'Los diagnósticos aplicados no rechazan los supuestos principales; '
            'el modelo se utiliza para imputación predictiva y se valida mediante CV.'
        )
    ),
    axis=1
)
assumption_evaluation.to_csv(
    TABLES / '06d_evaluacion_explicita_supuestos_regresiones_imputacion.csv',
    index=False
)

support_adjustment = (
    imputation_result['convergence']
    .groupby(['conjunto', 'variable'], as_index=False)
    .agg(
        n_actualizados_por_ciclo=('n_actualizados', 'max'),
        proporcion_maxima_modificada=(
            'proporcion_modificada_por_limites_o_redondeo',
            'max'
        ),
        proporcion_ultimo_ciclo_modificada=(
            'proporcion_modificada_por_limites_o_redondeo',
            'last'
        )
    )
)
support_adjustment.to_csv(
    TABLES / '08e_postprocesamiento_predicciones_imputacion.csv',
    index=False
)

low_r2_models = numeric_diagnostics.loc[
    numeric_diagnostics['R2_CV_escala_original'] < 0.30,
    ['variable', 'R2_CV_escala_original']
]
problematic_independence = assumption_evaluation.loc[
    ~assumption_evaluation['independencia_aproximada_por_Location'],
    'variable'
].tolist()
heteroscedastic_models = assumption_evaluation.loc[
    ~assumption_evaluation['homocedasticidad_no_rechazada_5pct'],
    'variable'
].tolist()
non_normal_models = assumption_evaluation.loc[
    ~assumption_evaluation['normalidad_no_rechazada_5pct'],
    'variable'
].tolist()
functional_form_models = assumption_evaluation.loc[
    ~assumption_evaluation['forma_lineal_no_rechazada_RESET_5pct'],
    'variable'
].tolist()
rain_today_auc = float(
    imputation_result['binary_diagnostics']['ROC_AUC_CV'].iloc[0]
)

imputation_notes = pd.DataFrame([
    {
        'aspecto': 'Cobertura del conjunto analítico trazable a S1/S2',
        'resultado': (
            f'Se ajustaron {len(IMPUTATION_SPECS)} regresiones lineales '
            'múltiples explícitas: una por cada variable numérica con '
            'faltantes retenida en el conjunto analítico trazable a la '
            'matriz de S1. Las seis variables excluidas fueron contabilizadas '
            'y justificadas antes del tratamiento. RainToday_bin se imputó '
            'separadamente mediante regresión logística.'
        )
    },
    {
        'aspecto': 'Capacidad predictiva de las regresiones',
        'resultado': (
            'Modelos con R2 CV inferior a 0,30: '
            + ', '.join(
                f'{row.variable} ({row.R2_CV_escala_original:.3f})'
                for row in low_r2_models.itertuples()
            )
            if not low_r2_models.empty
            else 'Ninguna regresión presentó R2 CV inferior a 0,30.'
        )
    },
    {
        'aspecto': 'Supuestos inferenciales',
        'resultado': (
            f'Heterocedasticidad detectada en {len(heteroscedastic_models)} '
            f'modelos; no normalidad en {len(non_normal_models)}; '
            f'forma funcional rechazada por RESET en '
            f'{len(functional_form_models)}; independencia temporal '
            f'aproximada no satisfecha en {len(problematic_independence)}. '
            'Estos resultados limitan la inferencia clásica, por lo que las '
            'regresiones se justifican por su finalidad predictiva de '
            'imputación y su desempeño de validación cruzada.'
        )
    },
    {
        'aspecto': 'Retransmisión de objetivos log1p',
        'resultado': (
            'Rainfall utiliza el factor de smearing de Duan '
            'estimado con entrenamiento tanto en validación cruzada como en '
            'la imputación definitiva.'
        )
    },
    {
        'aspecto': 'Imputación de RainToday',
        'resultado': (
            f'El AUC CV obtenido fue {rain_today_auc:.4f}. '
            'RainTomorrow no se utiliza como predictor y el umbral de '
            'imputación es 0,50.'
        )
    }
])
imputation_notes.to_csv(
    TABLES / '08d_notas_interpretativas_imputacion.csv',
    index=False
)

display(numeric_diagnostics)
display(imputation_result['binary_diagnostics'])
display(imputation_result['counts'])
display(
    imputation_result['convergence']
    .groupby(['ciclo', 'conjunto'])[
        'cambio_absoluto_medio'
    ]
    .mean()
    .reset_index()
)
display(imputation_independence)
display(assumption_evaluation)
display(support_adjustment)
display(imputation_notes)


,variable,rol_variable,transformacion_objetivo,factor_smearing_Duan,postprocesamiento_soporte,predictores,n_completos_ajuste,RMSE_CV_escala_original,MAE_CV_escala_original,R2_CV_escala_original,R2_ajustado_escala_modelo,Durbin_Watson_global_orden_Location_Date,Durbin_Watson_mediana_por_Location,Durbin_Watson_Q1_por_Location,Durbin_Watson_Q3_por_Location,n_Location_con_DW,proporcion_Location_DW_fuera_1_5_2_5,p_Breusch_Pagan,p_Jarque_Bera,p_RESET,VIF_max,Cook_max_muestra_diagnostico,criterio_interpretacion
0,MinTemp,Clave S1/S2,ninguna,1.000000,Recorte al rango observado y límites físicos,"Temp9am, MaxTemp, Pressure3pm, Humidity9am, Wi...",84996,2.238459,1.757898,0.877044,0.877056,1.291296,1.324843,1.170956,1.440130,44,0.772727,0.000000,0.000000,0.000000,4.993194,0.004634,Las pruebas de supuestos se interpretan junto ...
1,MaxTemp,Clave S1/S2,ninguna,1.000000,Recorte al rango observado y límites físicos,"Temp3pm, MinTemp, Humidity3pm, Pressure3pm",87949,1.157471,0.774024,0.972134,0.972137,1.713857,1.715079,1.611619,1.805439,45,0.133333,0.000000,0.000000,0.000061,5.307952,0.010326,Las pruebas de supuestos se interpretan junto ...
2,Rainfall,Clave S1/S2,log1p,1.576160,Recorte al rango observado y límites físicos,"Humidity3pm, Humidity9am, Pressure9am, WindGus...",83754,8.004596,2.807514,0.101314,0.264758,1.663268,1.727385,1.592981,1.819075,44,0.136364,0.000000,0.000000,0.000000,2.164429,0.003731,Las pruebas de supuestos se interpretan junto ...
3,WindGustSpeed,Clave S1/S2,ninguna,1.000000,Recorte al rango observado y límites físicos,"Pressure9am, Humidity9am, MinTemp, Temp9am, Ra...",84374,11.443156,8.868415,0.277806,0.277903,1.355319,1.406430,1.283617,1.497760,44,0.750000,0.000000,0.000000,0.000000,9.050435,0.049227,Las pruebas de supuestos se interpretan junto ...
4,Humidity9am,Clave S1/S2,ninguna,1.000000,Recorte al rango observado y límites físicos,"Humidity3pm, MaxTemp, MinTemp, Pressure3pm, Ra...",87011,13.001639,10.240237,0.532035,0.530032,1.364668,1.433880,1.309241,1.523768,45,0.688889,0.000000,0.000000,0.000000,5.082221,0.061772,Las pruebas de supuestos se interpretan junto ...
5,Humidity3pm,Clave S1/S2,ninguna,1.000000,Recorte al rango observado y límites físicos,"Humidity9am, Temp3pm, Rainfall, Pressure3pm",87116,13.939990,11.321927,0.544781,0.544392,1.112488,1.228025,0.828162,1.584464,45,0.666667,0.000000,0.000000,0.000000,1.535830,0.022800,Las pruebas de supuestos se interpretan junto ...
6,Pressure9am,Clave S1/S2,ninguna,1.000000,Recorte al rango observado y límites físicos,"Pressure3pm, WindGustSpeed, MinTemp, Humidity9...",84278,1.842843,1.393072,0.932806,0.932815,1.800888,1.847050,1.717669,1.937197,44,0.204545,0.000000,0.000000,0.000000,1.470908,0.076158,Las pruebas de supuestos se interpretan junto ...
7,Pressure3pm,Clave S1/S2,ninguna,1.000000,Recorte al rango observado y límites físicos,"Pressure9am, WindGustSpeed, MinTemp, Humidity9...",84278,1.861954,1.403964,0.930088,0.930105,1.840272,1.907783,1.745750,1.989799,44,0.181818,0.000000,0.000000,0.000000,1.545253,0.072942,Las pruebas de supuestos se interpretan junto ...
8,Temp9am,Clave S1/S2,ninguna,1.000000,Recorte al rango observado y límites físicos,"MinTemp, MaxTemp, Humidity9am, Pressure3pm, Ra...",88050,1.626793,1.277156,0.935847,0.935847,1.187132,1.303979,0.941533,1.550300,45,0.622222,0.000000,0.000000,0.040771,3.004352,0.004104,Las pruebas de supuestos se interpretan junto ...
9,Temp3pm,Clave S1/S2,ninguna,1.000000,Recorte al rango observado y límites físicos,"MaxTemp, MinTemp, Humidity3pm, Pressure3pm",87949,1.089044,0.749890,0.974197,0.974201,1.697087,1.751944,1.610858,1.838850,45,0.155556,0.000000,0.000000,0.001368,4.976680,0.029395,Las pruebas de supuestos se interpretan junto ...


,variable,modelo,predictores,n_completos_ajuste,ROC_AUC_CV,PR_AUC_CV,Brier_CV
0,RainToday_bin,regresión logística sin penalización,"Rainfall, Humidity3pm, Humidity9am, MaxTemp, P...",87081,1.000000,1.000000,0.000023


,variable,metodo_principal,imputados_train,imputados_otro,imputados_total
0,MinTemp,regresion_lineal_multiple_deterministica,458,179,637
1,MaxTemp,regresion_lineal_multiple_deterministica,224,98,322
2,Rainfall,regresion_lineal_multiple_deterministica,999,407,1406
3,WindGustSpeed,regresion_lineal_multiple_deterministica,6562,2708,9270
4,Humidity9am,regresion_lineal_multiple_deterministica,1259,515,1774
5,Humidity3pm,regresion_lineal_multiple_deterministica,2555,1055,3610
6,Pressure9am,regresion_lineal_multiple_deterministica,9846,4168,14014
7,Pressure3pm,regresion_lineal_multiple_deterministica,9802,4179,13981
8,Temp9am,regresion_lineal_multiple_deterministica,638,266,904
9,Temp3pm,regresion_lineal_multiple_deterministica,1931,795,2726


,ciclo,conjunto,cambio_absoluto_medio
0,1,otro,3.900322
1,1,train,3.767262
2,2,otro,0.664761
3,2,train,0.692509
4,3,otro,0.296179
5,3,train,0.299812
6,4,otro,0.182098
7,4,train,0.178278
8,5,otro,0.128124
9,5,train,0.124069


,variable,Durbin_Watson_global_orden_Location_Date,Durbin_Watson_mediana_por_Location,Durbin_Watson_Q1_por_Location,Durbin_Watson_Q3_por_Location,n_Location_con_DW,proporcion_Location_DW_fuera_1_5_2_5
0,MinTemp,1.291296,1.324843,1.170956,1.440130,44,0.772727
1,MaxTemp,1.713857,1.715079,1.611619,1.805439,45,0.133333
2,Rainfall,1.663268,1.727385,1.592981,1.819075,44,0.136364
3,WindGustSpeed,1.355319,1.406430,1.283617,1.497760,44,0.750000
4,Humidity9am,1.364668,1.433880,1.309241,1.523768,45,0.688889
5,Humidity3pm,1.112488,1.228025,0.828162,1.584464,45,0.666667
6,Pressure9am,1.800888,1.847050,1.717669,1.937197,44,0.204545
7,Pressure3pm,1.840272,1.907783,1.745750,1.989799,44,0.181818
8,Temp9am,1.187132,1.303979,0.941533,1.550300,45,0.622222
9,Temp3pm,1.697087,1.751944,1.610858,1.838850,45,0.155556


,variable,R2_CV_escala_original,R2_ajustado_escala_modelo,p_Breusch_Pagan,p_Jarque_Bera,p_RESET,VIF_max,Durbin_Watson_mediana_por_Location,proporcion_Location_DW_fuera_1_5_2_5,homocedasticidad_no_rechazada_5pct,normalidad_no_rechazada_5pct,forma_lineal_no_rechazada_RESET_5pct,VIF_menor_5,independencia_aproximada_por_Location,capacidad_predictiva_R2_CV,conclusion_operativa
0,MinTemp,0.877044,0.877056,0.000000,0.000000,0.000000,4.993194,1.324843,0.772727,False,False,False,True,False,muy_alta,Los supuestos inferenciales no se consideran s...
1,MaxTemp,0.972134,0.972137,0.000000,0.000000,0.000061,5.307952,1.715079,0.133333,False,False,False,False,True,muy_alta,Los supuestos inferenciales no se consideran s...
2,Rainfall,0.101314,0.264758,0.000000,0.000000,0.000000,2.164429,1.727385,0.136364,False,False,False,True,True,baja,Los supuestos inferenciales no se consideran s...
3,WindGustSpeed,0.277806,0.277903,0.000000,0.000000,0.000000,9.050435,1.406430,0.750000,False,False,False,False,False,baja,Los supuestos inferenciales no se consideran s...
4,Humidity9am,0.532035,0.530032,0.000000,0.000000,0.000000,5.082221,1.433880,0.688889,False,False,False,False,False,alta,Los supuestos inferenciales no se consideran s...
5,Humidity3pm,0.544781,0.544392,0.000000,0.000000,0.000000,1.535830,1.228025,0.666667,False,False,False,True,False,alta,Los supuestos inferenciales no se consideran s...
6,Pressure9am,0.932806,0.932815,0.000000,0.000000,0.000000,1.470908,1.847050,0.204545,False,False,False,True,True,muy_alta,Los supuestos inferenciales no se consideran s...
7,Pressure3pm,0.930088,0.930105,0.000000,0.000000,0.000000,1.545253,1.907783,0.181818,False,False,False,True,True,muy_alta,Los supuestos inferenciales no se consideran s...
8,Temp9am,0.935847,0.935847,0.000000,0.000000,0.040771,3.004352,1.303979,0.622222,False,False,False,True,False,muy_alta,Los supuestos inferenciales no se consideran s...
9,Temp3pm,0.974197,0.974201,0.000000,0.000000,0.001368,4.976680,1.751944,0.155556,False,False,False,True,True,muy_alta,Los supuestos inferenciales no se consideran s...


,conjunto,variable,n_actualizados_por_ciclo,proporcion_maxima_modificada,proporcion_ultimo_ciclo_modificada
0,otro,Humidity3pm,1055,0.001896,0.001896
1,otro,Humidity9am,515,0.060194,0.060194
2,otro,MaxTemp,98,0.000000,0.000000
3,otro,MinTemp,179,0.000000,0.000000
4,otro,Pressure3pm,4179,0.000000,0.000000
5,otro,Pressure9am,4168,0.000000,0.000000
6,otro,RainToday_bin,407,0.000000,0.000000
7,otro,Rainfall,407,0.000000,0.000000
8,otro,Temp3pm,795,0.013836,0.013836
9,otro,Temp9am,266,0.015038,0.015038


,aspecto,resultado
0,Cobertura del conjunto analítico trazable a S1/S2,Se ajustaron 10 regresiones lineales múltiples...
1,Capacidad predictiva de las regresiones,"Modelos con R2 CV inferior a 0,30: Rainfall (0..."
2,Supuestos inferenciales,Heterocedasticidad detectada en 10 modelos; no...
3,Retransmisión de objetivos log1p,Rainfall utiliza el factor de smearing de Duan...
4,Imputación de RainToday,El AUC CV obtenido fue 1.0000. RainTomorrow no...


In [12]:
for target, fitted in imputation_result['numeric_models'].items():
    complete = fitted['complete_data']
    X = complete[fitted['predictors']].astype(float)
    y = transform_target(complete[target].astype(float), target)
    fitted_values = fitted['model'].predict(X)
    residuals = y - fitted_values
    sample_n = min(8_000, len(complete))
    sample_index = np.random.default_rng(SEED).choice(len(complete), sample_n, replace=False)

    fig, axes = plt.subplots(1, 2, figsize=(11, 4))
    axes[0].scatter(fitted_values[sample_index], residuals[sample_index], s=8, alpha=0.25)
    axes[0].axhline(0, linewidth=1)
    axes[0].set_title(f'{target}: residuos frente a valores ajustados')
    axes[0].set_xlabel('Valor ajustado')
    axes[0].set_ylabel('Residuo')
    stats.probplot(residuals[sample_index], dist='norm', plot=axes[1])
    axes[1].set_title(f'{target}: gráfico Q–Q')
    fig.tight_layout()
    fig.savefig(FIGURES / f'imputacion_diagnostico_{target}.png', dpi=170, bbox_inches='tight')
    plt.close(fig)

print(f'Diagnósticos gráficos generados para {len(imputation_result["numeric_models"])} regresiones.')

Diagnósticos gráficos generados para 10 regresiones.


### 2.4 Comparación de estrategias de tratamiento de faltantes

La comparación se realiza exclusivamente con entrenamiento y sobre el mismo alcance analítico:

1. casos completos;
2. imputación simple mediante mediana y moda de entrenamiento;
3. imputación secuencial determinística mediante regresiones explícitas.

Para cada estrategia se evalúan tamaño muestral, media, mediana, desviación estándar, encogimiento de varianza, distribución y preservación de las correlaciones de S1. La preservación se cuantifica mediante signo, desviación absoluta, razón de magnitudes y categoría de intensidad. La estrategia se selecciona mediante una regla multicriterio acompañada por un análisis de sensibilidad de sus ponderaciones.


In [13]:
COMPARISON_NUMERIC = NUMERIC_IMPUTATION_TARGETS.copy()
COMPARISON_ALL = COMPARISON_NUMERIC + ['RainToday_bin']

complete_case_train = train_raw.dropna(subset=COMPARISON_ALL).copy()
complete_case_test = test_raw.dropna(subset=COMPARISON_ALL).copy()

simple_train = train_raw.copy()
simple_test = test_raw.copy()
for variable in COMPARISON_NUMERIC:
    median = simple_train[variable].median()
    simple_train[variable] = simple_train[variable].fillna(median)
    simple_test[variable] = simple_test[variable].fillna(median)

mode = int(simple_train['RainToday_bin'].mode().iloc[0])
simple_train['RainToday_bin'] = (
    simple_train['RainToday_bin'].fillna(mode)
)
simple_test['RainToday_bin'] = (
    simple_test['RainToday_bin'].fillna(mode)
)

strategies = {
    'casos_completos': (complete_case_train, complete_case_test),
    'imputacion_simple': (simple_train, simple_test),
    'imputacion_regresion': (regression_train, regression_test),
}

size_rows = []
for name, (train_data, test_data) in strategies.items():
    size_rows.append({
        'estrategia': name,
        'n_train': len(train_data),
        'n_test_disponible': len(test_data),
        'porcentaje_train_retenido': (
            100 * len(train_data) / len(train_raw)
        )
    })
strategy_sizes = pd.DataFrame(size_rows)
strategy_sizes.to_csv(
    TABLES / '09_tamano_muestral_estrategias.csv',
    index=False
)

# Comparación de distribución respecto de los valores observados en entrenamiento.
distribution_rows = []
for name, (train_data, _) in strategies.items():
    for variable in COMPARISON_NUMERIC:
        reference = train_raw[variable].dropna().to_numpy(dtype=float)
        values = train_data[variable].dropna().to_numpy(dtype=float)
        if len(values) == 0:
            raise ValueError(
                f'La estrategia {name} no conserva valores para {variable}.'
            )
        iqr = np.subtract(*np.percentile(reference, [75, 25]))
        distribution_rows.append({
            'estrategia': name,
            'variable': variable,
            'media': values.mean(),
            'mediana': np.median(values),
            'desviacion_estandar': values.std(ddof=1),
            'asimetria': stats.skew(values, bias=False),
            'distancia_Wasserstein_normalizada_IQR': (
                wasserstein_distance(reference, values)
                / (iqr if iqr > 0 else 1.0)
            )
        })

distribution_comparison = pd.DataFrame(distribution_rows)
distribution_comparison.to_csv(
    TABLES / '10_preservacion_distribuciones_train.csv',
    index=False
)

# Para cada variable imputada se controla su relación con el objetivo y con
# el predictor principal. También se conserva el par categórico central S1/S2.
correlation_pairs = []
for variable in COMPARISON_NUMERIC:
    correlation_pairs.append((variable, 'RainTomorrow_bin'))
    correlation_pairs.append((variable, IMPUTATION_SPECS[variable][0]))
correlation_pairs.append(('RainToday_bin', 'RainTomorrow_bin'))
correlation_pairs = list(dict.fromkeys(correlation_pairs))


def reference_correlation(first, second):
    if first in s1_corr.index and second in s1_corr.columns:
        return float(s1_corr.loc[first, second]), 'Matriz de correlaciones de S1'
    if second in s1_corr.index and first in s1_corr.columns:
        return float(s1_corr.loc[second, first]), 'Matriz de correlaciones de S1'
    raise KeyError(
        f'El par {first}–{second} no está disponible en la matriz de S1. '
        'No se permite reconstruir la referencia en S3.'
    )


def correlation_magnitude_category(value):
    absolute = abs(float(value))
    if absolute < 0.10:
        return 'muy_debil'
    if absolute < 0.30:
        return 'debil'
    if absolute < 0.50:
        return 'moderada'
    if absolute < 0.70:
        return 'fuerte'
    return 'muy_fuerte'


correlation_rows = []
for name, (train_data, _) in strategies.items():
    for first, second in correlation_pairs:
        valid_pair = train_data[[first, second]].dropna()
        if len(valid_pair) < 3:
            raise ValueError(
                f'No hay observaciones suficientes para {first}–{second} '
                f'en la estrategia {name}.'
            )
        r_strategy = valid_pair.corr().iloc[0, 1]
        r_reference, reference_source = reference_correlation(first, second)
        correlation_rows.append({
            'estrategia': name,
            'variable_1': first,
            'variable_2': second,
            'fuente_referencia': reference_source,
            'r_referencia': r_reference,
            'r_S1': (
                r_reference
                if reference_source == 'Matriz de correlaciones de S1'
                else np.nan
            ),
            'r_train_estrategia': r_strategy,
            'signo_preservado': (
                np.sign(r_strategy) == np.sign(r_reference)
            ),
            'magnitud_absoluta_referencia': abs(r_reference),
            'magnitud_absoluta_estrategia': abs(r_strategy),
            'razon_magnitud_absoluta': (
                abs(r_strategy) / abs(r_reference)
                if not np.isclose(r_reference, 0.0)
                else np.nan
            ),
            'categoria_magnitud_referencia': correlation_magnitude_category(
                r_reference
            ),
            'categoria_magnitud_estrategia': correlation_magnitude_category(
                r_strategy
            ),
            'orden_magnitud_preservado': (
                correlation_magnitude_category(r_strategy)
                == correlation_magnitude_category(r_reference)
            ),
            'desviacion_absoluta': abs(r_strategy - r_reference)
        })

correlation_comparison = pd.DataFrame(correlation_rows)
correlation_comparison.to_csv(
    TABLES / '11_preservacion_correlaciones_train.csv',
    index=False
)

# Tabla integrada solicitada: tamaño, estadísticos descriptivos y
# correlación con el objetivo en una misma estructura.
integrated_rows = []

for variable in COMPARISON_ALL:
    observed = train_raw.loc[
        train_raw[variable].notna(),
        [variable, 'RainTomorrow_bin']
    ].dropna()
    integrated_rows.append({
        'estrategia': 'observado_disponible',
        'variable': variable,
        'n_estrategia': len(train_raw),
        'n_variable_valido': len(observed),
        'n_imputados_variable': 0,
        'media': observed[variable].mean(),
        'mediana': observed[variable].median(),
        'desviacion_estandar': observed[variable].std(ddof=1),
        'correlacion_con_RainTomorrow': (
            observed[[variable, 'RainTomorrow_bin']]
            .corr().iloc[0, 1]
            if variable != 'RainTomorrow_bin'
            else 1.0
        )
    })

for name, (train_data, _) in strategies.items():
    for variable in COMPARISON_ALL:
        valid = train_data[[variable, 'RainTomorrow_bin']].dropna()
        integrated_rows.append({
            'estrategia': name,
            'variable': variable,
            'n_estrategia': len(train_data),
            'n_variable_valido': len(valid),
            'n_imputados_variable': (
                int(train_raw[variable].isna().sum())
                if name in {
                    'imputacion_simple',
                    'imputacion_regresion'
                }
                else 0
            ),
            'media': valid[variable].mean(),
            'mediana': valid[variable].median(),
            'desviacion_estandar': valid[variable].std(ddof=1),
            'correlacion_con_RainTomorrow': (
                valid[[variable, 'RainTomorrow_bin']]
                .corr().iloc[0, 1]
                if variable != 'RainTomorrow_bin'
                else 1.0
            )
        })

integrated_imputation_comparison = pd.DataFrame(integrated_rows)
observed_sd_reference = (
    integrated_imputation_comparison
    .loc[
        integrated_imputation_comparison['estrategia'].eq(
            'observado_disponible'
        ),
        ['variable', 'desviacion_estandar']
    ]
    .rename(columns={
        'desviacion_estandar': 'desviacion_estandar_observada'
    })
)
integrated_imputation_comparison = (
    integrated_imputation_comparison
    .merge(observed_sd_reference, on='variable', how='left')
)
integrated_imputation_comparison['razon_DE_vs_observado'] = (
    integrated_imputation_comparison['desviacion_estandar']
    / integrated_imputation_comparison['desviacion_estandar_observada']
)
integrated_imputation_comparison['encogimiento_varianza_porcentual'] = (
    100
    * (
        1
        - integrated_imputation_comparison['razon_DE_vs_observado'] ** 2
    )
)
integrated_imputation_comparison.to_csv(
    TABLES / '10b_comparacion_integrada_imputacion.csv',
    index=False
)

correlation_preservation_summary = (
    correlation_comparison
    .groupby('estrategia')
    .agg(
        desviacion_correlacion_media=('desviacion_absoluta', 'mean'),
        proporcion_signo_preservado=('signo_preservado', 'mean'),
        proporcion_orden_magnitud_preservado=(
            'orden_magnitud_preservado', 'mean'
        )
    )
)

strategy_summary = strategy_sizes.merge(
    distribution_comparison.groupby('estrategia')[
        'distancia_Wasserstein_normalizada_IQR'
    ].mean().rename('distancia_distribucion_media'),
    on='estrategia'
).merge(
    correlation_preservation_summary,
    on='estrategia'
)

strategy_summary['rango_retencion'] = (
    strategy_summary['porcentaje_train_retenido']
    .rank(method='min', ascending=False)
)
strategy_summary['rango_distribucion'] = (
    strategy_summary['distancia_distribucion_media']
    .rank(method='min', ascending=True)
)
strategy_summary['rango_correlacion'] = (
    strategy_summary['desviacion_correlacion_media']
    .rank(method='min', ascending=True)
)

WEIGHT_SCENARIOS = {
    'principal_40_30_30': (0.40, 0.30, 0.30),
    'equilibrada_tercios': (1 / 3, 1 / 3, 1 / 3),
    'prioriza_retencion_50_25_25': (0.50, 0.25, 0.25),
    'prioriza_preservacion_25_375_375': (0.25, 0.375, 0.375),
}
weight_sensitivity_rows = []
for scenario, (
    weight_retention,
    weight_distribution,
    weight_correlation
) in WEIGHT_SCENARIOS.items():
    scenario_table = strategy_summary.copy()
    scenario_table['puntaje_decision'] = (
        weight_retention * scenario_table['rango_retencion']
        + weight_distribution * scenario_table['rango_distribucion']
        + weight_correlation * scenario_table['rango_correlacion']
    )
    scenario_table = scenario_table.sort_values(
        [
            'puntaje_decision',
            'desviacion_correlacion_media',
            'distancia_distribucion_media',
            'porcentaje_train_retenido'
        ],
        ascending=[True, True, True, False]
    )
    selected_scenario = scenario_table.iloc[0]['estrategia']
    for row in scenario_table.itertuples():
        weight_sensitivity_rows.append({
            'escenario_ponderacion': scenario,
            'peso_retencion': weight_retention,
            'peso_distribucion': weight_distribution,
            'peso_correlacion': weight_correlation,
            'estrategia': row.estrategia,
            'puntaje_decision': row.puntaje_decision,
            'seleccionada': row.estrategia == selected_scenario
        })

weight_sensitivity = pd.DataFrame(weight_sensitivity_rows)
weight_sensitivity.to_csv(
    TABLES / '12b_sensibilidad_ponderaciones_seleccion_imputacion.csv',
    index=False
)

principal_weights = WEIGHT_SCENARIOS['principal_40_30_30']
strategy_summary['puntaje_decision'] = (
    principal_weights[0] * strategy_summary['rango_retencion']
    + principal_weights[1] * strategy_summary['rango_distribucion']
    + principal_weights[2] * strategy_summary['rango_correlacion']
)
SELECTED_IMPUTATION = strategy_summary.sort_values(
    [
        'puntaje_decision',
        'desviacion_correlacion_media',
        'distancia_distribucion_media',
        'porcentaje_train_retenido'
    ],
    ascending=[True, True, True, False]
).iloc[0]['estrategia']
strategy_summary['seleccionada'] = (
    strategy_summary['estrategia'].eq(SELECTED_IMPUTATION)
)
selected_by_scenario = (
    weight_sensitivity.loc[weight_sensitivity['seleccionada']]
    ['estrategia']
)
if len(selected_by_scenario) != len(WEIGHT_SCENARIOS):
    raise AssertionError(
        'Cada escenario de ponderación debe seleccionar exactamente una estrategia.'
    )
selection_stable_across_scenarios = bool(
    selected_by_scenario.nunique() == 1
    and selected_by_scenario.iloc[0] == SELECTED_IMPUTATION
)
strategy_summary['seleccion_robusta_en_todos_los_escenarios'] = (
    strategy_summary['estrategia'].eq(SELECTED_IMPUTATION)
    & selection_stable_across_scenarios
)
if strategy_summary['seleccion_robusta_en_todos_los_escenarios'].sum() != 1:
    raise AssertionError(
        'Solo la estrategia seleccionada puede quedar marcada como robusta en '
        'todos los escenarios de ponderación.'
    )
strategy_summary.to_csv(
    TABLES / '12_sintesis_seleccion_estrategia_imputacion.csv',
    index=False
)

display(
    integrated_imputation_comparison[
        [
            'estrategia',
            'variable',
            'n_estrategia',
            'n_variable_valido',
            'n_imputados_variable',
            'media',
            'mediana',
            'desviacion_estandar',
            'razon_DE_vs_observado',
            'encogimiento_varianza_porcentual',
            'correlacion_con_RainTomorrow'
        ]
    ].reset_index(drop=True)
)
display(strategy_summary)
display(weight_sensitivity)
display(Markdown(
    f'**Estrategia seleccionada con entrenamiento:** '
    f'`{SELECTED_IMPUTATION}`.'
))


,estrategia,variable,n_estrategia,n_variable_valido,n_imputados_variable,media,mediana,desviacion_estandar,razon_DE_vs_observado,encogimiento_varianza_porcentual,correlacion_con_RainTomorrow
0,observado_disponible,MinTemp,99535,99077,0,12.192028,12.000000,6.402115,1.000000,0.000000,0.083182
1,observado_disponible,MaxTemp,99535,99311,0,23.230502,22.600000,7.115120,1.000000,0.000000,-0.160305
2,observado_disponible,Rainfall,99535,98536,0,2.350140,0.000000,8.461160,1.000000,0.000000,0.241823
3,observado_disponible,WindGustSpeed,99535,92973,0,39.998075,39.000000,13.601413,1.000000,0.000000,0.236575
4,observado_disponible,Humidity9am,99535,98276,0,68.851622,70.000000,19.052473,1.000000,0.000000,0.255636
5,observado_disponible,Humidity3pm,99535,96980,0,51.491039,52.000000,20.784875,1.000000,0.000000,0.447155
6,observado_disponible,Pressure9am,99535,89689,0,"1,017.641565","1,017.600000",7.121478,1.000000,0.000000,-0.248307
7,observado_disponible,Pressure3pm,99535,89733,0,"1,015.246682","1,015.200000",7.057691,1.000000,0.000000,-0.227122
8,observado_disponible,Temp9am,99535,98897,0,16.996100,16.700000,6.491363,1.000000,0.000000,-0.026020
9,observado_disponible,Temp3pm,99535,97604,0,21.689804,21.100000,6.935750,1.000000,0.000000,-0.194393


,estrategia,n_train,n_test_disponible,porcentaje_train_retenido,distancia_distribucion_media,desviacion_correlacion_media,proporcion_signo_preservado,proporcion_orden_magnitud_preservado,rango_retencion,rango_distribucion,rango_correlacion,puntaje_decision,seleccionada,seleccion_robusta_en_todos_los_escenarios
0,casos_completos,83683,36019,84.073944,0.020352,0.003181,1.000000,1.000000,3.000000,2.000000,2.000000,2.400000,False,False
1,imputacion_simple,99535,42658,100.000000,0.022731,0.006543,1.000000,1.000000,1.000000,3.000000,3.000000,2.200000,False,False
2,imputacion_regresion,99535,42658,100.000000,0.016386,0.002314,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,True,True


,escenario_ponderacion,peso_retencion,peso_distribucion,peso_correlacion,estrategia,puntaje_decision,seleccionada
0,principal_40_30_30,0.400000,0.300000,0.300000,imputacion_regresion,1.000000,True
1,principal_40_30_30,0.400000,0.300000,0.300000,imputacion_simple,2.200000,False
2,principal_40_30_30,0.400000,0.300000,0.300000,casos_completos,2.400000,False
3,equilibrada_tercios,0.333333,0.333333,0.333333,imputacion_regresion,1.000000,True
4,equilibrada_tercios,0.333333,0.333333,0.333333,casos_completos,2.333333,False
5,equilibrada_tercios,0.333333,0.333333,0.333333,imputacion_simple,2.333333,False
6,prioriza_retencion_50_25_25,0.500000,0.250000,0.250000,imputacion_regresion,1.000000,True
7,prioriza_retencion_50_25_25,0.500000,0.250000,0.250000,imputacion_simple,2.000000,False
8,prioriza_retencion_50_25_25,0.500000,0.250000,0.250000,casos_completos,2.500000,False
9,prioriza_preservacion_25_375_375,0.250000,0.375000,0.375000,imputacion_regresion,1.000000,True


**Estrategia seleccionada con entrenamiento:** `imputacion_regresion`.

In [14]:
fig, ax = plt.subplots(figsize=(9, 5))
(
    distribution_comparison
    .groupby('estrategia')[
        'distancia_Wasserstein_normalizada_IQR'
    ]
    .mean()
    .sort_values()
    .plot.bar(ax=ax)
)
ax.set_ylabel('Distancia de Wasserstein media / IQR')
ax.set_title('Preservación de distribuciones en entrenamiento')
fig.tight_layout()
fig.savefig(
    FIGURES / 'fig_01_preservacion_distribuciones_train.png',
    dpi=170
)
plt.close(fig)

# Comparación visual entre datos observados y valores imputados.
variables_with_missing = [
    variable
    for variable in COMPARISON_NUMERIC
    if train_raw[variable].isna().any()
]
n_columns = 2
n_rows = math.ceil(len(variables_with_missing) / n_columns)
fig, axes = plt.subplots(
    n_rows,
    n_columns,
    figsize=(14, 3.6 * n_rows),
    squeeze=False
)

for axis, variable in zip(axes.ravel(), variables_with_missing):
    observed_values = train_raw[variable].dropna()
    missing_mask = train_raw[variable].isna()
    simple_imputed_values = simple_train.loc[
        missing_mask, variable
    ]
    regression_imputed_values = regression_train.loc[
        missing_mask, variable
    ]

    lower, upper = observed_values.quantile([0.005, 0.995])
    bins = np.linspace(lower, upper, 30)

    axis.hist(
        observed_values.clip(lower, upper),
        bins=bins,
        density=True,
        alpha=0.45,
        label='Observado'
    )
    axis.hist(
        simple_imputed_values.clip(lower, upper),
        bins=bins,
        density=True,
        alpha=0.45,
        label='Imputación simple'
    )
    axis.hist(
        regression_imputed_values.clip(lower, upper),
        bins=bins,
        density=True,
        alpha=0.45,
        label='Imputación por regresión'
    )
    axis.set_title(variable)
    axis.set_ylabel('Densidad')
    axis.legend(fontsize=8)

for axis in axes.ravel()[len(variables_with_missing):]:
    axis.axis('off')

fig.suptitle(
    'Distribución observada y valores imputados en entrenamiento',
    y=1.002
)
fig.tight_layout()
fig.savefig(
    FIGURES / 'fig_01b_observado_vs_imputado.png',
    dpi=170,
    bbox_inches='tight'
)
plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
(
    correlation_comparison
    .groupby('estrategia')['desviacion_absoluta']
    .mean()
    .sort_values()
    .plot.bar(ax=ax)
)
ax.set_ylabel('Desviación absoluta media respecto de la referencia')
ax.set_title('Preservación de correlaciones de referencia en entrenamiento')
fig.tight_layout()
fig.savefig(
    FIGURES / 'fig_02_preservacion_correlaciones_train.png',
    dpi=170
)
plt.close(fig)


## 3. Preparación para clasificación

S1 identificó asimetría y observaciones extremas mediante estadística descriptiva y boxplots; S2 evaluó su robustez mediante filtro IQR, winsorización y comparación de medianas. El modelo principal conserva los extremos meteorológicamente plausibles, transforma `Rainfall` con `log1p` y complementa el análisis con una sensibilidad de winsorización 1–99 % calculada solo en entrenamiento.

Las variables continuas se estandarizan con parámetros de entrenamiento. `RainToday` se codifica como binaria. El mes se transforma en `Season`; `Summer` es la categoría de referencia y las dummies `Season_Autumn`, `Season_Winter` y `Season_Spring` se evalúan como un bloque. `WindDir9am` y `Location` se excluyen del modelo principal por alta cardinalidad; `Location` se analiza por separado como sensibilidad.


In [15]:
SEASON_REFERENCE = 'Summer'
SEASON_LEVELS = ['Summer', 'Autumn', 'Winter', 'Spring']
SEASON_DUMMY_FEATURES = [
    'Season_Autumn',
    'Season_Winter',
    'Season_Spring'
]

def add_engineered_features(data):
    result = data.copy()
    result['Month'] = pd.to_datetime(result['Date']).dt.month

    season_map = {
        12: 'Summer', 1: 'Summer', 2: 'Summer',
        3: 'Autumn', 4: 'Autumn', 5: 'Autumn',
        6: 'Winter', 7: 'Winter', 8: 'Winter',
        9: 'Spring', 10: 'Spring', 11: 'Spring'
    }
    result['Season'] = result['Month'].map(season_map)

    if result['Season'].isna().any():
        raise ValueError('No fue posible asignar Season a todos los registros.')

    for level in SEASON_LEVELS:
        if level == SEASON_REFERENCE:
            continue
        result[f'Season_{level}'] = (
            result['Season'].eq(level).astype(int)
        )

    result['Rainfall_log1p'] = np.log1p(
        result['Rainfall'].clip(lower=0)
    )
    return result


selected_train_raw, selected_test_raw = strategies[SELECTED_IMPUTATION]
selected_train = add_engineered_features(selected_train_raw)
selected_test = add_engineered_features(selected_test_raw)

outlier_rows = []
for variable in MODEL_NUMERIC:
    q1, q3 = selected_train[variable].quantile([0.25, 0.75])
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outlier_rows.append({
        'variable': variable,
        'limite_inferior_IQR_train': lower,
        'limite_superior_IQR_train': upper,
        'n_fuera_IQR_train': int(
            (
                (selected_train[variable] < lower)
                | (selected_train[variable] > upper)
            ).sum()
        ),
        'decision_principal': (
            'Conservar extremos plausibles; transformar Rainfall '
            'y evaluar sensibilidad 1–99 %.'
        ),
        'evidencia_S2': (
            'Validación directa en S2: la diferencia de Humidity3pm '
            'fue estable frente a filtro IQR, winsorización y mediana.'
            if variable == 'Humidity3pm'
            else (
                'S2 no validó individualmente esta variable; se conserva '
                'por plausibilidad meteorológica y se incluye en la '
                'sensibilidad multivariable.'
            )
        )
    })

outlier_table = pd.DataFrame(outlier_rows)
outlier_table.to_csv(
    TABLES / '13_diagnostico_y_decision_outliers.csv',
    index=False
)
display(outlier_table)


outlier_traceability = pd.DataFrame([
    {
        'fase': 'S1',
        'evidencia': str(s1_descriptive_path.relative_to(ROOT)),
        'aporte': (
            'Asimetría, curtosis, IQR y rangos de variables meteorológicas '
            'priorizadas.'
        )
    },
    {
        'fase': 'S1',
        'evidencia': str(s1_boxplot_path.relative_to(ROOT)),
        'aporte': (
            'Inspección visual de dispersión y observaciones extremas por '
            'categoría de RainTomorrow.'
        )
    },
    {
        'fase': 'S2',
        'evidencia': str(s2_outliers_path.relative_to(ROOT)),
        'aporte': (
            'Diagnóstico IQR y cuantificación de observaciones extremas del '
            'contraste principal.'
        )
    },
    {
        'fase': 'S2',
        'evidencia': str(s2_robustness_path.relative_to(ROOT)),
        'aporte': (
            'Estabilidad del hallazgo frente a filtro IQR, winsorización y '
            'medianas.'
        )
    },
    {
        'fase': 'S3',
        'evidencia': '38_sensibilidad_winsorizacion.csv',
        'aporte': (
            'Decisión principal de conservar extremos plausibles y evaluar '
            'sensibilidad predictiva 1–99 %.'
        )
    }
])
outlier_traceability.to_csv(
    TABLES / '13a_trazabilidad_outliers_S1_S2.csv',
    index=False
)
display(outlier_traceability)


,variable,limite_inferior_IQR_train,limite_superior_IQR_train,n_fuera_IQR_train,decision_principal,evidencia_S2
0,MinTemp,-6.200000,30.600000,43,Conservar extremos plausibles; transformar Rai...,S2 no validó individualmente esta variable; se...
1,MaxTemp,2.450000,43.650000,330,Conservar extremos plausibles; transformar Rai...,S2 no validó individualmente esta variable; se...
2,Rainfall,-1.200000,2.000000,18016,Conservar extremos plausibles; transformar Rai...,S2 no validó individualmente esta variable; se...
3,WindGustSpeed,8.500000,68.500000,3814,Conservar extremos plausibles; transformar Rai...,S2 no validó individualmente esta variable; se...
4,Humidity9am,18.000000,122.000000,993,Conservar extremos plausibles; transformar Rai...,S2 no validó individualmente esta variable; se...
5,Humidity3pm,-5.301370,107.502283,0,Conservar extremos plausibles; transformar Rai...,Validación directa en S2: la diferencia de Hum...
6,Pressure9am,"1,001.050000","1,034.250000",1915,Conservar extremos plausibles; transformar Rai...,S2 no validó individualmente esta variable; se...
7,Pressure3pm,998.400000,"1,032.000000",1656,Conservar extremos plausibles; transformar Rai...,S2 no validó individualmente esta variable; se...
8,Temp9am,-1.650000,35.550000,184,Conservar extremos plausibles; transformar Rai...,S2 no validó individualmente esta variable; se...
9,Temp3pm,1.750000,41.350000,494,Conservar extremos plausibles; transformar Rai...,S2 no validó individualmente esta variable; se...


,fase,evidencia,aporte
0,S1,semana1\docs\tables\05_estadistica_descriptiva...,"Asimetría, curtosis, IQR y rangos de variables..."
1,S1,semana1\docs\figures\fig_04_boxplots_por_rain_...,Inspección visual de dispersión y observacione...
2,S2,semana2\docs\tables\08b_diagnostico_outliers_i...,Diagnóstico IQR y cuantificación de observacio...
3,S2,semana2\docs\tables\09b_sintesis_robustez.csv,"Estabilidad del hallazgo frente a filtro IQR, ..."
4,S3,38_sensibilidad_winsorizacion.csv,Decisión principal de conservar extremos plaus...


## 4. Tres modelos de regresión logística

- **M1 — S1/S2:** especificación fija con las cinco relaciones priorizadas en S1 y validadas en S2.
- **M2 — stepwise AIC:** selección *forward* por bloques, con incorporación solo cuando disminuye el AIC y detención cuando ningún candidato mejora el criterio.
- **M3 — BIC parsimonioso:** búsqueda exhaustiva de subconjuntos de bloques y selección del menor BIC.

`Season` se codifica con tres dummies y se trata como un único bloque. La selección se ajusta con `statsmodels.Logit` sin ponderación; `class_weight='balanced'` se reserva para la evaluación predictiva.

Para reducir redundancia, el universo automático conserva `Pressure3pm` en lugar de `Pressure9am`, `MaxTemp` en lugar de `Temp3pm` y excluye `Temp9am` por su alta correlación con `MinTemp` y `MaxTemp`. Estas decisiones se documentan antes de ejecutar stepwise o BIC.


In [16]:
M1_FEATURES = [
    'Humidity3pm',
    'RainToday_bin',
    'Rainfall_log1p',
    'Pressure3pm',
    'MaxTemp'
]

FEATURE_BLOCKS = {
    'Humidity3pm': ['Humidity3pm'],
    'RainToday_bin': ['RainToday_bin'],
    'Rainfall_log1p': ['Rainfall_log1p'],
    'Pressure3pm': ['Pressure3pm'],
    'MaxTemp': ['MaxTemp'],
    'MinTemp': ['MinTemp'],
    'WindGustSpeed': ['WindGustSpeed'],
    'Humidity9am': ['Humidity9am'],
    'Season': SEASON_DUMMY_FEATURES,
}

CANDIDATE_FEATURES = [
    feature
    for block in FEATURE_BLOCKS.values()
    for feature in block
]

BINARY_FEATURES = [
    'RainToday_bin',
    *SEASON_DUMMY_FEATURES
]
CONTINUOUS_FEATURES = [
    feature
    for feature in CANDIDATE_FEATURES
    if feature not in BINARY_FEATURES
]

scaler = StandardScaler().fit(
    selected_train[CONTINUOUS_FEATURES]
)
X_train = (
    selected_train[CANDIDATE_FEATURES]
    .copy()
    .reset_index(drop=True)
)
X_train[CONTINUOUS_FEATURES] = scaler.transform(
    selected_train[CONTINUOUS_FEATURES]
)
y_train = (
    selected_train['RainTomorrow_bin']
    .astype(int)
    .reset_index(drop=True)
)

scale_reference = pd.DataFrame({
    'variable': CONTINUOUS_FEATURES,
    'media_train_original': scaler.mean_,
    'desviacion_estandar_train_original': scaler.scale_
})
scale_reference.to_csv(
    TABLES / '13b_parametros_estandarizacion_train.csv',
    index=False
)

# Tratamiento explícito de variables categóricas.
categorical_decisions = pd.DataFrame([
    {
        'variable': 'Season',
        'n_niveles_train': int(selected_train['Season'].nunique()),
        'decision': 'Incluir en selección',
        'codificacion': (
            'Dummies Season_Autumn, Season_Winter y Season_Spring; '
            'referencia=Summer'
        ),
        'fundamento': (
            'Baja cardinalidad y representación directa de la '
            'estacionalidad solicitada en la pauta.'
        )
    },
    {
        'variable': 'WindDir9am',
        'n_niveles_train': int(
            selected_train['WindDir9am'].nunique(dropna=True)
        ),
        'decision': 'Excluir del modelo principal',
        'codificacion': 'No aplica',
        'fundamento': (
            'Alta cardinalidad, niveles poco poblados y ausencia de '
            'validación prioritaria en S1/S2.'
        )
    },
    {
        'variable': 'Location',
        'n_niveles_train': int(
            selected_train['Location'].nunique(dropna=True)
        ),
        'decision': 'Excluir del modelo principal; evaluar sensibilidad',
        'codificacion': (
            'One-hot con categoría de referencia en el análisis '
            'complementario'
        ),
        'fundamento': (
            'Alta cardinalidad y riesgo de expansión innecesaria de '
            'parámetros; su efecto se revisa en sensibilidad.'
        )
    }
])
categorical_decisions.to_csv(
    TABLES / '13d_tratamiento_variables_categoricas.csv',
    index=False
)
display(categorical_decisions)

candidate_evidence_rows = []
stable_s2_variables = set(stable_predictors['variable'])
m1_source_variables = {
    'Humidity3pm',
    'RainToday_bin',
    'Rainfall',
    'Pressure3pm',
    'MaxTemp'
}
missing_m1_validation = (
    m1_source_variables - stable_s2_variables
)
if missing_m1_validation:
    raise ValueError(
        'M1 requiere relaciones estables de S2 que no fueron '
        f'encontradas: {sorted(missing_m1_validation)}'
    )

for block, features in FEATURE_BLOCKS.items():
    for feature in features:
        if feature in SEASON_DUMMY_FEATURES:
            source_variable = 'Season'
        elif feature == 'Rainfall_log1p':
            source_variable = 'Rainfall'
        else:
            source_variable = feature

        if (
            source_variable in s1_corr.index
            and 'RainTomorrow_bin' in s1_corr.columns
        ):
            correlation_s1_target = float(
                s1_corr.loc[
                    source_variable,
                    'RainTomorrow_bin'
                ]
            )
        else:
            correlation_s1_target = np.nan

        stable_s2 = (
            source_variable in stable_s2_variables
        )

        if source_variable == 'Season':
            foundation = (
                'Variable categórica de baja cardinalidad incorporada '
                'para demostrar codificación dummy y evaluar señal '
                'estacional dentro de los procedimientos automáticos.'
            )
        elif stable_s2:
            foundation = (
                'Relación prioritaria validada mediante bootstrap en S2.'
            )
        else:
            foundation = (
                'Variable candidata derivada de la matriz y del contexto '
                'meteorológico de S1; su aporte multivariable se evalúa '
                'mediante selección.'
            )

        candidate_evidence_rows.append({
            'bloque': block,
            'variable_modelo': feature,
            'variable_origen': source_variable,
            'correlacion_S1_con_RainTomorrow': correlation_s1_target,
            'validada_estable_en_S2': stable_s2,
            'fundamento': foundation
        })

candidate_evidence = pd.DataFrame(candidate_evidence_rows)
candidate_evidence.to_csv(
    TABLES / '13c_trazabilidad_variables_candidatas_S1_S2.csv',
    index=False
)
display(candidate_evidence)

included_numeric_candidates = {
    'Humidity3pm', 'Rainfall', 'Pressure3pm', 'MaxTemp',
    'MinTemp', 'WindGustSpeed', 'Humidity9am'
}
excluded_numeric_candidates = {
    'Pressure9am', 'Temp9am', 'Temp3pm'
}
if (
    included_numeric_candidates | excluded_numeric_candidates
) != set(S1_CORE_NUMERIC):
    raise AssertionError(
        'La delimitación de predictoras logísticas no contabiliza las diez '
        'variables numéricas de la matriz de S1.'
    )
if included_numeric_candidates & excluded_numeric_candidates:
    raise AssertionError(
        'Una predictora numérica no puede estar incluida y excluida a la vez.'
    )

predictor_scope_rows = []
for variable in S1_CORE_NUMERIC:
    historical_redundancy = np.nan
    recalculated_redundancy = np.nan
    redundancy_source = 'No aplica'
    if variable in included_numeric_candidates:
        decision = 'Incluir como bloque candidato'
        retained_by = variable
        rationale = (
            'Variable disponible en la matriz de S1; su aporte ajustado se '
            'evalúa mediante stepwise AIC y búsqueda BIC.'
        )
        if variable in stable_s2_variables:
            rationale += ' Su relación con RainTomorrow fue estable en S2.'
    elif variable == 'Pressure9am':
        decision = 'Excluir antes de la selección automática'
        retained_by = 'Pressure3pm'
        historical_redundancy = float(
            s1_corr.loc['Pressure9am', 'Pressure3pm']
        )
        recalculated_redundancy = recalculated_pair_correlation(
            'Pressure9am', 'Pressure3pm'
        )
        redundancy_source = (
            'Recálculo S3 sobre casos completos de la base original; '
            'corroboración posterior mediante VIF en entrenamiento'
        )
        rationale = (
            f'Redundancia muy alta con Pressure3pm '
            f'(r recalculado={recalculated_redundancy:.4f}); se conserva '
            'Pressure3pm porque fue validada explícitamente en S2.'
        )
    elif variable == 'Temp3pm':
        decision = 'Excluir antes de la selección automática'
        retained_by = 'MaxTemp'
        historical_redundancy = float(s1_corr.loc['Temp3pm', 'MaxTemp'])
        recalculated_redundancy = recalculated_pair_correlation(
            'Temp3pm', 'MaxTemp'
        )
        redundancy_source = (
            'Recálculo S3 sobre casos completos de la base original; '
            'corroboración posterior mediante VIF en entrenamiento'
        )
        rationale = (
            f'Redundancia casi perfecta con MaxTemp '
            f'(r recalculado={recalculated_redundancy:.4f}); se conserva '
            'MaxTemp porque fue validada explícitamente en S2.'
        )
    else:  # Temp9am
        decision = 'Excluir antes de la selección automática'
        retained_by = 'MinTemp y MaxTemp'
        historical_redundancy = float(max(
            abs(s1_corr.loc['Temp9am', 'MinTemp']),
            abs(s1_corr.loc['Temp9am', 'MaxTemp'])
        ))
        recalculated_redundancy = float(max(
            abs(recalculated_pair_correlation('Temp9am', 'MinTemp')),
            abs(recalculated_pair_correlation('Temp9am', 'MaxTemp'))
        ))
        redundancy_source = (
            'Recálculo S3 sobre casos completos de la base original; '
            'corroboración posterior mediante VIF en entrenamiento'
        )
        rationale = (
            f'Alta redundancia con MinTemp y MaxTemp '
            f'(máximo |r| recalculado={recalculated_redundancy:.4f}); ambas '
            'permanecen disponibles y MaxTemp fue validada en S2.'
        )

    predictor_scope_rows.append({
        'variable_S1': variable,
        'correlacion_S1_con_RainTomorrow': float(
            s1_corr.loc[variable, 'RainTomorrow_bin']
        ),
        'validada_estable_en_S2': variable in stable_s2_variables,
        'decision_universo_logistico': decision,
        'representada_o_retenida_por': retained_by,
        'correlacion_redundancia_relevante_S1': historical_redundancy,
        'correlacion_redundancia_recalculada_base_completa_S3': recalculated_redundancy,
        'fuente_decision_redundancia': redundancy_source,
        'fundamento': rationale
    })

predictor_scope = pd.DataFrame(predictor_scope_rows)
predictor_scope.to_csv(
    TABLES / '13e_delimitacion_predictoras_modelos_logisticos.csv',
    index=False
)
display(predictor_scope)


fit_cache = {}
selection_fit_cache = {}
fit_warning_rows = []

def design_matrix(X_data, features):
    if features:
        return sm.add_constant(
            X_data[list(features)],
            has_constant='add'
        )
    return pd.DataFrame(
        {'const': np.ones(len(X_data), dtype=float)},
        index=X_data.index
    )


def fit_glm_features(
    features,
    X_data=X_train,
    y_data=y_train
):
    key = (tuple(features), id(X_data), id(y_data))
    if key in fit_cache:
        return fit_cache[key]

    design = design_matrix(X_data, features)
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        result = sm.GLM(
            y_data,
            design,
            family=sm.families.Binomial()
        ).fit(maxiter=200, disp=0)

    for warning in caught:
        fit_warning_rows.append({
            'modelo': (
                ', '.join(features)
                if features
                else 'intercepto'
            ),
            'advertencia': str(warning.message)
        })

    if not result.converged:
        raise RuntimeError(
            'No convergió el modelo con variables: '
            f'{features if features else ["intercepto"]}.'
        )

    fit_cache[key] = result
    return result


def fit_logit_selection(
    features,
    use_cache=True
):
    key = tuple(features)
    if use_cache and key in selection_fit_cache:
        return selection_fit_cache[key]

    design = design_matrix(
        X_train,
        features
    )
    with warnings.catch_warnings(
        record=True
    ) as caught:
        warnings.simplefilter('always')
        result = sm.Logit(
            y_train,
            design
        ).fit(
            method='lbfgs',
            maxiter=200,
            disp=0
        )

    for warning in caught:
        fit_warning_rows.append({
            'modelo': (
                ', '.join(features)
                if features
                else 'intercepto'
            ),
            'advertencia': str(
                warning.message
            )
        })

    converged = bool(
        result.mle_retvals.get(
            'converged',
            False
        )
    )
    if not converged:
        raise RuntimeError(
            'No convergió el modelo Logit de selección con '
            f'variables: {features if features else ["intercepto"]}.'
        )

    if use_cache:
        selection_fit_cache[key] = result
    return result


def flatten_blocks(block_names):
    return [
        feature
        for block in block_names
        for feature in FEATURE_BLOCKS[block]
    ]


def forward_stepwise_aic(
    blocks,
    min_aic_improvement=1e-6
):
    selected = []
    history = []
    iteration = 0

    current_result = fit_logit_selection([])
    current_aic = float(current_result.aic)

    while True:
        iteration += 1
        candidate_rows = []

        for block in blocks:
            if block in selected:
                continue

            candidate_blocks = selected + [block]
            candidate_features = flatten_blocks(
                candidate_blocks
            )
            candidate_result = fit_logit_selection(
                candidate_features
            )
            candidate_aic = float(candidate_result.aic)

            candidate_rows.append({
                'iteracion': iteration,
                'bloques_antes': (
                    ', '.join(selected)
                    if selected
                    else 'intercepto'
                ),
                'bloque_candidato': block,
                'AIC_antes': current_aic,
                'AIC_candidato': candidate_aic,
                'mejora_AIC': current_aic - candidate_aic,
                'aceptado_en_iteracion': False
            })

        if not candidate_rows:
            break

        best_index = int(
            np.argmax([
                row['mejora_AIC']
                for row in candidate_rows
            ])
        )
        best = candidate_rows[best_index]
        accepted = (
            best['mejora_AIC']
            > min_aic_improvement
        )

        if accepted:
            candidate_rows[best_index][
                'aceptado_en_iteracion'
            ] = True
            selected.append(
                best['bloque_candidato']
            )
            current_aic = best['AIC_candidato']

        for row in candidate_rows:
            row['decision_iteracion'] = (
                'incorporar'
                if row['aceptado_en_iteracion']
                else (
                    'detener'
                    if not accepted
                    and row['bloque_candidato']
                    == best['bloque_candidato']
                    else 'no_seleccionado'
                )
            )
            row['bloques_despues'] = (
                ', '.join(selected)
                if selected
                else 'intercepto'
            )
            history.append(row)

        if not accepted:
            break

    return (
        flatten_blocks(selected),
        pd.DataFrame(history)
    )


def exhaustive_bic(blocks):
    rows = []
    block_names = list(blocks)

    for subset_size in range(
        0,
        len(block_names) + 1
    ):
        for subset in combinations(
            block_names,
            subset_size
        ):
            features = flatten_blocks(
                list(subset)
            )
            result = fit_logit_selection(
                features,
                use_cache=False
            )

            bic = (
                -2 * result.llf
                + np.log(result.nobs)
                * len(result.params)
            )
            rows.append({
                'bloques': (
                    ', '.join(subset)
                    if subset
                    else 'intercepto'
                ),
                'variables': (
                    ', '.join(features)
                    if features
                    else 'intercepto'
                ),
                'n_bloques': len(subset),
                'n_parametros': len(
                    result.params
                ),
                'logLik': result.llf,
                'AIC': result.aic,
                'BIC': bic,
                'convergio': bool(
                    result.converged
                )
            })

            del result

    table = (
        pd.DataFrame(rows)
        .sort_values(['BIC', 'AIC'])
        .reset_index(drop=True)
    )

    best_blocks = (
        []
        if table.iloc[0]['bloques']
        == 'intercepto'
        else table.iloc[0]['bloques']
        .split(', ')
    )

    return (
        flatten_blocks(best_blocks),
        table
    )


M2_FEATURES, stepwise_history = (
    forward_stepwise_aic(
        list(FEATURE_BLOCKS)
    )
)
M3_FEATURES, bic_search = (
    exhaustive_bic(FEATURE_BLOCKS)
)

if not M2_FEATURES or not M3_FEATURES:
    raise RuntimeError(
        'Un procedimiento de selección '
        'no retuvo predictores.'
    )

stepwise_history.to_csv(
    TABLES / '14_historial_stepwise_AIC.csv',
    index=False
)
bic_search.to_csv(
    TABLES / '15_busqueda_exhaustiva_BIC.csv',
    index=False
)

selection_warning_table = pd.DataFrame(
    fit_warning_rows,
    columns=['modelo', 'advertencia']
)
selection_warning_table.to_csv(
    TABLES / '15b_advertencias_ajuste_seleccion.csv',
    index=False
)

MODEL_DEFINITIONS = {
    'M1_S1_S2': M1_FEATURES,
    'M2_stepwise_AIC': M2_FEATURES,
    'M3_BIC_parsimonioso': M3_FEATURES,
}

model_variables = pd.DataFrame([
    {
        'modelo': name,
        'n_variables': len(features),
        'variables': ', '.join(features)
    }
    for name, features
    in MODEL_DEFINITIONS.items()
])
model_variables.to_csv(
    TABLES / '16_variables_tres_modelos.csv',
    index=False
)
display(model_variables)

stepwise_path_display = stepwise_history.loc[
    stepwise_history['aceptado_en_iteracion']
    | stepwise_history['decision_iteracion'].eq('detener'),
    [
        'iteracion',
        'bloque_candidato',
        'AIC_antes',
        'AIC_candidato',
        'mejora_AIC',
        'decision_iteracion',
        'bloques_despues'
    ]
].reset_index(drop=True)
display(stepwise_path_display)


,variable,n_niveles_train,decision,codificacion,fundamento
0,Season,4,Incluir en selección,"Dummies Season_Autumn, Season_Winter y Season_...",Baja cardinalidad y representación directa de ...
1,WindDir9am,16,Excluir del modelo principal,No aplica,"Alta cardinalidad, niveles poco poblados y aus..."
2,Location,49,Excluir del modelo principal; evaluar sensibil...,One-hot con categoría de referencia en el anál...,Alta cardinalidad y riesgo de expansión innece...


,bloque,variable_modelo,variable_origen,correlacion_S1_con_RainTomorrow,validada_estable_en_S2,fundamento
0,Humidity3pm,Humidity3pm,Humidity3pm,0.446000,True,Relación prioritaria validada mediante bootstr...
1,RainToday_bin,RainToday_bin,RainToday_bin,0.313000,True,Relación prioritaria validada mediante bootstr...
2,Rainfall_log1p,Rainfall_log1p,Rainfall,0.239000,True,Relación prioritaria validada mediante bootstr...
3,Pressure3pm,Pressure3pm,Pressure3pm,-0.226000,True,Relación prioritaria validada mediante bootstr...
4,MaxTemp,MaxTemp,MaxTemp,-0.159000,True,Relación prioritaria validada mediante bootstr...
5,MinTemp,MinTemp,MinTemp,0.084000,False,Variable candidata derivada de la matriz y del...
6,WindGustSpeed,WindGustSpeed,WindGustSpeed,0.234000,False,Variable candidata derivada de la matriz y del...
7,Humidity9am,Humidity9am,Humidity9am,0.257000,False,Variable candidata derivada de la matriz y del...
8,Season,Season_Autumn,Season,NaN,False,Variable categórica de baja cardinalidad incor...
9,Season,Season_Winter,Season,NaN,False,Variable categórica de baja cardinalidad incor...


,variable_S1,correlacion_S1_con_RainTomorrow,validada_estable_en_S2,decision_universo_logistico,representada_o_retenida_por,correlacion_redundancia_relevante_S1,correlacion_redundancia_recalculada_base_completa_S3,fuente_decision_redundancia,fundamento
0,MinTemp,0.084000,False,Incluir como bloque candidato,MinTemp,NaN,NaN,No aplica,Variable disponible en la matriz de S1; su apo...
1,MaxTemp,-0.159000,True,Incluir como bloque candidato,MaxTemp,NaN,NaN,No aplica,Variable disponible en la matriz de S1; su apo...
2,Rainfall,0.239000,True,Incluir como bloque candidato,Rainfall,NaN,NaN,No aplica,Variable disponible en la matriz de S1; su apo...
3,WindGustSpeed,0.234000,False,Incluir como bloque candidato,WindGustSpeed,NaN,NaN,No aplica,Variable disponible en la matriz de S1; su apo...
4,Humidity9am,0.257000,False,Incluir como bloque candidato,Humidity9am,NaN,NaN,No aplica,Variable disponible en la matriz de S1; su apo...
5,Humidity3pm,0.446000,True,Incluir como bloque candidato,Humidity3pm,NaN,NaN,No aplica,Variable disponible en la matriz de S1; su apo...
6,Pressure9am,-0.246000,False,Excluir antes de la selección automática,Pressure3pm,0.961000,0.961326,Recálculo S3 sobre casos completos de la base ...,Redundancia muy alta con Pressure3pm (r recalc...
7,Pressure3pm,-0.226000,True,Incluir como bloque candidato,Pressure3pm,NaN,NaN,No aplica,Variable disponible en la matriz de S1; su apo...
8,Temp9am,-0.026000,False,Excluir antes de la selección automática,MinTemp y MaxTemp,0.902000,0.901821,Recálculo S3 sobre casos completos de la base ...,Alta redundancia con MinTemp y MaxTemp (máximo...
9,Temp3pm,-0.192000,False,Excluir antes de la selección automática,MaxTemp,0.985000,0.984503,Recálculo S3 sobre casos completos de la base ...,Redundancia casi perfecta con MaxTemp (r recal...


,modelo,n_variables,variables
0,M1_S1_S2,5,"Humidity3pm, RainToday_bin, Rainfall_log1p, Pr..."
1,M2_stepwise_AIC,11,"Humidity3pm, WindGustSpeed, Pressure3pm, Rainf..."
2,M3_BIC_parsimonioso,10,"Humidity3pm, RainToday_bin, Rainfall_log1p, Pr..."


,iteracion,bloque_candidato,AIC_antes,AIC_candidato,mejora_AIC,decision_iteracion,bloques_despues
0,1,Humidity3pm,"105,937.260830","83,879.574174","22,057.686656",incorporar,Humidity3pm
1,2,WindGustSpeed,"83,879.574174","78,199.929167","5,679.645007",incorporar,"Humidity3pm, WindGustSpeed"
2,3,Pressure3pm,"78,199.929167","75,644.771830","2,555.157337",incorporar,"Humidity3pm, WindGustSpeed, Pressure3pm"
3,4,Rainfall_log1p,"75,644.771830","74,585.501825","1,059.270005",incorporar,"Humidity3pm, WindGustSpeed, Pressure3pm, Rainf..."
4,5,Season,"74,585.501825","74,231.388970",354.112855,incorporar,"Humidity3pm, WindGustSpeed, Pressure3pm, Rainf..."
5,6,MaxTemp,"74,231.388970","74,169.281380",62.107590,incorporar,"Humidity3pm, WindGustSpeed, Pressure3pm, Rainf..."
6,7,RainToday_bin,"74,169.281380","74,128.810890",40.470490,incorporar,"Humidity3pm, WindGustSpeed, Pressure3pm, Rainf..."
7,8,Humidity9am,"74,128.810890","74,097.027292",31.783599,incorporar,"Humidity3pm, WindGustSpeed, Pressure3pm, Rainf..."
8,9,MinTemp,"74,097.027292","74,094.777085",2.250207,incorporar,"Humidity3pm, WindGustSpeed, Pressure3pm, Rainf..."


In [17]:
def model_coefficient_table(
    result,
    model_name,
    features
):
    confidence = result.conf_int(
        alpha=ALPHA
    )
    table = pd.DataFrame({
        'modelo': model_name,
        'termino': result.params.index,
        'beta': result.params.values,
        'error_estandar': result.bse.values,
        'p_value': result.pvalues.values,
        'IC95_beta_li': confidence[0].values,
        'IC95_beta_ls': confidence[1].values,
        'odds_ratio': np.exp(
            result.params.values
        ),
        'IC95_OR_li': np.exp(
            confidence[0].values
        ),
        'IC95_OR_ls': np.exp(
            confidence[1].values
        ),
    })

    scale_map = (
        scale_reference
        .set_index('variable')[
            'desviacion_estandar_train_original'
        ]
        .to_dict()
    )

    unit_map = {
        'Humidity3pm': 'puntos porcentuales',
        'Humidity9am': 'puntos porcentuales',
        'Pressure3pm': 'hPa',
        'MaxTemp': '°C',
        'MinTemp': '°C',
        'WindGustSpeed': 'km/h',
        'Rainfall_log1p': 'unidades log1p(mm)'
    }

    binary_contrasts = {
        'RainToday_bin': (
            'RainToday=Yes respecto de RainToday=No'
        ),
        'Season_Autumn': (
            'Autumn respecto de Summer'
        ),
        'Season_Winter': (
            'Winter respecto de Summer'
        ),
        'Season_Spring': (
            'Spring respecto de Summer'
        ),
    }

    def interpretation_unit(term):
        if term == 'const':
            return 'intercepto'
        if term in binary_contrasts:
            return binary_contrasts[term]
        return (
            f'1 DE = {scale_map.get(term, np.nan):.4f} '
            f'{unit_map.get(term, "unidades")}'
        )

    table['unidad_interpretacion'] = (
        table['termino'].map(
            interpretation_unit
        )
    )
    table['cambio_porcentual_odds'] = (
        100 * (
            table['odds_ratio'] - 1
        )
    )

    table['interpretacion'] = table.apply(
        lambda row: (
            'Intercepto; no se interpreta de forma aislada.'
            if row['termino'] == 'const'
            else (
                f"Manteniendo constantes los demás predictores, "
                f"{row['unidad_interpretacion']} se asocia con "
                f"un cambio de "
                f"{row['cambio_porcentual_odds']:.2f}% en las "
                f"odds de lluvia. IC95% del OR "
                f"[{row['IC95_OR_li']:.4f}, "
                f"{row['IC95_OR_ls']:.4f}]."
            )
        ),
        axis=1
    )
    return table


fitted_models = {}
coefficient_tables = []
information_rows = []

for model_name, features in (
    MODEL_DEFINITIONS.items()
):
    result = fit_glm_features(features)
    fitted_models[model_name] = result
    coefficient_tables.append(
        model_coefficient_table(
            result,
            model_name,
            features
        )
    )
    information_rows.append({
        'modelo': model_name,
        'logLik': result.llf,
        'AIC': result.aic,
        'BIC': (
            -2 * result.llf
            + np.log(result.nobs)
            * len(result.params)
        ),
        'n_parametros': len(
            result.params
        ),
        'convergio': bool(
            result.converged
        ),
        'iteraciones': (
            result.fit_history.get(
                'iteration',
                np.nan
            )
        )
    })

coefficients_three_models = pd.concat(
    coefficient_tables,
    ignore_index=True
)
information_criteria = pd.DataFrame(
    information_rows
)

coefficients_three_models.to_csv(
    TABLES / '17_coeficientes_odds_ratios_tres_modelos.csv',
    index=False
)
information_criteria.to_csv(
    TABLES / '18_AIC_BIC_convergencia_tres_modelos.csv',
    index=False
)

display(coefficients_three_models)
display(
    information_criteria.sort_values(
        'BIC'
    )
)


,modelo,termino,beta,error_estandar,p_value,IC95_beta_li,IC95_beta_ls,odds_ratio,IC95_OR_li,IC95_OR_ls,unidad_interpretacion,cambio_porcentual_odds,interpretacion
0,M1_S1_S2,const,-1.796526,0.014429,0.000000,-1.824807,-1.768245,0.165874,0.161249,0.170632,intercepto,-83.412590,Intercepto; no se interpreta de forma aislada.
1,M1_S1_S2,Humidity3pm,1.217549,0.012315,0.000000,1.193413,1.241686,3.378897,3.298318,3.461445,1 DE = 20.6545 puntos porcentuales,237.889706,"Manteniendo constantes los demás predictores, ..."
2,M1_S1_S2,RainToday_bin,0.256884,0.038133,0.000000,0.182144,0.331623,1.292895,1.199787,1.393228,RainToday=Yes respecto de RainToday=No,29.289467,"Manteniendo constantes los demás predictores, ..."
3,M1_S1_S2,Rainfall_log1p,0.227057,0.016088,0.000000,0.195524,0.258590,1.254902,1.215949,1.295103,1 DE = 0.9029 unidades log1p(mm),25.490172,"Manteniendo constantes los demás predictores, ..."
4,M1_S1_S2,Pressure3pm,-0.665432,0.010456,0.000000,-0.685927,-0.644938,0.514051,0.503623,0.524695,1 DE = 6.7201 hPa,-48.594879,"Manteniendo constantes los demás predictores, ..."
5,M1_S1_S2,MaxTemp,-0.109945,0.011372,0.000000,-0.132234,-0.087656,0.895884,0.876136,0.916076,1 DE = 7.1103 °C,-10.411639,"Manteniendo constantes los demás predictores, ..."
6,M2_stepwise_AIC,const,-2.196415,0.025050,0.000000,-2.245512,-2.147319,0.111201,0.105873,0.116797,intercepto,-88.879893,Intercepto; no se interpreta de forma aislada.
7,M2_stepwise_AIC,Humidity3pm,1.317380,0.016708,0.000000,1.284633,1.350127,3.733628,3.613343,3.857916,1 DE = 20.6545 puntos porcentuales,273.362775,"Manteniendo constantes los demás predictores, ..."
8,M2_stepwise_AIC,WindGustSpeed,0.471965,0.010463,0.000000,0.451458,0.492471,1.603141,1.570600,1.636355,1 DE = 13.2275 km/h,60.314053,"Manteniendo constantes los demás predictores, ..."
9,M2_stepwise_AIC,Pressure3pm,-0.505311,0.011378,0.000000,-0.527612,-0.483010,0.603318,0.590012,0.616924,1 DE = 6.7201 hPa,-39.668201,"Manteniendo constantes los demás predictores, ..."


,modelo,logLik,AIC,BIC,n_parametros,convergio,iteraciones
2,M3_BIC_parsimonioso,"-37,037.513437","74,097.026874","74,201.617785",11,True,6
1,M2_stepwise_AIC,"-37,035.388320","74,094.776640","74,208.875815",12,True,6
0,M1_S1_S2,"-38,287.159418","76,586.318836","76,643.368424",6,True,6


## 5. Evaluación interna y selección del modelo

La comparación entre M1, M2 y M3 utiliza validación anidada estratificada. En cada pliegue externo se repiten, exclusivamente con el subentrenamiento, la selección de la estrategia de faltantes, la imputación, la codificación de `Season`, la estandarización, el stepwise AIC, la búsqueda BIC y el ajuste de los modelos. El pliegue externo se utiliza solo para evaluar el procedimiento completo.

La selección de variables, los coeficientes, AIC, BIC, odds ratios e intervalos se estiman con `statsmodels.Logit` sin ponderación. Una vez fijadas las variables, las métricas predictivas se calculan con `LogisticRegression`, comparando configuraciones ponderadas y no ponderadas.


In [18]:
def calculate_metrics(y_true, probability, threshold=0.50):
    y_true = np.asarray(y_true, dtype=int)
    probability = np.asarray(probability, dtype=float)
    prediction = (probability >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, prediction, labels=[0, 1]).ravel()
    return {
        'n': len(y_true), 'threshold': threshold,
        'TN': int(tn), 'FP': int(fp), 'FN': int(fn), 'TP': int(tp),
        'accuracy': accuracy_score(y_true, prediction),
        'balanced_accuracy': balanced_accuracy_score(y_true, prediction),
        'precision': precision_score(y_true, prediction, zero_division=0),
        'recall': recall_score(y_true, prediction, zero_division=0),
        'F1': f1_score(y_true, prediction, zero_division=0),
        'ROC_AUC': roc_auc_score(y_true, probability),
        'PR_AUC': average_precision_score(y_true, probability),
        'Brier': brier_score_loss(y_true, probability)
    }


def apply_simple_imputation(fit_data, transform_data, variables):
    fit_result = fit_data.copy()
    transform_result = transform_data.copy()
    for variable in variables:
        if variable == 'RainToday_bin':
            value = int(fit_result[variable].mode().iloc[0])
        else:
            value = fit_result[variable].median()
        fit_result[variable] = fit_result[variable].fillna(value)
        transform_result[variable] = transform_result[variable].fillna(value)
    return fit_result, transform_result


def build_fold_strategies(fit_raw, validation_raw, seed):
    complete_fit = fit_raw.dropna(subset=COMPARISON_ALL).copy()
    complete_validation = validation_raw.dropna(subset=COMPARISON_ALL).copy()
    simple_fit, simple_validation = apply_simple_imputation(
        fit_raw, validation_raw, COMPARISON_ALL
    )
    regression_result = chained_regression_imputation(
        fit_raw, validation_raw, IMPUTATION_SPECS,
        seed=seed, cycles=N_IMPUTATION_CYCLES,
        diagnostics=False
    )
    return {
        'casos_completos': (complete_fit, complete_validation),
        'imputacion_simple': (simple_fit, simple_validation),
        'imputacion_regresion': (
            regression_result['train'], regression_result['other']
        ),
    }


def safe_pairwise_correlation(data, first, second, context):
    valid = data[[first, second]].dropna()
    if len(valid) < 3:
        raise ValueError(
            f'Observaciones insuficientes para {first}–{second} en {context}: '
            f'n={len(valid)}.'
        )
    if valid[first].nunique(dropna=True) < 2 or valid[second].nunique(dropna=True) < 2:
        raise ValueError(
            f'Varianza nula para {first}–{second} en {context}.'
        )
    correlation = float(valid.corr().iloc[0, 1])
    if not np.isfinite(correlation):
        raise ValueError(
            f'Correlación no finita para {first}–{second} en {context}.'
        )
    return correlation


def select_strategy_with_fit_only(strategy_pairs, reference_fit):
    size_rows = []
    distribution_rows = []
    correlation_rows = []

    # Dentro de la validación anidada, toda referencia se calcula únicamente
    # con el subentrenamiento del pliegue. Esto evita fuga de información y
    # mantiene la selección de la estrategia completamente interna al fold.
    fold_reference_correlations = {
        (first, second): safe_pairwise_correlation(
            reference_fit,
            first,
            second,
            context='subentrenamiento del pliegue externo'
        )
        for first, second in correlation_pairs
    }

    for name, (fit_data, _) in strategy_pairs.items():
        if fit_data.empty:
            raise ValueError(
                f'La estrategia {name} produjo un subentrenamiento vacío.'
            )
        size_rows.append({
            'estrategia': name,
            'porcentaje_retenido': 100 * len(fit_data) / len(reference_fit)
        })

        for variable in COMPARISON_NUMERIC:
            reference = reference_fit[variable].dropna().to_numpy(dtype=float)
            values = fit_data[variable].dropna().to_numpy(dtype=float)
            if len(reference) < 3 or len(values) < 3:
                raise ValueError(
                    f'Datos insuficientes para comparar la distribución de '
                    f'{variable} con la estrategia {name}.'
                )
            iqr = np.subtract(*np.percentile(reference, [75, 25]))
            distribution_rows.append({
                'estrategia': name,
                'variable': variable,
                'distancia': (
                    wasserstein_distance(reference, values)
                    / (iqr if iqr > 0 else 1.0)
                )
            })

        for first, second in correlation_pairs:
            strategy_correlation = safe_pairwise_correlation(
                fit_data,
                first,
                second,
                context=f'estrategia {name} dentro del pliegue externo'
            )
            reference_correlation_fold = fold_reference_correlations[
                (first, second)
            ]
            correlation_rows.append({
                'estrategia': name,
                'variable_1': first,
                'variable_2': second,
                'r_referencia_subentrenamiento': reference_correlation_fold,
                'r_estrategia': strategy_correlation,
                'desviacion': abs(
                    strategy_correlation - reference_correlation_fold
                )
            })

    summary = (
        pd.DataFrame(size_rows)
        .merge(
            pd.DataFrame(distribution_rows)
            .groupby('estrategia')['distancia']
            .mean()
            .rename('distancia_distribucion_media'),
            on='estrategia'
        )
        .merge(
            pd.DataFrame(correlation_rows)
            .groupby('estrategia')['desviacion']
            .mean()
            .rename('desviacion_correlacion_media'),
            on='estrategia'
        )
    )
    summary['rango_retencion'] = (
        summary['porcentaje_retenido']
        .rank(method='min', ascending=False)
    )
    summary['rango_distribucion'] = (
        summary['distancia_distribucion_media']
        .rank(method='min', ascending=True)
    )
    summary['rango_correlacion'] = (
        summary['desviacion_correlacion_media']
        .rank(method='min', ascending=True)
    )
    principal_weights = WEIGHT_SCENARIOS['principal_40_30_30']
    summary['puntaje_decision'] = (
        principal_weights[0] * summary['rango_retencion']
        + principal_weights[1] * summary['rango_distribucion']
        + principal_weights[2] * summary['rango_correlacion']
    )
    selected = summary.sort_values(
        [
            'puntaje_decision',
            'desviacion_correlacion_media',
            'distancia_distribucion_media',
            'porcentaje_retenido'
        ],
        ascending=[True, True, True, False]
    ).iloc[0]['estrategia']
    return selected, summary

def standardize_fold(fit_data, validation_data):
    fit_data = add_engineered_features(fit_data)
    validation_data = add_engineered_features(validation_data)
    scaler_fold = StandardScaler().fit(fit_data[CONTINUOUS_FEATURES])
    X_fit = fit_data[CANDIDATE_FEATURES].copy().reset_index(drop=True)
    X_validation = validation_data[CANDIDATE_FEATURES].copy().reset_index(drop=True)
    X_fit[CONTINUOUS_FEATURES] = scaler_fold.transform(fit_data[CONTINUOUS_FEATURES])
    X_validation[CONTINUOUS_FEATURES] = scaler_fold.transform(validation_data[CONTINUOUS_FEATURES])
    y_fit = fit_data['RainTomorrow_bin'].astype(int).reset_index(drop=True)
    y_validation = validation_data['RainTomorrow_bin'].astype(int).reset_index(drop=True)
    return fit_data, validation_data, X_fit, X_validation, y_fit, y_validation, scaler_fold


def select_models_within_fold(X_fit, y_fit):
    cache = {}

    def local_design(features):
        if features:
            return sm.add_constant(
                X_fit[list(features)],
                has_constant='add'
            )
        return pd.DataFrame(
            {'const': np.ones(len(X_fit), dtype=float)},
            index=X_fit.index
        )

    def local_fit(features):
        key = tuple(features)
        if key not in cache:
            design = local_design(features)
            cache[key] = sm.Logit(
                y_fit,
                design
            ).fit(
                method='lbfgs',
                maxiter=200,
                disp=0
            )
            if not bool(
                cache[key].mle_retvals.get(
                    'converged',
                    False
                )
            ):
                raise RuntimeError(
                    'No convergió un modelo Logit dentro de la '
                    'validación anidada.'
                )
        return cache[key]

    def local_stepwise_aic(
        min_aic_improvement=1e-6
    ):
        selected_blocks = []
        current_result = local_fit([])
        current_aic = float(
            current_result.aic
        )

        while True:
            options = []

            for block in FEATURE_BLOCKS:
                if block in selected_blocks:
                    continue

                candidate_blocks = (
                    selected_blocks + [block]
                )
                candidate_result = local_fit(
                    flatten_blocks(
                        candidate_blocks
                    )
                )
                candidate_aic = float(
                    candidate_result.aic
                )
                options.append({
                    'block': block,
                    'aic': candidate_aic,
                    'improvement': (
                        current_aic
                        - candidate_aic
                    )
                })

            if not options:
                break

            best = max(
                options,
                key=lambda item: (
                    item['improvement']
                )
            )

            if (
                best['improvement']
                <= min_aic_improvement
            ):
                break

            selected_blocks.append(
                best['block']
            )
            current_aic = best['aic']

        return flatten_blocks(
            selected_blocks
        )

    def local_bic():
        best_bic = np.inf
        best_features = None
        block_names = list(FEATURE_BLOCKS)

        # La búsqueda conserva solo el mejor criterio; no almacena los 512
        # objetos de ajuste y, por tanto, controla el uso de memoria.
        for subset_size in range(len(block_names) + 1):
            for subset in combinations(block_names, subset_size):
                features = flatten_blocks(list(subset))
                design = local_design(features)
                result = sm.Logit(
                    y_fit,
                    design
                ).fit(
                    method='lbfgs',
                    maxiter=200,
                    disp=0
                )
                if not bool(
                    result.mle_retvals.get(
                        'converged',
                        False
                    )
                ):
                    raise RuntimeError(
                        'Un modelo Logit de la búsqueda BIC '
                        'anidada no convergió.'
                    )
                bic = (
                    -2 * result.llf
                    + np.log(result.nobs) * len(result.params)
                )
                if bic < best_bic:
                    best_bic = bic
                    best_features = features.copy()
                del result, design
        return best_features

    m2_features = local_stepwise_aic()
    m3_features = local_bic()
    if not m2_features or not m3_features:
        raise RuntimeError('Un procedimiento anidado no retuvo predictores.')
    return {
        'M1_S1_S2': M1_FEATURES,
        'M2_stepwise_AIC': m2_features,
        'M3_BIC_parsimonioso': m3_features
    }


def nested_selection_validation(raw_training, n_splits=N_NESTED_CV):
    folds = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    raw_training = raw_training.reset_index(drop=True)
    target = raw_training['RainTomorrow_bin'].astype(int).to_numpy()
    metric_rows = []
    oof_rows = []
    selection_rows = []
    strategy_rows = []

    for fold_number, (fit_index, validation_index) in enumerate(
        folds.split(raw_training, target), start=1
    ):
        print(
            f'Validación anidada: fold {fold_number}/{n_splits}'
        )
        fit_raw = raw_training.iloc[fit_index].copy()
        validation_raw = raw_training.iloc[validation_index].copy()

        strategy_pairs = build_fold_strategies(
            fit_raw, validation_raw, seed=SEED + 10_000 + fold_number
        )
        fold_strategy, fold_strategy_summary = select_strategy_with_fit_only(
            strategy_pairs, fit_raw
        )
        fold_strategy_summary = fold_strategy_summary.copy()
        fold_strategy_summary['fold'] = fold_number
        fold_strategy_summary['seleccionada'] = (
            fold_strategy_summary['estrategia'] == fold_strategy
        )
        strategy_rows.append(fold_strategy_summary)

        fit_prepared, validation_prepared = strategy_pairs[fold_strategy]
        (
            fit_prepared, validation_prepared,
            X_fit, X_validation, y_fit, y_validation, _
        ) = standardize_fold(fit_prepared, validation_prepared)

        fold_models = select_models_within_fold(X_fit, y_fit)

        for model_name, features in fold_models.items():
            model = LogisticRegression(
                penalty=None,
                solver='lbfgs',
                max_iter=2_000,
                class_weight='balanced'
            )
            with warnings.catch_warnings(record=True) as caught:
                warnings.simplefilter('always')
                model.fit(X_fit[features], y_fit)
            convergence_warnings = [
                item for item in caught
                if issubclass(item.category, ConvergenceWarning)
            ]
            if convergence_warnings:
                raise RuntimeError(
                    f'No convergió {model_name} en el fold {fold_number}: '
                    + ' | '.join(str(item.message) for item in convergence_warnings)
                )
            probability = model.predict_proba(X_validation[features])[:, 1]
            metrics = calculate_metrics(y_validation, probability, threshold=0.50)
            metric_rows.append({
                'fold': fold_number,
                'estrategia_faltantes': fold_strategy,
                'modelo': model_name,
                'estimador_predictivo': (
                    'LogisticRegression sin penalizacion; '
                    'class_weight=balanced'
                ),
                **metrics
            })
            selection_rows.append({
                'fold': fold_number,
                'modelo': model_name,
                'estrategia_faltantes': fold_strategy,
                'class_weight_predictivo': 'balanced',
                'n_variables': len(features),
                'variables': ', '.join(features),
                'n_advertencias': len(caught)
            })
            oof_rows.extend([
                {
                    'row_id': row_id,
                    'fold': fold_number,
                    'modelo': model_name,
                    'y_true': y_value,
                    'probabilidad': p_value
                }
                for row_id, y_value, p_value in zip(
                    validation_prepared['row_id'].to_numpy(),
                    y_validation.to_numpy(),
                    probability
                )
            ])

    return (
        pd.DataFrame(metric_rows),
        pd.DataFrame(oof_rows),
        pd.DataFrame(selection_rows),
        pd.concat(strategy_rows, ignore_index=True)
    )


def cross_validate_fixed_specification(
    raw_training, strategy, features, class_weight=None, n_splits=N_CV
):
    folds = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED + 900)
    raw_training = raw_training.reset_index(drop=True)
    target = raw_training['RainTomorrow_bin'].astype(int).to_numpy()
    rows = []
    oof_rows = []

    quadratic_bases = sorted({
        feature[:-3] for feature in features if feature.endswith('_sq')
    })

    for fold_number, (fit_index, validation_index) in enumerate(
        folds.split(raw_training, target), start=1
    ):
        fit_raw = raw_training.iloc[fit_index].copy()
        validation_raw = raw_training.iloc[validation_index].copy()

        if strategy == 'casos_completos':
            fit_prepared = fit_raw.dropna(subset=COMPARISON_ALL).copy()
            validation_prepared = validation_raw.dropna(subset=COMPARISON_ALL).copy()
        elif strategy == 'imputacion_simple':
            fit_prepared, validation_prepared = apply_simple_imputation(
                fit_raw, validation_raw, COMPARISON_ALL
            )
        elif strategy == 'imputacion_regresion':
            result = chained_regression_imputation(
                fit_raw, validation_raw, IMPUTATION_SPECS,
                seed=SEED + 20_000 + fold_number,
                cycles=N_IMPUTATION_CYCLES,
                diagnostics=False
            )
            fit_prepared, validation_prepared = result['train'], result['other']
        else:
            raise ValueError(f'Estrategia desconocida: {strategy}')

        (
            fit_prepared, validation_prepared,
            X_fit, X_validation, y_fit, y_validation, _
        ) = standardize_fold(fit_prepared, validation_prepared)

        for variable in quadratic_bases:
            X_fit[f'{variable}_sq'] = X_fit[variable] ** 2
            X_validation[f'{variable}_sq'] = X_validation[variable] ** 2

        model = LogisticRegression(
            penalty=None,
            solver='lbfgs',
            max_iter=2_000,
            class_weight=class_weight
        )
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            model.fit(X_fit[features], y_fit)
        convergence_warnings = [
            item for item in caught
            if issubclass(item.category, ConvergenceWarning)
        ]
        if convergence_warnings:
            raise RuntimeError(
                f'El modelo final no convergió en el fold {fold_number}: '
                + ' | '.join(str(item.message) for item in convergence_warnings)
            )
        probability = model.predict_proba(X_validation[features])[:, 1]
        rows.append({
            'fold': fold_number,
            'estimador_predictivo': (
                'LogisticRegression sin penalizacion; '
                f'class_weight={class_weight if class_weight is not None else "None"}'
            ),
            **calculate_metrics(y_validation, probability, threshold=0.50)
        })
        oof_rows.extend([
            {
                'row_id': row_id,
                'fold': fold_number,
                'y_true': y_value,
                'probabilidad': p_value
            }
            for row_id, y_value, p_value in zip(
                validation_prepared['row_id'].to_numpy(),
                y_validation.to_numpy(),
                probability
            )
        ])

    return pd.DataFrame(rows), pd.DataFrame(oof_rows)


nested_cv_metrics, nested_oof_predictions, nested_selections, nested_strategy_choices = (
    nested_selection_validation(train_raw, n_splits=N_NESTED_CV)
)
cv_summary = nested_cv_metrics.groupby('modelo').agg(
    ROC_AUC_media=('ROC_AUC', 'mean'),
    ROC_AUC_sd=('ROC_AUC', 'std'),
    PR_AUC_media=('PR_AUC', 'mean'),
    PR_AUC_sd=('PR_AUC', 'std'),
    F1_media=('F1', 'mean'),
    F1_sd=('F1', 'std'),
    recall_medio=('recall', 'mean'),
    recall_sd=('recall', 'std'),
    balanced_accuracy_media=('balanced_accuracy', 'mean')
).reset_index()

nested_cv_metrics.to_csv(TABLES / '19a_metricas_validacion_anidada_por_fold.csv', index=False)
cv_summary.to_csv(TABLES / '19b_resumen_validacion_anidada.csv', index=False)
nested_oof_predictions.to_csv(TABLES / '19c_probabilidades_out_of_fold_anidadas.csv', index=False)
nested_selections.to_csv(TABLES / '19d_variables_seleccionadas_en_cada_fold.csv', index=False)
nested_strategy_choices.to_csv(TABLES / '19e_imputacion_seleccionada_en_cada_fold.csv', index=False)

nested_selection_stability = (
    nested_selections
    .groupby(['modelo', 'variables'], as_index=False)
    .agg(
        n_folds=('fold', 'nunique'),
        n_variables=('n_variables', 'first'),
        advertencias=('n_advertencias', 'sum')
    )
    .sort_values(['modelo', 'n_folds'], ascending=[True, False])
)

display(cv_summary)
display(nested_selection_stability)
display(
    nested_strategy_choices.loc[
        nested_strategy_choices['seleccionada'],
        ['fold', 'estrategia', 'porcentaje_retenido']
    ]
)


Validación anidada: fold 1/5
Validación anidada: fold 2/5
Validación anidada: fold 3/5
Validación anidada: fold 4/5
Validación anidada: fold 5/5


,modelo,ROC_AUC_media,ROC_AUC_sd,PR_AUC_media,PR_AUC_sd,F1_media,F1_sd,recall_medio,recall_sd,balanced_accuracy_media
0,M1_S1_S2,0.838517,0.001420,0.652237,0.005518,0.589448,0.001636,0.745048,0.001126,0.759407
1,M2_stepwise_AIC,0.851657,0.000841,0.670964,0.004935,0.603117,0.001900,0.752398,0.004205,0.768903
2,M3_BIC_parsimonioso,0.851624,0.000857,0.671120,0.004877,0.602342,0.001649,0.753249,0.003804,0.768577


,modelo,variables,n_folds,n_variables,advertencias
0,M1_S1_S2,"Humidity3pm, RainToday_bin, Rainfall_log1p, Pr...",5,5,0
3,M2_stepwise_AIC,"Humidity3pm, WindGustSpeed, Pressure3pm, Rainf...",3,11,0
1,M2_stepwise_AIC,"Humidity3pm, Pressure3pm, WindGustSpeed, Rainf...",1,11,0
2,M2_stepwise_AIC,"Humidity3pm, WindGustSpeed, Pressure3pm, Rainf...",1,10,0
4,M3_BIC_parsimonioso,"Humidity3pm, RainToday_bin, Rainfall_log1p, Pr...",5,10,0


,fold,estrategia,porcentaje_retenido
2,1,imputacion_regresion,100.000000
5,2,imputacion_regresion,100.000000
8,3,imputacion_regresion,100.000000
11,4,imputacion_regresion,100.000000
14,5,imputacion_regresion,100.000000


In [19]:
predictive_models_balanced = {}
training_rows = []
training_confusion_rows = []

for model_name, features in MODEL_DEFINITIONS.items():
    model = LogisticRegression(
        penalty=None,
        solver='lbfgs',
        max_iter=2_000,
        class_weight='balanced'
    )
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        model.fit(X_train[features], y_train)

    convergence_warnings = [
        item for item in caught
        if issubclass(item.category, ConvergenceWarning)
    ]
    if convergence_warnings:
        raise RuntimeError(
            f'El modelo predictivo balanceado {model_name} no convergió: '
            + ' | '.join(
                str(item.message)
                for item in convergence_warnings
            )
        )

    predictive_models_balanced[model_name] = model
    train_probability = model.predict_proba(
        X_train[features]
    )[:, 1]
    metrics = calculate_metrics(
        y_train,
        train_probability,
        threshold=0.50
    )
    training_rows.append({
        'modelo': model_name,
        'conjunto': 'train',
        'estimador_predictivo': (
            'LogisticRegression sin penalizacion; '
            'class_weight=balanced'
        ),
        **metrics
    })
    training_confusion_rows.append({
        'modelo': model_name,
        'conjunto': 'train',
        'TN': metrics['TN'],
        'FP': metrics['FP'],
        'FN': metrics['FN'],
        'TP': metrics['TP']
    })

training_performance = pd.DataFrame(training_rows)
training_confusion = pd.DataFrame(training_confusion_rows)
training_performance.to_csv(
    TABLES / '20a_metricas_train_tres_modelos.csv',
    index=False
)
training_confusion.to_csv(
    TABLES / '21a_matrices_confusion_train_tres_modelos.csv',
    index=False
)
display(Markdown(
    '**Reserva del test:** los modelos predictivos fueron ajustados con '
    '`class_weight="balanced"` y sus métricas de entrenamiento fueron '
    'exportadas. El conjunto de prueba permanece cerrado hasta fijar la '
    'especificación y el umbral del modelo final.'
))


**Reserva del test:** los modelos predictivos fueron ajustados con `class_weight="balanced"` y sus métricas de entrenamiento fueron exportadas. El conjunto de prueba permanece cerrado hasta fijar la especificación y el umbral del modelo final.

In [20]:
# Regla predefinida:
# 1) retener procedimientos a menos de 0,005 del mejor ROC-AUC anidado;
# 2) entre ellos, retener los que están a menos de 0,005 del mejor PR-AUC;
# 3) escoger la especificación de entrenamiento completo con menor BIC.
selection_table = cv_summary.merge(
    information_criteria[['modelo', 'BIC']], on='modelo'
)
best_roc = selection_table['ROC_AUC_media'].max()
eligible_models = selection_table.loc[
    selection_table['ROC_AUC_media'] >= best_roc - 0.005
].copy()
best_pr = eligible_models['PR_AUC_media'].max()
eligible_models = eligible_models.loc[
    eligible_models['PR_AUC_media'] >= best_pr - 0.005
].copy()

BEST_MODEL_NAME = eligible_models.sort_values(
    ['BIC', 'F1_media'], ascending=[True, False]
).iloc[0]['modelo']
BEST_MAIN_FEATURES = MODEL_DEFINITIONS[BEST_MODEL_NAME]
selection_table['modelo_seleccionado'] = (
    selection_table['modelo'] == BEST_MODEL_NAME
)
selection_table.to_csv(TABLES / '22_seleccion_modelo_final_sin_test.csv', index=False)
display(selection_table)
display(Markdown(
    f'**Modelo seleccionado sin utilizar test:** `{BEST_MODEL_NAME}`.'
))


,modelo,ROC_AUC_media,ROC_AUC_sd,PR_AUC_media,PR_AUC_sd,F1_media,F1_sd,recall_medio,recall_sd,balanced_accuracy_media,BIC,modelo_seleccionado
0,M1_S1_S2,0.838517,0.001420,0.652237,0.005518,0.589448,0.001636,0.745048,0.001126,0.759407,"76,643.368424",False
1,M2_stepwise_AIC,0.851657,0.000841,0.670964,0.004935,0.603117,0.001900,0.752398,0.004205,0.768903,"74,208.875815",False
2,M3_BIC_parsimonioso,0.851624,0.000857,0.671120,0.004877,0.602342,0.001649,0.753249,0.003804,0.768577,"74,201.617785",True


**Modelo seleccionado sin utilizar test:** `M3_BIC_parsimonioso`.

## 6. Diagnóstico y refinamiento del modelo seleccionado

Se evalúan multicolinealidad, linealidad en el logit, residuos e influencia. Box–Tidwell se corrige mediante Holm. Cuando una variable presenta evidencia de no linealidad se evalúa un término cuadrático, que se conserva solo si mejora el BIC en al menos dos unidades y la prueba de razón de verosimilitud es significativa.


In [21]:
base_result = fitted_models[BEST_MODEL_NAME]
base_features = BEST_MAIN_FEATURES.copy()
continuous_base = [
    feature for feature in base_features
    if feature not in BINARY_FEATURES
]

box_tidwell_rows = []
if continuous_base:
    shifted = selected_train[continuous_base].copy()
    bt_terms = []
    shift_values = {}

    for variable in continuous_base:
        minimum = shifted[variable].min()
        shift = 1 - minimum if minimum <= 0 else 0
        shift_values[variable] = shift
        positive = shifted[variable] + shift
        term_name = f'BT_{variable}'
        shifted[term_name] = positive * np.log(positive)
        bt_terms.append(term_name)

    bt_design = pd.concat([
        X_train[base_features].reset_index(drop=True),
        shifted[bt_terms].reset_index(drop=True)
    ], axis=1)
    bt_result = sm.GLM(
        y_train,
        sm.add_constant(bt_design, has_constant='add'),
        family=sm.families.Binomial()
    ).fit(maxiter=200, disp=0)

    raw_p = [bt_result.pvalues[term] for term in bt_terms]
    adjusted_p = multipletests(raw_p, method='holm')[1]

    for variable, term, p_raw, p_holm in zip(
        continuous_base, bt_terms, raw_p, adjusted_p
    ):
        box_tidwell_rows.append({
            'variable': variable,
            'termino_Box_Tidwell': term,
            'desplazamiento_aplicado': shift_values[variable],
            'p_raw': p_raw,
            'p_Holm': p_holm,
            'evidencia_no_linealidad': p_holm < ALPHA
        })

box_tidwell = pd.DataFrame(box_tidwell_rows)
box_tidwell.to_csv(
    TABLES / '25_linealidad_logit_Box_Tidwell.csv',
    index=False
)
display(box_tidwell)

if not box_tidwell.empty:
    fig, ax = plt.subplots(figsize=(8, 5))
    plot_table = box_tidwell.sort_values('p_Holm')
    ax.bar(
        plot_table['variable'],
        -np.log10(plot_table['p_Holm'].clip(lower=1e-300))
    )
    ax.axhline(-np.log10(ALPHA), linestyle='--', linewidth=1)
    ax.set_ylabel('-log10(p ajustado por Holm)')
    ax.set_title('Diagnóstico de linealidad en el logit')
    ax.tick_params(axis='x', rotation=45)
    fig.tight_layout()
    fig.savefig(
        FIGURES / 'fig_07_linealidad_logit_Box_Tidwell.png',
        dpi=170
    )
    plt.close(fig)

detected_nonlinear_variables = (
    box_tidwell.loc[
        box_tidwell['evidencia_no_linealidad'], 'variable'
    ].tolist()
    if not box_tidwell.empty
    else []
)

X_train_refined = X_train.copy()
current_features = base_features.copy()
current_result = base_result
current_bic = (
    -2 * current_result.llf
    + np.log(current_result.nobs) * len(current_result.params)
)
refinement_rows = []
retained_nonlinear_variables = []

# Se respeta jerarquía: un término cuadrático solo se evalúa junto a su término lineal.
# Cada candidato se contrasta contra la especificación aceptada en el paso anterior.
for variable in detected_nonlinear_variables:
    term = f'{variable}_sq'
    X_train_refined[term] = X_train_refined[variable] ** 2
    candidate_features = current_features + [term]
    candidate_result = sm.GLM(
        y_train,
        sm.add_constant(
            X_train_refined[candidate_features],
            has_constant='add'
        ),
        family=sm.families.Binomial()
    ).fit(maxiter=200, disp=0)

    lr_statistic = 2 * (
        candidate_result.llf - current_result.llf
    )
    lr_p = chi2.sf(max(lr_statistic, 0), 1)
    candidate_bic = (
        -2 * candidate_result.llf
        + np.log(candidate_result.nobs)
        * len(candidate_result.params)
    )
    retained = bool(
        lr_p < ALPHA
        and candidate_bic <= current_bic - 2
    )

    refinement_rows.append({
        'modelo_base_inicial': BEST_MODEL_NAME,
        'variable_evaluada': variable,
        'termino_cuadratico': term,
        'estadistico_LR': lr_statistic,
        'p_LR': lr_p,
        'BIC_antes': current_bic,
        'BIC_candidato': candidate_bic,
        'mejora_BIC': current_bic - candidate_bic,
        'termino_retenido': retained
    })

    if retained:
        current_features = candidate_features
        current_result = candidate_result
        current_bic = candidate_bic
        retained_nonlinear_variables.append(variable)

nonlinear_variables = retained_nonlinear_variables
quadratic_terms = [
    f'{variable}_sq'
    for variable in retained_nonlinear_variables
]
KEEP_REFINEMENT = bool(quadratic_terms)
FINAL_FEATURES = current_features
FINAL_RESULT = current_result

if refinement_rows:
    refinement_table = pd.DataFrame(refinement_rows)
else:
    refinement_table = pd.DataFrame([{
        'modelo_base_inicial': BEST_MODEL_NAME,
        'variable_evaluada': 'ninguna',
        'termino_cuadratico': 'ninguno',
        'estadistico_LR': np.nan,
        'p_LR': np.nan,
        'BIC_antes': current_bic,
        'BIC_candidato': current_bic,
        'mejora_BIC': 0.0,
        'termino_retenido': False
    }])

refinement_table['variables_finales'] = ', '.join(FINAL_FEATURES)
refinement_table.to_csv(
    TABLES / '26_refinamiento_linealidad_logit.csv',
    index=False
)
display(refinement_table)
display(Markdown(
    f'**No linealidad detectada:** `{detected_nonlinear_variables}`.  '
    f'**Términos cuadráticos retenidos:** `{quadratic_terms}`.'
))


,variable,termino_Box_Tidwell,desplazamiento_aplicado,p_raw,p_Holm,evidencia_no_linealidad
0,Humidity3pm,BT_Humidity3pm,1.000000,0.000000,0.000000,True
1,Rainfall_log1p,BT_Rainfall_log1p,1.000000,0.000000,0.000000,True
2,Pressure3pm,BT_Pressure3pm,0.000000,0.003923,0.007846,True
3,MaxTemp,BT_MaxTemp,5.800000,0.000000,0.000000,True
4,WindGustSpeed,BT_WindGustSpeed,0.000000,0.000018,0.000054,True
5,Humidity9am,BT_Humidity9am,1.000000,0.171964,0.171964,False


,modelo_base_inicial,variable_evaluada,termino_cuadratico,estadistico_LR,p_LR,BIC_antes,BIC_candidato,mejora_BIC,termino_retenido,variables_finales
0,M3_BIC_parsimonioso,Humidity3pm,Humidity3pm_sq,367.857853,0.000000,"74,201.617785","73,845.268196",356.349589,True,"Humidity3pm, RainToday_bin, Rainfall_log1p, Pr..."
1,M3_BIC_parsimonioso,Rainfall_log1p,Rainfall_log1p_sq,59.480506,0.000000,"73,845.268196","73,797.295955",47.972241,True,"Humidity3pm, RainToday_bin, Rainfall_log1p, Pr..."
2,M3_BIC_parsimonioso,Pressure3pm,Pressure3pm_sq,3.527828,0.060347,"73,797.295955","73,805.276391",-7.980436,False,"Humidity3pm, RainToday_bin, Rainfall_log1p, Pr..."
3,M3_BIC_parsimonioso,MaxTemp,MaxTemp_sq,125.139006,0.000000,"73,797.295955","73,683.665214",113.630741,True,"Humidity3pm, RainToday_bin, Rainfall_log1p, Pr..."
4,M3_BIC_parsimonioso,WindGustSpeed,WindGustSpeed_sq,25.505845,0.000000,"73,683.665214","73,669.667633",13.997581,True,"Humidity3pm, RainToday_bin, Rainfall_log1p, Pr..."


**No linealidad detectada:** `['Humidity3pm', 'Rainfall_log1p', 'Pressure3pm', 'MaxTemp', 'WindGustSpeed']`.  **Términos cuadráticos retenidos:** `['Humidity3pm_sq', 'Rainfall_log1p_sq', 'MaxTemp_sq', 'WindGustSpeed_sq']`.

### 6.1 Resolución de multicolinealidad y especificación definitiva

El VIF se utiliza como diagnóstico. Si la especificación refinada presenta VIF superior a 10, se comparan variantes jerárquicamente válidas usando entrenamiento. La especificación definitiva se elige entre los modelos con VIF máximo no superior a 10 y, dentro de ese conjunto, por el menor BIC.


In [22]:
def vif_summary_for_features(X_data, features):
    design = sm.add_constant(X_data[features], has_constant='add')
    rows = []
    matrix = design.to_numpy(dtype=float)
    for index, variable in enumerate(design.columns):
        if variable == 'const':
            continue
        rows.append({
            'variable': variable,
            'VIF': variance_inflation_factor(matrix, index)
        })
    table = pd.DataFrame(rows).sort_values('VIF', ascending=False)
    return table, float(table['VIF'].max())


candidate_specs = {
    'refinamiento_completo': FINAL_FEATURES.copy()
}
if 'RainToday_bin' in FINAL_FEATURES:
    candidate_specs['sin_RainToday_bin'] = [
        feature for feature in FINAL_FEATURES
        if feature != 'RainToday_bin'
    ]
if 'Rainfall_log1p_sq' in FINAL_FEATURES:
    candidate_specs['sin_Rainfall_cuadratico'] = [
        feature for feature in FINAL_FEATURES
        if feature != 'Rainfall_log1p_sq'
    ]
if (
    'RainToday_bin' in FINAL_FEATURES
    and 'Rainfall_log1p_sq' in FINAL_FEATURES
):
    candidate_specs['sin_RainToday_y_sin_Rainfall_cuadratico'] = [
        feature for feature in FINAL_FEATURES
        if feature not in {'RainToday_bin', 'Rainfall_log1p_sq'}
    ]

multicollinearity_rows = []
multicollinearity_models = {}
multicollinearity_vif_tables = {}
for candidate_name, candidate_features in candidate_specs.items():
    candidate_result = sm.GLM(
        y_train,
        sm.add_constant(
            X_train_refined[candidate_features],
            has_constant='add'
        ),
        family=sm.families.Binomial()
    ).fit(maxiter=200, disp=0)
    vif_candidate, max_vif_candidate = vif_summary_for_features(
        X_train_refined,
        candidate_features
    )
    candidate_bic = (
        -2 * candidate_result.llf
        + np.log(candidate_result.nobs) * len(candidate_result.params)
    )
    multicollinearity_rows.append({
        'especificacion': candidate_name,
        'variables': ', '.join(candidate_features),
        'n_predictores': len(candidate_features),
        'AIC': candidate_result.aic,
        'BIC': candidate_bic,
        'VIF_max': max_vif_candidate,
        'cumple_VIF_max_10': max_vif_candidate <= 10,
        'convergio': bool(candidate_result.converged)
    })
    multicollinearity_models[candidate_name] = candidate_result
    multicollinearity_vif_tables[candidate_name] = vif_candidate

multicollinearity_comparison = (
    pd.DataFrame(multicollinearity_rows)
    .sort_values(['cumple_VIF_max_10', 'BIC'], ascending=[False, True])
    .reset_index(drop=True)
)
if not multicollinearity_comparison['convergio'].all():
    raise RuntimeError(
        'Al menos una especificación de sensibilidad por multicolinealidad no convergió.'
    )

admissible = multicollinearity_comparison.loc[
    multicollinearity_comparison['cumple_VIF_max_10']
].copy()
if admissible.empty:
    raise RuntimeError(
        'Ninguna especificación jerárquicamente válida alcanzó VIF máximo <= 10. '
        'Revise la familia Rainfall/RainToday antes de continuar.'
    )

selected_multicollinearity_row = admissible.sort_values('BIC').iloc[0]
MULTICOLLINEARITY_SPECIFICATION = selected_multicollinearity_row['especificacion']
FINAL_FEATURES = candidate_specs[MULTICOLLINEARITY_SPECIFICATION]
FINAL_RESULT = multicollinearity_models[MULTICOLLINEARITY_SPECIFICATION]
FINAL_MAX_VIF = float(selected_multicollinearity_row['VIF_max'])

multicollinearity_comparison['seleccionada'] = (
    multicollinearity_comparison['especificacion']
    == MULTICOLLINEARITY_SPECIFICATION
)
multicollinearity_comparison.to_csv(
    TABLES / '27b_resolucion_multicolinealidad_modelo_final.csv',
    index=False
)
multicollinearity_vif_tables[MULTICOLLINEARITY_SPECIFICATION].to_csv(
    TABLES / '27c_VIF_especificacion_definitiva.csv',
    index=False
)
multicollinearity_decision = pd.DataFrame([{
    'especificacion_seleccionada': MULTICOLLINEARITY_SPECIFICATION,
    'variables_finales': ', '.join(FINAL_FEATURES),
    'BIC': float(selected_multicollinearity_row['BIC']),
    'VIF_max': FINAL_MAX_VIF,
    'criterio': (
        'Menor BIC entre especificaciones jerárquicamente válidas con VIF máximo <= 10; '
        'decisión tomada solo con entrenamiento.'
    )
}])
multicollinearity_decision.to_csv(
    TABLES / '27d_decision_especificacion_definitiva.csv',
    index=False
)
display(multicollinearity_comparison)
display(multicollinearity_decision)
display(Markdown(
    f'**Especificación definitiva:** `{MULTICOLLINEARITY_SPECIFICATION}`.  '
    f'**VIF máximo:** `{FINAL_MAX_VIF:.3f}`.'
))


,especificacion,variables,n_predictores,AIC,BIC,VIF_max,cumple_VIF_max_10,convergio,seleccionada
0,sin_RainToday_bin,"Humidity3pm, Rainfall_log1p, Pressure3pm, MaxT...",13,"73,526.071735","73,659.187439",4.789744,True,True,True
1,sin_Rainfall_cuadratico,"Humidity3pm, RainToday_bin, Rainfall_log1p, Pr...",13,"73,588.558948","73,721.674653",4.502781,True,True,False
2,sin_RainToday_y_sin_Rainfall_cuadratico,"Humidity3pm, Rainfall_log1p, Pressure3pm, MaxT...",12,"73,632.415893","73,756.023333",2.457435,True,True,False
3,refinamiento_completo,"Humidity3pm, RainToday_bin, Rainfall_log1p, Pr...",14,"73,527.043664","73,669.667633",23.569339,False,True,False


,especificacion_seleccionada,variables_finales,BIC,VIF_max,criterio
0,sin_RainToday_bin,"Humidity3pm, Rainfall_log1p, Pressure3pm, MaxT...","73,659.187439",4.789744,Menor BIC entre especificaciones jerárquicamen...


**Especificación definitiva:** `sin_RainToday_bin`.  **VIF máximo:** `4.790`.

In [23]:
# Comparación simétrica de configuraciones predictivas mediante OOF.
# La imputación, el escalamiento y el ajuste se recalculan dentro de cada pliegue.

final_cv_unweighted, final_oof_unweighted = cross_validate_fixed_specification(
    train_raw,
    SELECTED_IMPUTATION,
    FINAL_FEATURES,
    class_weight=None,
    n_splits=N_CV
)
final_cv_balanced, final_oof_balanced = cross_validate_fixed_specification(
    train_raw,
    SELECTED_IMPUTATION,
    FINAL_FEATURES,
    class_weight='balanced',
    n_splits=N_CV
)

final_cv_unweighted['configuracion'] = 'sin_ponderacion'
final_cv_balanced['configuracion'] = 'balanceada'
final_cv_metrics = pd.concat(
    [final_cv_unweighted, final_cv_balanced],
    ignore_index=True
)
final_cv_metrics.to_csv(
    TABLES / '26b_validacion_interna_modelo_final.csv',
    index=False
)

final_oof_unweighted = final_oof_unweighted.rename(
    columns={'probabilidad': 'probabilidad_sin_ponderacion'}
)
final_oof_balanced = final_oof_balanced.rename(
    columns={'probabilidad': 'probabilidad_balanceada'}
)
final_oof_predictions = final_oof_unweighted.merge(
    final_oof_balanced[
        ['row_id', 'fold', 'y_true', 'probabilidad_balanceada']
    ],
    on=['row_id', 'fold', 'y_true'],
    how='inner',
    validate='one_to_one'
)
final_oof_predictions.to_csv(
    TABLES / '26c_probabilidades_OOF_modelo_final.csv',
    index=False
)


def threshold_grid(y_true, probability, configuration):
    rows = []
    for threshold in np.linspace(0.05, 0.95, 181):
        rows.append({
            'configuracion': configuration,
            **calculate_metrics(y_true, probability, threshold=threshold)
        })
    return pd.DataFrame(rows)


threshold_unweighted = threshold_grid(
    final_oof_predictions['y_true'],
    final_oof_predictions['probabilidad_sin_ponderacion'],
    'sin_ponderacion'
)
threshold_balanced = threshold_grid(
    final_oof_predictions['y_true'],
    final_oof_predictions['probabilidad_balanceada'],
    'balanceada'
)
threshold_table = pd.concat(
    [threshold_unweighted, threshold_balanced],
    ignore_index=True
)

best_threshold_rows = []
for configuration, group in threshold_table.groupby('configuracion'):
    selected_row = group.sort_values(
        ['F1', 'balanced_accuracy', 'recall', 'precision'],
        ascending=False
    ).iloc[0].copy()
    selected_row['tipo_umbral'] = 'OOF_optimizado'
    best_threshold_rows.append(selected_row)

best_thresholds = pd.DataFrame(best_threshold_rows)
best_threshold_lookup = best_thresholds.set_index('configuracion')['threshold'].to_dict()

configuration_rows = []
probability_lookup = {
    'sin_ponderacion': final_oof_predictions['probabilidad_sin_ponderacion'],
    'balanceada': final_oof_predictions['probabilidad_balanceada']
}
for configuration, probability in probability_lookup.items():
    configuration_rows.append({
        'configuracion': configuration,
        'tipo_umbral': 'fijo_0_50',
        **calculate_metrics(
            final_oof_predictions['y_true'],
            probability,
            threshold=0.50
        )
    })
    configuration_rows.append({
        'configuracion': configuration,
        'tipo_umbral': 'OOF_optimizado',
        **calculate_metrics(
            final_oof_predictions['y_true'],
            probability,
            threshold=best_threshold_lookup[configuration]
        )
    })

predictive_configuration_oof = pd.DataFrame(configuration_rows)
F1_TOLERANCE = 0.005
max_f1 = predictive_configuration_oof['F1'].max()
eligible = predictive_configuration_oof.loc[
    predictive_configuration_oof['F1'] >= max_f1 - F1_TOLERANCE
].copy()
eligible['preferencia_no_ponderada'] = (
    eligible['configuracion'].eq('sin_ponderacion').astype(int)
)
selected_predictive_row = eligible.sort_values(
    [
        'balanced_accuracy',
        'recall',
        'Brier',
        'preferencia_no_ponderada'
    ],
    ascending=[False, False, True, False]
).iloc[0]

SELECTED_PREDICTIVE_CONFIGURATION = selected_predictive_row['configuracion']
SELECTED_CLASS_WEIGHT = (
    None
    if SELECTED_PREDICTIVE_CONFIGURATION == 'sin_ponderacion'
    else 'balanced'
)
BEST_THRESHOLD = float(selected_predictive_row['threshold'])

predictive_configuration_oof['seleccionada'] = (
    predictive_configuration_oof['configuracion'].eq(
        SELECTED_PREDICTIVE_CONFIGURATION
    )
    & np.isclose(
        predictive_configuration_oof['threshold'],
        BEST_THRESHOLD
    )
)
predictive_configuration_oof['regla_seleccion'] = (
    'Conservar configuraciones dentro de 0,005 del F1 OOF máximo; '
    'ordenar por balanced accuracy, recall, Brier y preferir no ponderada '
    'ante equivalencia.'
)
predictive_configuration_oof.to_csv(
    TABLES / '23b_seleccion_configuracion_predictiva_OOF.csv',
    index=False
)

threshold_table['seleccionado'] = (
    threshold_table['configuracion'].eq(
        SELECTED_PREDICTIVE_CONFIGURATION
    )
    & np.isclose(threshold_table['threshold'], BEST_THRESHOLD)
)
threshold_table.to_csv(
    TABLES / '23_sensibilidad_umbral_out_of_fold.csv',
    index=False
)


def calibration_by_quantile(y_true, probability, configuration, n_bins=10):
    frame = pd.DataFrame({
        'y_true': np.asarray(y_true, dtype=int),
        'probabilidad': np.asarray(probability, dtype=float)
    })
    frame['decil'] = pd.qcut(
        frame['probabilidad'],
        q=n_bins,
        labels=False,
        duplicates='drop'
    )
    return (
        frame.groupby('decil', observed=True)
        .agg(
            n=('y_true', 'size'),
            probabilidad_media=('probabilidad', 'mean'),
            proporcion_observada=('y_true', 'mean')
        )
        .reset_index()
        .assign(configuracion=configuration)
    )


calibration_table = pd.concat([
    calibration_by_quantile(
        final_oof_predictions['y_true'],
        final_oof_predictions['probabilidad_sin_ponderacion'],
        'sin_ponderacion'
    ),
    calibration_by_quantile(
        final_oof_predictions['y_true'],
        final_oof_predictions['probabilidad_balanceada'],
        'balanceada'
    )
], ignore_index=True)
calibration_table.to_csv(
    TABLES / '26d_calibracion_configuraciones_predictivas.csv',
    index=False
)

calibration_summary = pd.DataFrame([
    {
        'configuracion': 'sin_ponderacion',
        'Brier_OOF': brier_score_loss(
            final_oof_predictions['y_true'],
            final_oof_predictions['probabilidad_sin_ponderacion']
        )
    },
    {
        'configuracion': 'balanceada',
        'Brier_OOF': brier_score_loss(
            final_oof_predictions['y_true'],
            final_oof_predictions['probabilidad_balanceada']
        )
    }
])
calibration_summary['mejor_Brier_OOF'] = (
    calibration_summary['Brier_OOF']
    == calibration_summary['Brier_OOF'].min()
)
calibration_summary.to_csv(
    TABLES / '26f_resumen_calibracion_predictiva.csv',
    index=False
)

fig, ax = plt.subplots(figsize=(7, 6))
for configuration, group in calibration_table.groupby('configuracion'):
    ax.plot(
        group['probabilidad_media'],
        group['proporcion_observada'],
        marker='o',
        label=configuration
    )
ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1, label='calibración ideal')
ax.set_xlabel('Probabilidad media predicha')
ax.set_ylabel('Proporción observada')
ax.set_title('Calibración OOF: configuraciones predictivas')
ax.legend()
fig.tight_layout()
fig.savefig(
    FIGURES / 'fig_09b_calibracion_balanceado_vs_no_balanceado.png',
    dpi=170
)
plt.close(fig)

baseline = pd.DataFrame([{
    'prevalencia_train': y_train.mean(),
    'accuracy_clasificador_mayoritario': max(
        y_train.mean(),
        1 - y_train.mean()
    ),
    'configuracion_predictiva_seleccionada': SELECTED_PREDICTIVE_CONFIGURATION,
    'class_weight_seleccionado': (
        'None' if SELECTED_CLASS_WEIGHT is None else SELECTED_CLASS_WEIGHT
    ),
    'umbral_operativo_out_of_fold': BEST_THRESHOLD,
    'criterio_configuracion': (
        'F1 OOF con tolerancia 0,005; desempate por balanced accuracy, '
        'recall, Brier y preferencia por no ponderación.'
    )
}])
baseline.to_csv(
    TABLES / '24_desbalance_y_umbral_operativo.csv',
    index=False
)

display(final_cv_metrics)
display(predictive_configuration_oof)
display(calibration_table)
display(calibration_summary)
display(baseline)

fig, ax = plt.subplots(figsize=(8, 5))
for configuration, group in threshold_table.groupby('configuracion'):
    ax.plot(
        group['threshold'],
        group['F1'],
        label=f'F1 {configuration}'
    )
ax.axvline(
    BEST_THRESHOLD,
    linestyle='--',
    linewidth=1,
    label=f'umbral seleccionado={BEST_THRESHOLD:.3f}'
)
ax.set_xlabel('Umbral de clasificación')
ax.set_ylabel('F1 out-of-fold')
ax.set_title('Comparación simétrica de umbrales OOF')
ax.legend()
fig.tight_layout()
fig.savefig(
    FIGURES / 'fig_09_sensibilidad_umbral_OOF.png',
    dpi=170
)
plt.close(fig)


,fold,estimador_predictivo,n,threshold,TN,FP,FN,TP,accuracy,balanced_accuracy,precision,recall,F1,ROC_AUC,PR_AUC,Brier,configuracion
0,1,LogisticRegression sin penalizacion; class_wei...,19907,0.500000,14662,782,2353,2110,0.842518,0.711071,0.729599,0.472776,0.573759,0.851435,0.677996,0.115604,sin_ponderacion
1,2,LogisticRegression sin penalizacion; class_wei...,19907,0.500000,14677,767,2302,2161,0.845833,0.717270,0.738046,0.484203,0.584765,0.853464,0.689448,0.113676,sin_ponderacion
2,3,LogisticRegression sin penalizacion; class_wei...,19907,0.500000,14634,810,2243,2220,0.846637,0.722488,0.732673,0.497423,0.592553,0.860367,0.687817,0.112312,sin_ponderacion
3,4,LogisticRegression sin penalizacion; class_wei...,19907,0.500000,14605,839,2349,2114,0.839855,0.709674,0.715882,0.473672,0.570119,0.849772,0.674705,0.116651,sin_ponderacion
4,5,LogisticRegression sin penalizacion; class_wei...,19907,0.500000,14614,831,2373,2089,0.839052,0.707186,0.715411,0.468176,0.565971,0.851679,0.674662,0.115984,sin_ponderacion
5,1,LogisticRegression sin penalizacion; class_wei...,19907,0.500000,12321,3123,1167,3296,0.784498,0.768151,0.513476,0.738517,0.605771,0.851234,0.675639,0.152308,balanceada
6,2,LogisticRegression sin penalizacion; class_wei...,19907,0.500000,12369,3075,1173,3290,0.786608,0.769033,0.516889,0.737172,0.607684,0.853899,0.688865,0.150403,balanceada
7,3,LogisticRegression sin penalizacion; class_wei...,19907,0.500000,12330,3114,1068,3395,0.789923,0.779534,0.521585,0.760699,0.618848,0.860649,0.686180,0.149776,balanceada
8,4,LogisticRegression sin penalizacion; class_wei...,19907,0.500000,12344,3100,1187,3276,0.784649,0.766655,0.513802,0.734035,0.604484,0.850087,0.673144,0.152808,balanceada
9,5,LogisticRegression sin penalizacion; class_wei...,19907,0.500000,12259,3186,1145,3317,0.782438,0.768554,0.510072,0.743389,0.605016,0.851853,0.673226,0.152763,balanceada


,configuracion,tipo_umbral,n,threshold,TN,FP,FN,TP,accuracy,balanced_accuracy,precision,recall,F1,ROC_AUC,PR_AUC,Brier,seleccionada,regla_seleccion
0,sin_ponderacion,fijo_0_50,99535,0.500000,73192,4029,11620,10694,0.842779,0.713538,0.726347,0.479251,0.577477,0.853349,0.680930,0.114846,False,"Conservar configuraciones dentro de 0,005 del ..."
1,sin_ponderacion,OOF_optimizado,99535,0.300000,66774,10447,7542,14772,0.819270,0.763359,0.585749,0.662006,0.621547,0.853349,0.680930,0.114846,False,"Conservar configuraciones dentro de 0,005 del ..."
2,balanceada,fijo_0_50,99535,0.500000,61623,15598,5740,16574,0.785623,0.770385,0.515168,0.742762,0.608376,0.853549,0.679420,0.151612,False,"Conservar configuraciones dentro de 0,005 del ..."
3,balanceada,OOF_optimizado,99535,0.600000,66715,10506,7515,14799,0.818948,0.763582,0.584825,0.663216,0.621559,0.853549,0.679420,0.151612,True,"Conservar configuraciones dentro de 0,005 del ..."


,decil,n,probabilidad_media,proporcion_observada,configuracion
0,0,9954,0.021573,0.011955,sin_ponderacion
1,1,9953,0.036270,0.027429,sin_ponderacion
2,2,9954,0.051872,0.053044,sin_ponderacion
3,3,9953,0.071739,0.076359,sin_ponderacion
4,4,9954,0.099136,0.107093,sin_ponderacion
5,5,9953,0.139433,0.152919,sin_ponderacion
6,6,9953,0.201599,0.203155,sin_ponderacion
7,7,9954,0.307764,0.308620,sin_ponderacion
8,8,9953,0.500611,0.495730,sin_ponderacion
9,9,9954,0.811846,0.805505,sin_ponderacion


,configuracion,Brier_OOF,mejor_Brier_OOF
0,sin_ponderacion,0.114846,True
1,balanceada,0.151612,False


,prevalencia_train,accuracy_clasificador_mayoritario,configuracion_predictiva_seleccionada,class_weight_seleccionado,umbral_operativo_out_of_fold,criterio_configuracion
0,0.224182,0.775818,balanceada,balanced,0.600000,"F1 OOF con tolerancia 0,005; desempate por bal..."


In [24]:
X_final_train = X_train_refined[FINAL_FEATURES].copy()
X_vif = sm.add_constant(X_final_train, has_constant='add')
vif_rows = []
for index, variable in enumerate(X_vif.columns):
    if variable == 'const':
        continue
    vif_rows.append({
        'variable': variable,
        'VIF': variance_inflation_factor(X_vif.to_numpy(), index)
    })
vif_table = pd.DataFrame(vif_rows).sort_values('VIF', ascending=False)
vif_table.to_csv(TABLES / '27_VIF_modelo_final.csv', index=False)
display(vif_table)
fig, ax = plt.subplots(figsize=(8, 5))
vif_table.sort_values('VIF').plot.barh(x='variable', y='VIF', legend=False, ax=ax)
ax.axvline(5, linestyle='--', linewidth=1)
ax.axvline(10, linestyle=':', linewidth=1)
ax.set_xlabel('VIF')
ax.set_ylabel('Variable')
ax.set_title('Multicolinealidad del modelo final')
fig.tight_layout()
fig.savefig(FIGURES / 'fig_08_VIF_modelo_final.png', dpi=170)
plt.close(fig)

final_design_train = sm.add_constant(X_final_train, has_constant='add')
final_probability_train = FINAL_RESULT.predict(final_design_train)

influence = FINAL_RESULT.get_influence(observed=True)
influence_frame = influence.summary_frame()
influence_frame['row_id'] = selected_train['row_id'].to_numpy()
influence_frame['residuo_deviance'] = FINAL_RESULT.resid_deviance
influence_frame['residuo_Pearson'] = FINAL_RESULT.resid_pearson
p_parameters = len(FINAL_RESULT.params)
n_train_final = len(y_train)
leverage_threshold = 2 * p_parameters / n_train_final
cook_threshold = 4 / n_train_final
influence_frame['alto_leverage'] = influence_frame['hat_diag'] > leverage_threshold
influence_frame['Cook_superior_umbral'] = influence_frame['cooks_d'] > cook_threshold
influence_frame['residuo_deviance_extremo'] = influence_frame['residuo_deviance'].abs() > 3
influence_frame.sort_values('cooks_d', ascending=False).head(200).to_csv(
    TABLES / '28_observaciones_influyentes_top200.csv', index=False
)

influence_summary = pd.DataFrame([{
    'n_train': n_train_final,
    'n_parametros': p_parameters,
    'umbral_leverage_2p_n': leverage_threshold,
    'umbral_Cook_4_n': cook_threshold,
    'n_alto_leverage': int(influence_frame['alto_leverage'].sum()),
    'n_Cook_superior_umbral': int(influence_frame['Cook_superior_umbral'].sum()),
    'n_residuo_deviance_absoluto_mayor_3': int(influence_frame['residuo_deviance_extremo'].sum())
}])
influence_summary.to_csv(TABLES / '29_resumen_residuos_influencia.csv', index=False)
display(influence_summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].scatter(final_probability_train, FINAL_RESULT.resid_deviance, s=7, alpha=0.20)
axes[0].axhline(0, linewidth=1)
axes[0].set_xlabel('Probabilidad ajustada')
axes[0].set_ylabel('Residuo de devianza')
axes[0].set_title('Residuos de devianza')
axes[1].scatter(influence_frame['hat_diag'], influence_frame['cooks_d'], s=7, alpha=0.20)
axes[1].axvline(leverage_threshold, linestyle='--', linewidth=1)
axes[1].axhline(cook_threshold, linestyle='--', linewidth=1)
axes[1].set_xlabel('Leverage')
axes[1].set_ylabel('Distancia de Cook')
axes[1].set_title('Influencia')
fig.tight_layout()
fig.savefig(FIGURES / 'fig_06_residuos_e_influencia.png', dpi=170)
plt.close(fig)

# Sensibilidad: exclusión de observaciones por encima del percentil 99,9 de Cook.
cook_cutoff = influence_frame['cooks_d'].quantile(0.999)
keep_mask = influence_frame['cooks_d'].to_numpy() <= cook_cutoff
sensitivity_result = sm.GLM(
    y_train.loc[keep_mask].reset_index(drop=True),
    sm.add_constant(X_final_train.loc[keep_mask].reset_index(drop=True), has_constant='add'),
    family=sm.families.Binomial()
).fit(maxiter=200, disp=0)
common_terms = FINAL_RESULT.params.index.intersection(sensitivity_result.params.index)
coefficient_change = pd.DataFrame({
    'termino': common_terms,
    'beta_modelo_final': FINAL_RESULT.params[common_terms].values,
    'beta_sin_0_1pct_mayor_Cook': sensitivity_result.params[common_terms].values
})
coefficient_change['cambio_absoluto'] = (
    coefficient_change['beta_sin_0_1pct_mayor_Cook'] - coefficient_change['beta_modelo_final']
).abs()
coefficient_change['cambio_relativo_porcentual'] = 100 * coefficient_change['cambio_absoluto'] / coefficient_change['beta_modelo_final'].abs().replace(0, np.nan)
coefficient_change.to_csv(TABLES / '30_sensibilidad_observaciones_influyentes.csv', index=False)
display(coefficient_change)


,variable,VIF
1,Rainfall_log1p,4.789744
10,Rainfall_log1p_sq,4.192526
3,MaxTemp,2.487495
5,Humidity9am,2.392374
7,Season_Winter,2.354628
0,Humidity3pm,2.266744
6,Season_Autumn,1.782906
8,Season_Spring,1.735824
4,WindGustSpeed,1.722456
2,Pressure3pm,1.597880


,n_train,n_parametros,umbral_leverage_2p_n,umbral_Cook_4_n,n_alto_leverage,n_Cook_superior_umbral,n_residuo_deviance_absoluto_mayor_3
0,99535,14,0.000281,0.000040,9085,7305,0


,termino,beta_modelo_final,beta_sin_0_1pct_mayor_Cook,cambio_absoluto,cambio_relativo_porcentual
0,const,-2.111505,-2.132758,0.021253,1.006531
1,Humidity3pm,1.128523,1.138501,0.009978,0.884189
2,Rainfall_log1p,0.422367,0.410218,0.012149,2.876485
3,Pressure3pm,-0.514223,-0.517764,0.003541,0.688612
4,MaxTemp,0.115169,0.103518,0.011651,10.116262
5,WindGustSpeed,0.499262,0.501471,0.002210,0.442591
6,Humidity9am,0.122485,0.123947,0.001461,1.192831
7,Season_Autumn,0.429122,0.432949,0.003826,0.891668
8,Season_Winter,0.664631,0.660803,0.003827,0.575835
9,Season_Spring,0.366340,0.369160,0.002819,0.769620


### 6.2 Interpretación ajustada del modelo final

Para variables continuas lineales se informa el odds ratio por una desviación estándar. Cuando existe un término cuadrático, se interpreta el contraste conjunto entre los percentiles 25 y 75 de entrenamiento. `RainToday_bin` se interpreta como Yes respecto de No y las dummies de `Season` respecto de `Summer`.


In [25]:
def linear_predictor_contrast(
    result,
    row_a,
    row_b
):
    row_a = pd.Series(
        row_a,
        index=result.params.index,
        dtype=float
    )
    row_b = pd.Series(
        row_b,
        index=result.params.index,
        dtype=float
    )
    difference = row_b - row_a
    log_or = float(
        difference @ result.params
    )
    variance = float(
        difference.to_numpy()
        @ result.cov_params().to_numpy()
        @ difference.to_numpy()
    )
    standard_error = math.sqrt(
        max(variance, 0)
    )
    return {
        'log_OR_contraste': log_or,
        'OR_contraste': math.exp(log_or),
        'IC95_OR_li': math.exp(
            log_or
            - 1.96 * standard_error
        ),
        'IC95_OR_ls': math.exp(
            log_or
            + 1.96 * standard_error
        )
    }, difference


reference_row = pd.Series(
    0.0,
    index=FINAL_RESULT.params.index
)
reference_row['const'] = 1.0

contrast_rows = []
contrast_vectors = {}

binary_descriptions = {
    'RainToday_bin': (
        'RainToday=Yes respecto de RainToday=No'
    ),
    'Season_Autumn': (
        'Autumn respecto de Summer'
    ),
    'Season_Winter': (
        'Winter respecto de Summer'
    ),
    'Season_Spring': (
        'Spring respecto de Summer'
    ),
}

contrast_base_features = []
for term in FINAL_FEATURES:
    base_term = (
        term[:-3]
        if term.endswith('_sq')
        else term
    )
    if base_term not in contrast_base_features:
        contrast_base_features.append(base_term)

for feature in contrast_base_features:
    row_a = reference_row.copy()
    row_b = reference_row.copy()

    if feature in binary_descriptions:
        row_a[feature] = 0
        row_b[feature] = 1
        description = (
            binary_descriptions[feature]
        )

    elif f'{feature}_sq' in FINAL_FEATURES:
        raw_values = selected_train[
            feature
        ]
        q25, q75 = raw_values.quantile(
            [0.25, 0.75]
        )
        reference = (
            scale_reference
            .set_index('variable')
            .loc[feature]
        )
        z25 = (
            q25
            - reference[
                'media_train_original'
            ]
        ) / reference[
            'desviacion_estandar_train_original'
        ]
        z75 = (
            q75
            - reference[
                'media_train_original'
            ]
        ) / reference[
            'desviacion_estandar_train_original'
        ]
        row_a[feature] = z25
        row_b[feature] = z75
        row_a[f'{feature}_sq'] = (
            z25 ** 2
        )
        row_b[f'{feature}_sq'] = (
            z75 ** 2
        )

        if feature == 'Rainfall_log1p':
            description = (
                'Percentil 75 respecto de '
                'percentil 25 '
                f'({np.expm1(q75):.4f} '
                f'frente a '
                f'{np.expm1(q25):.4f} mm)'
            )
        else:
            description = (
                'Percentil 75 respecto de '
                'percentil 25 '
                f'({q75:.4f} frente a '
                f'{q25:.4f} unidades '
                'originales)'
            )

    else:
        row_a[feature] = 0
        row_b[feature] = 1

        sd = (
            scale_reference
            .set_index('variable')
            .loc[
                feature,
                'desviacion_estandar_train_original'
            ]
        )
        unit_map = {
            'Humidity3pm': (
                'puntos porcentuales'
            ),
            'Humidity9am': (
                'puntos porcentuales'
            ),
            'Pressure3pm': 'hPa',
            'MaxTemp': '°C',
            'MinTemp': '°C',
            'WindGustSpeed': 'km/h',
            'Rainfall_log1p': (
                'unidades log1p(mm)'
            )
        }
        description = (
            'Aumento de 1 desviación '
            f'estándar ({sd:.4f} '
            f'{unit_map.get(feature, "unidades")})'
        )

    result_values, difference = (
        linear_predictor_contrast(
            FINAL_RESULT,
            row_a,
            row_b
        )
    )

    contrast_rows.append({
        'efecto': feature,
        'comparacion': description,
        **result_values
    })
    contrast_vectors[feature] = (
        difference
    )

adjusted_contrasts = pd.DataFrame(
    contrast_rows
)
adjusted_contrasts[
    'cambio_porcentual_odds'
] = (
    100 * (
        adjusted_contrasts[
            'OR_contraste'
        ] - 1
    )
)

adjusted_contrasts.to_csv(
    TABLES
    / '30b_contrastes_odds_ratio_modelo_final.csv',
    index=False
)
display(adjusted_contrasts)


,efecto,comparacion,log_OR_contraste,OR_contraste,IC95_OR_li,IC95_OR_ls,cambio_porcentual_odds
0,Humidity3pm,Percentil 75 respecto de percentil 25 (65.2009...,1.528182,4.609788,4.422222,4.805310,360.978823
1,Rainfall_log1p,Percentil 75 respecto de percentil 25 (0.8000 ...,0.296645,1.345337,1.310226,1.381389,34.533709
2,Pressure3pm,Aumento de 1 desviación estándar (6.7201 hPa),-0.514223,0.597965,0.584575,0.611661,-40.203498
3,MaxTemp,Percentil 75 respecto de percentil 25 (28.2000...,0.173626,1.189610,1.142414,1.238756,18.960998
4,WindGustSpeed,Percentil 75 respecto de percentil 25 (46.0000...,0.573219,1.773968,1.722591,1.826877,77.396787
5,Humidity9am,Aumento de 1 desviación estándar (19.0107 punt...,0.122485,1.130303,1.097376,1.164218,13.030269
6,Season_Autumn,Autumn respecto de Summer,0.429122,1.535909,1.452172,1.624474,53.590882
7,Season_Winter,Winter respecto de Summer,0.664631,1.943772,1.822411,2.073215,94.377220
8,Season_Spring,Spring respecto de Summer,0.366340,1.442446,1.362094,1.527538,44.244598


### 6.3 Suficiencia muestral, separación e independencia por localidad

Se verifican eventos por parámetro, convergencia, coeficientes y probabilidades extremas, y la sensibilidad de los errores estándar al agrupamiento por `Location`.


In [26]:
n_events = int(y_train.sum())
n_non_events = int((1 - y_train).sum())
n_parameters = int(len(FINAL_RESULT.params))
epv_events = n_events / n_parameters
epv_min_class = min(n_events, n_non_events) / n_parameters

fitted_train_inferential = np.asarray(
    FINAL_RESULT.predict(
        sm.add_constant(X_final_train, has_constant='add')
    ),
    dtype=float
)
finite_coefficients = bool(np.isfinite(FINAL_RESULT.params.to_numpy()).all())
finite_standard_errors = bool(np.isfinite(FINAL_RESULT.bse.to_numpy()).all())
extreme_coefficient = bool(
    np.nanmax(np.abs(FINAL_RESULT.params.to_numpy())) > 10
)
extreme_standard_error = bool(
    np.nanmax(FINAL_RESULT.bse.to_numpy()) > 10
)
near_boundary_probability = bool(
    np.any(fitted_train_inferential < 1e-8)
    or np.any(fitted_train_inferential > 1 - 1e-8)
)
separation_signal = (
    (not finite_coefficients)
    or (not finite_standard_errors)
    or extreme_coefficient
    or extreme_standard_error
    or near_boundary_probability
)

sample_separation_control = pd.DataFrame([{
    'n_train': len(y_train),
    'n_eventos_RainTomorrow_Yes': n_events,
    'n_no_eventos': n_non_events,
    'n_parametros_incluyendo_intercepto': n_parameters,
    'eventos_por_parametro': epv_events,
    'minima_clase_por_parametro': epv_min_class,
    'coeficientes_finitos': finite_coefficients,
    'errores_estandar_finitos': finite_standard_errors,
    'max_beta_absoluto': float(
        np.nanmax(np.abs(FINAL_RESULT.params.to_numpy()))
    ),
    'max_error_estandar': float(
        np.nanmax(FINAL_RESULT.bse.to_numpy())
    ),
    'probabilidad_cercana_a_0_o_1': near_boundary_probability,
    'senal_numerica_separacion': separation_signal,
    'convergio': bool(FINAL_RESULT.converged)
}])
sample_separation_control.to_csv(
    TABLES / '27e_suficiencia_muestral_y_separacion.csv',
    index=False
)

cluster_groups = (
    selected_train['Location']
    .reset_index(drop=True)
)
cluster_robust_result = sm.GLM(
    y_train,
    sm.add_constant(X_final_train, has_constant='add'),
    family=sm.families.Binomial()
).fit(
    maxiter=200,
    disp=0,
    cov_type='cluster',
    cov_kwds={'groups': cluster_groups}
)

cluster_confidence = cluster_robust_result.conf_int(alpha=ALPHA)
conventional_confidence = FINAL_RESULT.conf_int(alpha=ALPHA)
cluster_rows = []
for term in FINAL_RESULT.params.index:
    cluster_rows.append({
        'termino': term,
        'beta': float(FINAL_RESULT.params[term]),
        'SE_convencional': float(FINAL_RESULT.bse[term]),
        'p_convencional': float(FINAL_RESULT.pvalues[term]),
        'IC95_convencional_li': float(conventional_confidence.loc[term, 0]),
        'IC95_convencional_ls': float(conventional_confidence.loc[term, 1]),
        'SE_robusto_cluster_Location': float(cluster_robust_result.bse[term]),
        'p_robusto_cluster_Location': float(cluster_robust_result.pvalues[term]),
        'IC95_cluster_li': float(cluster_confidence.loc[term, 0]),
        'IC95_cluster_ls': float(cluster_confidence.loc[term, 1]),
        'razon_SE_cluster_convencional': float(
            cluster_robust_result.bse[term] / FINAL_RESULT.bse[term]
        ),
        'cambia_conclusion_IC_incluye_cero': bool(
            (
                conventional_confidence.loc[term, 0] <= 0
                <= conventional_confidence.loc[term, 1]
            )
            != (
                cluster_confidence.loc[term, 0] <= 0
                <= cluster_confidence.loc[term, 1]
            )
        )
    })
cluster_robust_sensitivity = pd.DataFrame(cluster_rows)
cluster_robust_sensitivity.to_csv(
    TABLES / '27f_sensibilidad_errores_robustos_cluster_Location.csv',
    index=False
)

display(sample_separation_control)
display(cluster_robust_sensitivity)


,n_train,n_eventos_RainTomorrow_Yes,n_no_eventos,n_parametros_incluyendo_intercepto,eventos_por_parametro,minima_clase_por_parametro,coeficientes_finitos,errores_estandar_finitos,max_beta_absoluto,max_error_estandar,probabilidad_cercana_a_0_o_1,senal_numerica_separacion,convergio
0,99535,22314,77221,14,"1,593.857143","1,593.857143",True,True,2.111505,0.032893,False,False,True


,termino,beta,SE_convencional,p_convencional,IC95_convencional_li,IC95_convencional_ls,SE_robusto_cluster_Location,p_robusto_cluster_Location,IC95_cluster_li,IC95_cluster_ls,razon_SE_cluster_convencional,cambia_conclusion_IC_incluye_cero
0,const,-2.111505,0.025344,0.000000,-2.161178,-2.061831,0.086513,0.000000,-2.281067,-1.941942,3.413517,False
1,Humidity3pm,1.128523,0.015346,0.000000,1.098445,1.158600,0.059971,0.000000,1.010982,1.246063,3.907904,False
2,Rainfall_log1p,0.422367,0.017946,0.000000,0.387193,0.457542,0.050800,0.000000,0.322801,0.521934,2.830657,False
3,Pressure3pm,-0.514223,0.011554,0.000000,-0.536869,-0.491577,0.057792,0.000000,-0.627494,-0.400952,5.001780,False
4,MaxTemp,0.115169,0.014283,0.000000,0.087175,0.143162,0.072979,0.114541,-0.027868,0.258205,5.109623,True
5,WindGustSpeed,0.499262,0.012508,0.000000,0.474746,0.523778,0.037143,0.000000,0.426462,0.572061,2.969466,False
6,Humidity9am,0.122485,0.015084,0.000000,0.092922,0.152049,0.052900,0.020591,0.018803,0.226168,3.507140,False
7,Season_Autumn,0.429122,0.028603,0.000000,0.373061,0.485183,0.095832,0.000008,0.241294,0.616950,3.350429,False
8,Season_Winter,0.664631,0.032893,0.000000,0.600161,0.729100,0.176304,0.000163,0.319081,1.010180,5.359923,False
9,Season_Spring,0.366340,0.029243,0.000000,0.309024,0.423656,0.088115,0.000032,0.193637,0.539043,3.013172,False


## 7. Estabilidad mediante bootstrap

Se aplica bootstrap no paramétrico sobre las filas de entrenamiento, con 10.000 réplicas del modelo final. Los intervalos percentiles al 95 % se comparan con los intervalos de Wald. Las réplicas con problemas de convergencia se descartan y se exige una tasa de éxito mínima de 95 %.

El bootstrap evalúa la estabilidad condicional a la estrategia de imputación, la escala y la especificación seleccionada. Se aplica al modelo inferencial sin ponderación; el balanceo se utiliza únicamente en la evaluación predictiva.


In [27]:
X_boot = X_final_train.to_numpy(dtype=float)
y_boot = y_train.to_numpy(dtype=int)
feature_names_boot = FINAL_FEATURES.copy()
rng_boot = np.random.default_rng(BOOTSTRAP_SEED)
bootstrap_coefficients = []
bootstrap_events = []
start_time = time.time()

for replicate in range(1, N_BOOT + 1):
    if (
        replicate == 1
        or replicate % 500 == 0
        or replicate == N_BOOT
    ):
        elapsed_minutes = (
            time.time() - start_time
        ) / 60
        print(
            f'Bootstrap inferencial: '
            f'{replicate:,}/{N_BOOT:,} réplicas; '
            f'{elapsed_minutes:.1f} minutos transcurridos.'
        )

    sample_index = rng_boot.integers(0, len(y_boot), size=len(y_boot))
    X_sample = X_boot[sample_index]
    y_sample = y_boot[sample_index]

    if np.unique(y_sample).size < 2:
        bootstrap_events.append({
            'replica': replicate,
            'tipo': 'error',
            'detalle': 'La remuestra contiene una sola clase.'
        })
        continue

    try:
        model = LogisticRegression(
            penalty=None, solver='lbfgs', max_iter=2_000
        )
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            model.fit(X_sample, y_sample)

        convergence_warnings = [
            item for item in caught
            if issubclass(item.category, ConvergenceWarning)
        ]
        if convergence_warnings:
            for warning in convergence_warnings:
                bootstrap_events.append({
                    'replica': replicate,
                    'tipo': 'error_convergencia',
                    'detalle': str(warning.message)
                })
            continue

        for warning in caught:
            bootstrap_events.append({
                'replica': replicate,
                'tipo': 'advertencia',
                'detalle': str(warning.message)
            })

        values = np.concatenate([model.intercept_, model.coef_.ravel()])
        if not np.isfinite(values).all():
            raise FloatingPointError('Se obtuvieron coeficientes no finitos.')
        bootstrap_coefficients.append([replicate, *values])

    except Exception as error:
        bootstrap_events.append({
            'replica': replicate,
            'tipo': 'error',
            'detalle': str(error)
        })

bootstrap_columns = ['replica', 'const', *feature_names_boot]
bootstrap_distribution = pd.DataFrame(
    bootstrap_coefficients, columns=bootstrap_columns
)
bootstrap_event_table = pd.DataFrame(
    bootstrap_events,
    columns=['replica', 'tipo', 'detalle']
)
bootstrap_distribution.to_csv(
    TABLES / '31a_distribuciones_bootstrap_coeficientes.csv', index=False
)
bootstrap_event_table.to_csv(
    TABLES / '31b_fallos_advertencias_bootstrap.csv', index=False
)

bootstrap_success_rate = len(bootstrap_distribution) / N_BOOT
bootstrap_control = pd.DataFrame([{
    'metodo': 'bootstrap no parametrico n-out-of-n',
    'estimador': 'LogisticRegression sin ponderacion',
    'replicas_solicitadas': N_BOOT,
    'replicas_exitosas': len(bootstrap_distribution),
    'replicas_sin_coeficientes': N_BOOT - len(bootstrap_distribution),
    'tasa_exito': bootstrap_success_rate,
    'tasa_minima_exigida': MIN_BOOTSTRAP_SUCCESS,
    'cumple_tasa_minima': bootstrap_success_rate >= MIN_BOOTSTRAP_SUCCESS
}])
bootstrap_control.to_csv(
    TABLES / '31c_control_convergencia_bootstrap.csv', index=False
)
display(bootstrap_control)

if bootstrap_success_rate < MIN_BOOTSTRAP_SUCCESS:
    raise RuntimeError(
        f'El bootstrap alcanzó una tasa de éxito de '
        f'{bootstrap_success_rate:.2%}, inferior al mínimo '
        f'exigido de {MIN_BOOTSTRAP_SUCCESS:.0%}.'
    )

wald_confidence = FINAL_RESULT.conf_int(alpha=ALPHA)
bootstrap_rows = []
for term in ['const', *feature_names_boot]:
    values = bootstrap_distribution[term].dropna().to_numpy()
    lower, upper = np.percentile(values, [2.5, 97.5])
    beta = FINAL_RESULT.params[term]
    sign_consistency = max(np.mean(values > 0), np.mean(values < 0))
    bootstrap_rows.append({
        'termino': term,
        'beta_modelo_final': beta,
        'IC95_Wald_beta_li': wald_confidence.loc[term, 0],
        'IC95_Wald_beta_ls': wald_confidence.loc[term, 1],
        'IC95_bootstrap_beta_li': lower,
        'IC95_bootstrap_beta_ls': upper,
        'OR_modelo_final': np.exp(beta),
        'IC95_Wald_OR_li': np.exp(wald_confidence.loc[term, 0]),
        'IC95_Wald_OR_ls': np.exp(wald_confidence.loc[term, 1]),
        'IC95_bootstrap_OR_li': np.exp(lower),
        'IC95_bootstrap_OR_ls': np.exp(upper),
        'consistencia_signo': sign_consistency,
        'intervalo_bootstrap_incluye_cero': lower <= 0 <= upper,
        'parametro_inestable': bool(
            lower <= 0 <= upper or sign_consistency < 0.95
        )
    })

bootstrap_summary = pd.DataFrame(bootstrap_rows)
bootstrap_summary.to_csv(
    TABLES / '32_estabilidad_coeficientes_odds_ratios_bootstrap.csv',
    index=False
)
display(bootstrap_summary)

terms_to_plot = [
    term for term in feature_names_boot
    if term in bootstrap_distribution.columns
]
n_columns = 2
n_rows = math.ceil(len(terms_to_plot) / n_columns)
fig, axes = plt.subplots(
    n_rows, n_columns, figsize=(11, 3.5 * n_rows)
)
axes = np.atleast_1d(axes).ravel()
for axis, term in zip(axes, terms_to_plot):
    axis.hist(
        bootstrap_distribution[term].dropna(),
        bins=35, density=True, alpha=0.75
    )
    axis.axvline(FINAL_RESULT.params[term], linewidth=1.5)
    axis.axvline(0, linestyle='--', linewidth=1)
    axis.set_title(term)
    axis.set_xlabel('Coeficiente bootstrap')
for axis in axes[len(terms_to_plot):]:
    axis.axis('off')
fig.tight_layout()
fig.savefig(
    FIGURES / 'fig_10_distribuciones_bootstrap_coeficientes.png',
    dpi=170
)
plt.close(fig)


Bootstrap inferencial: 1/10,000 réplicas; 0.0 minutos transcurridos.
Bootstrap inferencial: 500/10,000 réplicas; 1.3 minutos transcurridos.
Bootstrap inferencial: 1,000/10,000 réplicas; 2.5 minutos transcurridos.
Bootstrap inferencial: 1,500/10,000 réplicas; 3.8 minutos transcurridos.
Bootstrap inferencial: 2,000/10,000 réplicas; 5.0 minutos transcurridos.
Bootstrap inferencial: 2,500/10,000 réplicas; 6.2 minutos transcurridos.
Bootstrap inferencial: 3,000/10,000 réplicas; 7.4 minutos transcurridos.
Bootstrap inferencial: 3,500/10,000 réplicas; 8.7 minutos transcurridos.
Bootstrap inferencial: 4,000/10,000 réplicas; 10.0 minutos transcurridos.
Bootstrap inferencial: 4,500/10,000 réplicas; 11.3 minutos transcurridos.
Bootstrap inferencial: 5,000/10,000 réplicas; 12.5 minutos transcurridos.
Bootstrap inferencial: 5,500/10,000 réplicas; 13.8 minutos transcurridos.
Bootstrap inferencial: 6,000/10,000 réplicas; 15.0 minutos transcurridos.
Bootstrap inferencial: 6,500/10,000 réplicas; 16.3 m

,metodo,estimador,replicas_solicitadas,replicas_exitosas,replicas_sin_coeficientes,tasa_exito,tasa_minima_exigida,cumple_tasa_minima
0,bootstrap no parametrico n-out-of-n,LogisticRegression sin ponderacion,10000,10000,0,1.000000,0.950000,True


,termino,beta_modelo_final,IC95_Wald_beta_li,IC95_Wald_beta_ls,IC95_bootstrap_beta_li,IC95_bootstrap_beta_ls,OR_modelo_final,IC95_Wald_OR_li,IC95_Wald_OR_ls,IC95_bootstrap_OR_li,IC95_bootstrap_OR_ls,consistencia_signo,intervalo_bootstrap_incluye_cero,parametro_inestable
0,const,-2.111505,-2.161178,-2.061831,-2.161039,-2.062818,0.121056,0.115189,0.127221,0.115205,0.127095,1.000000,False,False
1,Humidity3pm,1.128523,1.098445,1.158600,1.099557,1.157679,3.091086,2.999498,3.185471,3.002834,3.182538,1.000000,False,False
2,Rainfall_log1p,0.422367,0.387193,0.457542,0.386221,0.457821,1.525569,1.472841,1.580185,1.471410,1.580627,1.000000,False,False
3,Pressure3pm,-0.514223,-0.536869,-0.491577,-0.537876,-0.491303,0.597965,0.584576,0.611661,0.583988,0.611829,1.000000,False,False
4,MaxTemp,0.115169,0.087175,0.143162,0.086450,0.143687,1.122063,1.091088,1.153917,1.090297,1.154523,1.000000,False,False
5,WindGustSpeed,0.499262,0.474746,0.523778,0.473975,0.524536,1.647504,1.607605,1.688394,1.606367,1.689674,1.000000,False,False
6,Humidity9am,0.122485,0.092922,0.152049,0.093795,0.152687,1.130303,1.097376,1.164217,1.098334,1.164960,1.000000,False,False
7,Season_Autumn,0.429122,0.373061,0.485183,0.373552,0.485390,1.535909,1.452173,1.624473,1.452886,1.624809,1.000000,False,False
8,Season_Winter,0.664631,0.600161,0.729100,0.600669,0.726263,1.943772,1.822413,2.073213,1.823339,2.067341,1.000000,False,False
9,Season_Spring,0.366340,0.309024,0.423656,0.309212,0.424462,1.442446,1.362095,1.527536,1.362351,1.528768,1.000000,False,False


In [28]:
# Estabilidad acumulada de los intervalos bootstrap.
bootstrap_checkpoints = [
    checkpoint
    for checkpoint in [1_000, 2_500, 5_000, 7_500, 10_000]
    if checkpoint <= len(bootstrap_distribution)
]
bootstrap_cumulative_rows = []
for checkpoint in bootstrap_checkpoints:
    subset = bootstrap_distribution.iloc[:checkpoint]
    for term in ['const', *feature_names_boot]:
        values = subset[term].dropna().to_numpy(dtype=float)
        lower, upper = np.percentile(values, [2.5, 97.5])
        bootstrap_cumulative_rows.append({
            'replicas_acumuladas': checkpoint,
            'termino': term,
            'media_bootstrap': float(np.mean(values)),
            'error_estandar_bootstrap': float(np.std(values, ddof=1)),
            'IC95_bootstrap_li': float(lower),
            'IC95_bootstrap_ls': float(upper),
            'consistencia_signo': float(
                max(np.mean(values > 0), np.mean(values < 0))
            )
        })

bootstrap_cumulative_stability = pd.DataFrame(
    bootstrap_cumulative_rows
)
bootstrap_cumulative_stability.to_csv(
    TABLES / '31d_estabilidad_acumulada_bootstrap.csv',
    index=False
)
display(bootstrap_cumulative_stability)


,replicas_acumuladas,termino,media_bootstrap,error_estandar_bootstrap,IC95_bootstrap_li,IC95_bootstrap_ls,consistencia_signo
0,1000,const,-2.111862,0.025246,-2.162939,-2.064120,1.000000
1,1000,Humidity3pm,1.128137,0.014699,1.096838,1.155986,1.000000
2,1000,Rainfall_log1p,0.422099,0.018360,0.385363,0.459028,1.000000
3,1000,Pressure3pm,-0.514673,0.011614,-0.536747,-0.491611,1.000000
4,1000,MaxTemp,0.115163,0.014852,0.086431,0.145970,1.000000
...,...,...,...,...,...,...,...
65,10000,Season_Spring,0.366777,0.029118,0.309212,0.424462,1.000000
66,10000,Humidity3pm_sq,0.220328,0.009271,0.202510,0.238537,1.000000
67,10000,Rainfall_log1p_sq,-0.076087,0.007254,-0.090148,-0.061695,1.000000
68,10000,MaxTemp_sq,-0.092296,0.009167,-0.109887,-0.073979,1.000000


### 7.1 Interpretación sustantiva y estabilidad de los efectos

Los efectos lineales se interpretan por una desviación estándar, las variables binarias mediante contrastes de categorías y los componentes no lineales mediante contrastes conjuntos. Los intervalos bootstrap se calculan sobre esos mismos contrastes.


In [29]:
bootstrap_parameter_matrix = bootstrap_distribution[
    ['const', *FINAL_FEATURES]
].to_numpy(dtype=float)

stable_s2_lookup = set(stable_predictors['variable'])
final_interpretation_rows = []

for _, contrast in adjusted_contrasts.iterrows():
    effect = contrast['efecto']
    difference = contrast_vectors[effect].reindex(
        ['const', *FINAL_FEATURES], fill_value=0.0
    ).to_numpy(dtype=float)
    bootstrap_log_or = bootstrap_parameter_matrix @ difference
    bootstrap_or = np.exp(bootstrap_log_or)
    boot_lower, boot_upper = np.percentile(bootstrap_or, [2.5, 97.5])

    if effect in SEASON_DUMMY_FEATURES:
        source_variable = 'Season'
        associated_terms = [effect]
        s1_correlation = np.nan
        validated_s2 = False
    else:
        source_variable = (
            'Rainfall'
            if effect == 'Rainfall_log1p'
            else effect
        )
        associated_terms = [
            term
            for term in [
                effect,
                f'{effect}_sq'
            ]
            if term in FINAL_FEATURES
        ]
        s1_correlation = (
            float(
                s1_corr.loc[
                    source_variable,
                    'RainTomorrow_bin'
                ]
            )
            if (
                source_variable
                in s1_corr.index
                and 'RainTomorrow_bin'
                in s1_corr.columns
            )
            else np.nan
        )
        validated_s2 = (
            source_variable
            in stable_s2_lookup
        )

    contrast_stable = not (boot_lower <= 1 <= boot_upper)
    direction = (
        'aumento' if contrast['OR_contraste'] > 1
        else 'disminución'
    )
    change = abs(100 * (contrast['OR_contraste'] - 1))
    adjusted_direction = np.sign(np.log(contrast['OR_contraste']))
    marginal_direction = np.sign(s1_correlation) if pd.notna(s1_correlation) else np.nan
    direction_consistent = (
        bool(adjusted_direction == marginal_direction)
        if pd.notna(marginal_direction)
        else np.nan
    )

    if pd.notna(s1_correlation):
        direction_note = (
            'La dirección ajustada coincide con la asociación marginal de S1.'
            if direction_consistent
            else (
                'La dirección ajustada difiere de la asociación marginal de S1; '
                'esto se interpreta como un efecto condicional asociado a redundancia o supresión '
                'entre predictores y no como una contradicción causal.'
            )
        )
        evidence_text = (
            f'En S1, r con RainTomorrow={s1_correlation:.4f}; '
            f'la relación con el objetivo '
            f'{"fue validada como estable en S2" if validated_s2 else "no fue validada directamente en S2"}. '
            f'{direction_note}'
        )
    else:
        direction_consistent = np.nan
        evidence_text = (
            'Season se incorporó como variable categórica de baja '
            'cardinalidad, codificada con Summer como referencia. '
            'Su aporte fue evaluado dentro de los procedimientos '
            'automáticos de selección.'
        )

    narrative = (
        f'{contrast["comparacion"]}: manteniendo constantes los demás '
        f'predictores, se estimó una {direction} de {change:.2f}% en las '
        f'odds de lluvia (OR={contrast["OR_contraste"]:.4f}; '
        f'IC95% Wald [{contrast["IC95_OR_li"]:.4f}, '
        f'{contrast["IC95_OR_ls"]:.4f}]; IC95% bootstrap '
        f'[{boot_lower:.4f}, {boot_upper:.4f}]). '
        f'El efecto se clasifica como '
        f'{"estable" if contrast_stable else "inestable"} porque el '
        f'intervalo bootstrap {"excluye" if contrast_stable else "incluye"} 1. '
        f'{evidence_text}'
    )

    final_interpretation_rows.append({
        'efecto': effect,
        'parametros_asociados': ', '.join(associated_terms),
        'comparacion': contrast['comparacion'],
        'OR_ajustado': contrast['OR_contraste'],
        'IC95_Wald_OR_li': contrast['IC95_OR_li'],
        'IC95_Wald_OR_ls': contrast['IC95_OR_ls'],
        'IC95_bootstrap_OR_li': boot_lower,
        'IC95_bootstrap_OR_ls': boot_upper,
        'efecto_estable_bootstrap': contrast_stable,
        'correlacion_S1_con_objetivo': s1_correlation,
        'validada_estable_en_S2': validated_s2,
        'direccion_ajustada_coincide_con_S1': direction_consistent,
        'interpretacion': narrative
    })

final_effect_interpretation = pd.DataFrame(final_interpretation_rows)
final_effect_interpretation.to_csv(
    TABLES / '32b_interpretacion_final_efectos_y_estabilidad.csv',
    index=False
)
display(final_effect_interpretation)
for text in final_effect_interpretation['interpretacion']:
    display(Markdown(f'- {text}'))

,efecto,parametros_asociados,comparacion,OR_ajustado,IC95_Wald_OR_li,IC95_Wald_OR_ls,IC95_bootstrap_OR_li,IC95_bootstrap_OR_ls,efecto_estable_bootstrap,correlacion_S1_con_objetivo,validada_estable_en_S2,direccion_ajustada_coincide_con_S1,interpretacion
0,Humidity3pm,"Humidity3pm, Humidity3pm_sq",Percentil 75 respecto de percentil 25 (65.2009...,4.609788,4.422222,4.805310,4.429129,4.798833,True,0.446000,True,True,Percentil 75 respecto de percentil 25 (65.2009...
1,Rainfall_log1p,"Rainfall_log1p, Rainfall_log1p_sq",Percentil 75 respecto de percentil 25 (0.8000 ...,1.345337,1.310226,1.381389,1.309361,1.381838,True,0.239000,True,True,Percentil 75 respecto de percentil 25 (0.8000 ...
2,Pressure3pm,Pressure3pm,Aumento de 1 desviación estándar (6.7201 hPa),0.597965,0.584575,0.611661,0.583988,0.611829,True,-0.226000,True,True,Aumento de 1 desviación estándar (6.7201 hPa):...
3,MaxTemp,"MaxTemp, MaxTemp_sq",Percentil 75 respecto de percentil 25 (28.2000...,1.189610,1.142414,1.238756,1.140960,1.239814,True,-0.159000,True,False,Percentil 75 respecto de percentil 25 (28.2000...
4,WindGustSpeed,"WindGustSpeed, WindGustSpeed_sq",Percentil 75 respecto de percentil 25 (46.0000...,1.773968,1.722591,1.826877,1.721145,1.828275,True,0.234000,False,True,Percentil 75 respecto de percentil 25 (46.0000...
5,Humidity9am,Humidity9am,Aumento de 1 desviación estándar (19.0107 punt...,1.130303,1.097376,1.164218,1.098334,1.164960,True,0.257000,False,True,Aumento de 1 desviación estándar (19.0107 punt...
6,Season_Autumn,Season_Autumn,Autumn respecto de Summer,1.535909,1.452172,1.624474,1.452886,1.624809,True,NaN,False,NaN,Autumn respecto de Summer: manteniendo constan...
7,Season_Winter,Season_Winter,Winter respecto de Summer,1.943772,1.822411,2.073215,1.823339,2.067341,True,NaN,False,NaN,Winter respecto de Summer: manteniendo constan...
8,Season_Spring,Season_Spring,Spring respecto de Summer,1.442446,1.362094,1.527538,1.362351,1.528768,True,NaN,False,NaN,Spring respecto de Summer: manteniendo constan...


- Percentil 75 respecto de percentil 25 (65.2009 frente a 37.0000 unidades originales): manteniendo constantes los demás predictores, se estimó una aumento de 360.98% en las odds de lluvia (OR=4.6098; IC95% Wald [4.4222, 4.8053]; IC95% bootstrap [4.4291, 4.7988]). El efecto se clasifica como estable porque el intervalo bootstrap excluye 1. En S1, r con RainTomorrow=0.4460; la relación con el objetivo fue validada como estable en S2. La dirección ajustada coincide con la asociación marginal de S1.

- Percentil 75 respecto de percentil 25 (0.8000 frente a 0.0000 mm): manteniendo constantes los demás predictores, se estimó una aumento de 34.53% en las odds de lluvia (OR=1.3453; IC95% Wald [1.3102, 1.3814]; IC95% bootstrap [1.3094, 1.3818]). El efecto se clasifica como estable porque el intervalo bootstrap excluye 1. En S1, r con RainTomorrow=0.2390; la relación con el objetivo fue validada como estable en S2. La dirección ajustada coincide con la asociación marginal de S1.

- Aumento de 1 desviación estándar (6.7201 hPa): manteniendo constantes los demás predictores, se estimó una disminución de 40.20% en las odds de lluvia (OR=0.5980; IC95% Wald [0.5846, 0.6117]; IC95% bootstrap [0.5840, 0.6118]). El efecto se clasifica como estable porque el intervalo bootstrap excluye 1. En S1, r con RainTomorrow=-0.2260; la relación con el objetivo fue validada como estable en S2. La dirección ajustada coincide con la asociación marginal de S1.

- Percentil 75 respecto de percentil 25 (28.2000 frente a 17.9000 unidades originales): manteniendo constantes los demás predictores, se estimó una aumento de 18.96% en las odds de lluvia (OR=1.1896; IC95% Wald [1.1424, 1.2388]; IC95% bootstrap [1.1410, 1.2398]). El efecto se clasifica como estable porque el intervalo bootstrap excluye 1. En S1, r con RainTomorrow=-0.1590; la relación con el objetivo fue validada como estable en S2. La dirección ajustada difiere de la asociación marginal de S1; esto se interpreta como un efecto condicional asociado a redundancia o supresión entre predictores y no como una contradicción causal.

- Percentil 75 respecto de percentil 25 (46.0000 frente a 31.0000 unidades originales): manteniendo constantes los demás predictores, se estimó una aumento de 77.40% en las odds de lluvia (OR=1.7740; IC95% Wald [1.7226, 1.8269]; IC95% bootstrap [1.7211, 1.8283]). El efecto se clasifica como estable porque el intervalo bootstrap excluye 1. En S1, r con RainTomorrow=0.2340; la relación con el objetivo no fue validada directamente en S2. La dirección ajustada coincide con la asociación marginal de S1.

- Aumento de 1 desviación estándar (19.0107 puntos porcentuales): manteniendo constantes los demás predictores, se estimó una aumento de 13.03% en las odds de lluvia (OR=1.1303; IC95% Wald [1.0974, 1.1642]; IC95% bootstrap [1.0983, 1.1650]). El efecto se clasifica como estable porque el intervalo bootstrap excluye 1. En S1, r con RainTomorrow=0.2570; la relación con el objetivo no fue validada directamente en S2. La dirección ajustada coincide con la asociación marginal de S1.

- Autumn respecto de Summer: manteniendo constantes los demás predictores, se estimó una aumento de 53.59% en las odds de lluvia (OR=1.5359; IC95% Wald [1.4522, 1.6245]; IC95% bootstrap [1.4529, 1.6248]). El efecto se clasifica como estable porque el intervalo bootstrap excluye 1. Season se incorporó como variable categórica de baja cardinalidad, codificada con Summer como referencia. Su aporte fue evaluado dentro de los procedimientos automáticos de selección.

- Winter respecto de Summer: manteniendo constantes los demás predictores, se estimó una aumento de 94.38% en las odds de lluvia (OR=1.9438; IC95% Wald [1.8224, 2.0732]; IC95% bootstrap [1.8233, 2.0673]). El efecto se clasifica como estable porque el intervalo bootstrap excluye 1. Season se incorporó como variable categórica de baja cardinalidad, codificada con Summer como referencia. Su aporte fue evaluado dentro de los procedimientos automáticos de selección.

- Spring respecto de Summer: manteniendo constantes los demás predictores, se estimó una aumento de 44.24% en las odds de lluvia (OR=1.4424; IC95% Wald [1.3621, 1.5275]; IC95% bootstrap [1.3624, 1.5288]). El efecto se clasifica como estable porque el intervalo bootstrap excluye 1. Season se incorporó como variable categórica de baja cardinalidad, codificada con Summer como referencia. Su aporte fue evaluado dentro de los procedimientos automáticos de selección.

## 8. Apertura del conjunto de prueba y evaluación final

El conjunto de prueba se utiliza una vez definidas con entrenamiento la estrategia de imputación, las variables, la forma funcional, el modelo y el umbral. Sus resultados cuantifican desempeño final y no modifican decisiones previas.

Las métricas principales corresponden a la configuración predictiva seleccionada. El ajuste no ponderado se conserva para inferencia, criterios de información, odds ratios y bootstrap.


In [30]:
# Preparación del test con parámetros estimados exclusivamente en entrenamiento.
X_test = selected_test[CANDIDATE_FEATURES].copy().reset_index(drop=True)
X_test[CONTINUOUS_FEATURES] = scaler.transform(
    selected_test[CONTINUOUS_FEATURES]
)
y_test = (
    selected_test['RainTomorrow_bin']
    .astype(int)
    .reset_index(drop=True)
)

X_test_refined = X_test.copy()
for variable in nonlinear_variables if KEEP_REFINEMENT else []:
    X_test_refined[f'{variable}_sq'] = (
        X_test_refined[variable] ** 2
    )

X_final_test = X_test_refined[FINAL_FEATURES].copy()
final_design_test = sm.add_constant(
    X_final_test,
    has_constant='add'
)

# Probabilidad del modelo inferencial sin ponderación.
final_probability_train_inferential = np.asarray(
    FINAL_RESULT.predict(
        sm.add_constant(X_final_train, has_constant='add')
    ),
    dtype=float
)
final_probability_test_inferential = np.asarray(
    FINAL_RESULT.predict(final_design_test),
    dtype=float
)

# Configuraciones predictivas con la misma especificación.
FINAL_PREDICTIVE_MODEL_UNWEIGHTED = LogisticRegression(
    penalty=None,
    solver='lbfgs',
    max_iter=2_000,
    class_weight=None
)
FINAL_PREDICTIVE_MODEL_BALANCED = LogisticRegression(
    penalty=None,
    solver='lbfgs',
    max_iter=2_000,
    class_weight='balanced'
)

predictive_fit_warnings = []
for model_name, model in {
    'sin_ponderacion': FINAL_PREDICTIVE_MODEL_UNWEIGHTED,
    'balanceada': FINAL_PREDICTIVE_MODEL_BALANCED
}.items():
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always')
        model.fit(X_final_train, y_train)
    convergence_warnings = [
        item
        for item in caught
        if issubclass(item.category, ConvergenceWarning)
    ]
    if convergence_warnings:
        raise RuntimeError(
            f'La configuración predictiva {model_name} no convergió: '
            + ' | '.join(
                str(item.message)
                for item in convergence_warnings
            )
        )
    predictive_fit_warnings.extend([
        {
            'configuracion': model_name,
            'advertencia': str(item.message)
        }
        for item in caught
    ])

pd.DataFrame(
    predictive_fit_warnings,
    columns=['configuracion', 'advertencia']
).to_csv(
    TABLES / '20d_advertencias_configuraciones_predictivas.csv',
    index=False
)

final_probability_train_unweighted = (
    FINAL_PREDICTIVE_MODEL_UNWEIGHTED
    .predict_proba(X_final_train)[:, 1]
)
final_probability_test_unweighted = (
    FINAL_PREDICTIVE_MODEL_UNWEIGHTED
    .predict_proba(X_final_test)[:, 1]
)
final_probability_train_balanced = (
    FINAL_PREDICTIVE_MODEL_BALANCED
    .predict_proba(X_final_train)[:, 1]
)
final_probability_test_balanced = (
    FINAL_PREDICTIVE_MODEL_BALANCED
    .predict_proba(X_final_test)[:, 1]
)

if SELECTED_PREDICTIVE_CONFIGURATION == 'sin_ponderacion':
    FINAL_PREDICTIVE_MODEL = FINAL_PREDICTIVE_MODEL_UNWEIGHTED
    final_probability_train_selected = final_probability_train_unweighted
    final_probability_test_selected = final_probability_test_unweighted
else:
    FINAL_PREDICTIVE_MODEL = FINAL_PREDICTIVE_MODEL_BALANCED
    final_probability_train_selected = final_probability_train_balanced
    final_probability_test_selected = final_probability_test_balanced

# Alias utilizado por los análisis de sensibilidad posteriores.
final_probability_test = final_probability_test_selected

performance_rows = []
confusion_rows = []
roc_data = {'train': {}, 'test': {}}

for model_name, features in MODEL_DEFINITIONS.items():
    model = predictive_models_balanced[model_name]
    train_probability = model.predict_proba(
        X_train[features]
    )[:, 1]
    test_probability = model.predict_proba(
        X_test[features]
    )[:, 1]

    for dataset_name, y_true, probability in [
        ('train', y_train, train_probability),
        ('test', y_test, test_probability)
    ]:
        metrics = calculate_metrics(
            y_true,
            probability,
            threshold=0.50
        )
        performance_rows.append({
            'modelo': model_name,
            'conjunto': dataset_name,
            'estimador_predictivo': (
                'LogisticRegression sin penalizacion; '
                'class_weight=balanced'
            ),
            **metrics
        })
        confusion_rows.append({
            'modelo': model_name,
            'conjunto': dataset_name,
            'TN': metrics['TN'],
            'FP': metrics['FP'],
            'FN': metrics['FN'],
            'TP': metrics['TP']
        })
        roc_data[dataset_name][model_name] = (
            np.asarray(y_true),
            np.asarray(probability)
        )

performance_three_models = pd.DataFrame(performance_rows)
confusion_three_models = pd.DataFrame(confusion_rows)
performance_three_models.to_csv(
    TABLES / '20_metricas_train_test_tres_modelos_umbral_050.csv',
    index=False
)
confusion_three_models.to_csv(
    TABLES / '21_matrices_confusion_train_test_tres_modelos.csv',
    index=False
)
display(performance_three_models)
display(confusion_three_models)

for dataset_name in ['train', 'test']:
    fig, ax = plt.subplots(figsize=(7, 6))
    for model_name, (y_true, probability) in roc_data[dataset_name].items():
        false_positive_rate, true_positive_rate, _ = roc_curve(
            y_true,
            probability
        )
        auc = roc_auc_score(y_true, probability)
        ax.plot(
            false_positive_rate,
            true_positive_rate,
            label=f'{model_name} (AUC={auc:.3f})'
        )
    ax.plot([0, 1], [0, 1], linestyle='--', linewidth=1)
    ax.set_xlabel('Tasa de falsos positivos')
    ax.set_ylabel('Tasa de verdaderos positivos')
    ax.set_title(
        f'Curvas ROC — {dataset_name} '
        '(comparación de los tres modelos)'
    )
    ax.legend()
    fig.tight_layout()
    figure_number = 3 if dataset_name == 'train' else 4
    fig.savefig(
        FIGURES / (
            f'fig_0{figure_number}_ROC_'
            f'{dataset_name}_tres_modelos.png'
        ),
        dpi=170
    )
    plt.close(fig)

fig, ax = plt.subplots(figsize=(8, 5))
test_metrics_plot = (
    performance_three_models
    .loc[performance_three_models['conjunto'].eq('test')]
    .set_index('modelo')
)
test_metrics_plot[
    ['accuracy', 'precision', 'recall', 'F1']
].plot.bar(ax=ax)
ax.set_ylim(0, 1)
ax.set_ylabel('Métrica')
ax.set_title('Desempeño en prueba de los tres modelos')
fig.tight_layout()
fig.savefig(
    FIGURES / 'fig_05_metricas_test_tres_modelos.png',
    dpi=170
)
plt.close(fig)

test_configuration_rows = []
test_probability_lookup = {
    'sin_ponderacion': final_probability_test_unweighted,
    'balanceada': final_probability_test_balanced
}
for configuration, probability in test_probability_lookup.items():
    test_configuration_rows.append({
        'escenario': f'{configuration}_umbral_0_50',
        'configuracion': configuration,
        'tipo_umbral': 'fijo_0_50',
        'seleccionada_OOF': False,
        **calculate_metrics(y_test, probability, threshold=0.50)
    })
    test_configuration_rows.append({
        'escenario': f'{configuration}_umbral_OOF',
        'configuracion': configuration,
        'tipo_umbral': 'OOF_optimizado',
        'seleccionada_OOF': bool(
            configuration == SELECTED_PREDICTIVE_CONFIGURATION
        ),
        **calculate_metrics(
            y_test,
            probability,
            threshold=best_threshold_lookup[configuration]
        )
    })

inference_prediction_comparison = pd.DataFrame([
    {
        'escenario': 'inferencial_statsmodels_umbral_0_50',
        'configuracion': 'inferencial_sin_ponderacion',
        'tipo_umbral': 'fijo_0_50',
        'seleccionada_OOF': False,
        **calculate_metrics(
            y_test,
            final_probability_test_inferential,
            threshold=0.50
        )
    },
    *test_configuration_rows
])
inference_prediction_comparison.to_csv(
    TABLES / '20b_comparacion_inferencia_y_prediccion_balanceada.csv',
    index=False
)

final_test_metrics = inference_prediction_comparison.loc[
    inference_prediction_comparison['seleccionada_OOF']
].reset_index(drop=True)
final_test_metrics.to_csv(
    TABLES / '22b_desempeno_final_test_umbral_050_y_operativo.csv',
    index=False
)

decision_model = pd.DataFrame([
    {
        'componente': 'Estrategia de imputación',
        'seleccion': SELECTED_IMPUTATION,
        'fuente_decision': 'Entrenamiento'
    },
    {
        'componente': 'Modelo inferencial',
        'seleccion': BEST_MODEL_NAME,
        'fuente_decision': 'AIC/BIC y validación interna sin test'
    },
    {
        'componente': 'Especificación inferencial definitiva',
        'seleccion': MULTICOLLINEARITY_SPECIFICATION,
        'fuente_decision': 'BIC y VIF con entrenamiento'
    },
    {
        'componente': 'Configuración predictiva',
        'seleccion': SELECTED_PREDICTIVE_CONFIGURATION,
        'fuente_decision': 'Comparación simétrica OOF'
    },
    {
        'componente': 'Umbral operativo',
        'seleccion': f'{BEST_THRESHOLD:.6f}',
        'fuente_decision': 'Predicciones OOF'
    },
    {
        'componente': 'Probabilidades para interpretación',
        'seleccion': 'Modelo inferencial sin ponderación',
        'fuente_decision': 'Coherencia con prevalencia y calibración'
    }
])
decision_model.to_csv(
    TABLES / '26e_decision_modelo_inferencial_y_predictivo.csv',
    index=False
)

display(inference_prediction_comparison)
display(decision_model)


,modelo,conjunto,estimador_predictivo,n,threshold,TN,FP,FN,TP,accuracy,balanced_accuracy,precision,recall,F1,ROC_AUC,PR_AUC,Brier
0,M1_S1_S2,train,LogisticRegression sin penalizacion; class_wei...,99535,0.500000,59761,17460,5685,16629,0.767469,0.759561,0.487811,0.745227,0.589649,0.838540,0.652189,0.160810
1,M1_S1_S2,test,LogisticRegression sin penalizacion; class_wei...,42658,0.500000,25542,7553,2430,7133,0.765976,0.758837,0.485701,0.745896,0.588313,0.837115,0.646096,0.161620
2,M2_stepwise_AIC,train,LogisticRegression sin penalizacion; class_wei...,99535,0.500000,60653,16568,5525,16789,0.778038,0.768922,0.503313,0.752398,0.603151,0.851776,0.671064,0.154106
3,M2_stepwise_AIC,test,LogisticRegression sin penalizacion; class_wei...,42658,0.500000,25900,7195,2366,7197,0.775869,0.767592,0.500069,0.752588,0.600877,0.850580,0.664361,0.154983
4,M3_BIC_parsimonioso,train,LogisticRegression sin penalizacion; class_wei...,99535,0.500000,60534,16687,5495,16819,0.777144,0.768824,0.501970,0.753742,0.602616,0.851701,0.671195,0.154252
5,M3_BIC_parsimonioso,test,LogisticRegression sin penalizacion; class_wei...,42658,0.500000,25869,7226,2366,7197,0.775142,0.767123,0.498995,0.752588,0.600100,0.850491,0.664555,0.155148


,modelo,conjunto,TN,FP,FN,TP
0,M1_S1_S2,train,59761,17460,5685,16629
1,M1_S1_S2,test,25542,7553,2430,7133
2,M2_stepwise_AIC,train,60653,16568,5525,16789
3,M2_stepwise_AIC,test,25900,7195,2366,7197
4,M3_BIC_parsimonioso,train,60534,16687,5495,16819
5,M3_BIC_parsimonioso,test,25869,7226,2366,7197


,escenario,configuracion,tipo_umbral,seleccionada_OOF,n,threshold,TN,FP,FN,TP,accuracy,balanced_accuracy,precision,recall,F1,ROC_AUC,PR_AUC,Brier
0,inferencial_statsmodels_umbral_0_50,inferencial_sin_ponderacion,fijo_0_50,False,42658,0.500000,31303,1792,5039,4524,0.839866,0.709463,0.716276,0.473073,0.569809,0.852087,0.673900,0.116144
1,sin_ponderacion_umbral_0_50,sin_ponderacion,fijo_0_50,False,42658,0.500000,31306,1789,5042,4521,0.839866,0.709352,0.716482,0.472760,0.569647,0.852098,0.673892,0.116141
2,sin_ponderacion_umbral_OOF,sin_ponderacion,OOF_optimizado,False,42658,0.300000,28594,4501,3245,6318,0.818416,0.762334,0.583973,0.660671,0.619959,0.852098,0.673892,0.116141
3,balanceada_umbral_0_50,balanceada,fijo_0_50,False,42658,0.500000,26313,6782,2455,7108,0.783464,0.769178,0.511735,0.743281,0.606148,0.852474,0.672708,0.152491
4,balanceada_umbral_OOF,balanceada,OOF_optimizado,True,42658,0.600000,28554,4541,3211,6352,0.818276,0.763508,0.583127,0.664227,0.621040,0.852474,0.672708,0.152491


,componente,seleccion,fuente_decision
0,Estrategia de imputación,imputacion_regresion,Entrenamiento
1,Modelo inferencial,M3_BIC_parsimonioso,AIC/BIC y validación interna sin test
2,Especificación inferencial definitiva,sin_RainToday_bin,BIC y VIF con entrenamiento
3,Configuración predictiva,balanceada,Comparación simétrica OOF
4,Umbral operativo,0.600000,Predicciones OOF
5,Probabilidades para interpretación,Modelo inferencial sin ponderación,Coherencia con prevalencia y calibración


## 9. Análisis comparativo del impacto de la imputación

Los tres datasets parten del mismo universo de diez variables numéricas y `RainToday_bin`:

1. **Casos completos:** elimina registros con faltantes en cualquiera de las once variables.
2. **Imputación simple:** utiliza medianas para las variables numéricas y moda para `RainToday_bin`, calculadas con entrenamiento.
3. **Imputación por regresión:** utiliza las diez regresiones lineales explícitas y una regresión logística para `RainToday_bin`.

Las seis variables excluidas quedan fuera de los tres datasets. Después del tratamiento de faltantes, las tres estrategias ajustan la misma especificación final, emplean la misma escala de referencia y se evalúan en su test nativo y en una intersección común de identificadores.


In [31]:
IMPACT_NUMERIC_VARIABLES = COMPARISON_NUMERIC.copy()
IMPACT_ALL_VARIABLES = COMPARISON_ALL.copy()
N_IMPACT_NUMERIC = len(IMPACT_NUMERIC_VARIABLES)
N_IMPACT_ALL = len(IMPACT_ALL_VARIABLES)

# Las tres estrategias parten exactamente del mismo alcance de variables.
final_complete_train = train_raw.dropna(
    subset=IMPACT_ALL_VARIABLES
).copy()
final_complete_test = test_raw.dropna(
    subset=IMPACT_ALL_VARIABLES
).copy()
final_simple_train, final_simple_test = apply_simple_imputation(
    train_raw,
    test_raw,
    IMPACT_ALL_VARIABLES
)
final_regression_train = regression_train.copy()
final_regression_test = regression_test.copy()

final_strategy_data = {
    'casos_completos': (final_complete_train, final_complete_test),
    'imputacion_simple': (final_simple_train, final_simple_test),
    'imputacion_regresion': (
        final_regression_train,
        final_regression_test
    ),
}

expected_strategy_names = {
    'casos_completos',
    'imputacion_simple',
    'imputacion_regresion'
}
if set(final_strategy_data) != expected_strategy_names:
    raise AssertionError(
        'El análisis de impacto no contiene exactamente las tres estrategias.'
    )

expected_complete_train_ids = set(
    train_raw.dropna(subset=IMPACT_ALL_VARIABLES)['row_id']
)
expected_complete_test_ids = set(
    test_raw.dropna(subset=IMPACT_ALL_VARIABLES)['row_id']
)
if set(final_complete_train['row_id']) != expected_complete_train_ids:
    raise AssertionError(
        f'Casos completos de entrenamiento no utiliza las {N_IMPACT_NUMERIC} '
        'variables numéricas y RainToday_bin de forma íntegra.'
    )
if set(final_complete_test['row_id']) != expected_complete_test_ids:
    raise AssertionError(
        f'Casos completos de prueba no utiliza las {N_IMPACT_NUMERIC} variables '
        'numéricas y RainToday_bin de forma íntegra.'
    )
if set(final_simple_train['row_id']) != set(train_raw['row_id']):
    raise AssertionError(
        'La imputación simple modificó indebidamente el universo de entrenamiento.'
    )
if set(final_simple_test['row_id']) != set(test_raw['row_id']):
    raise AssertionError(
        'La imputación simple modificó indebidamente el universo de prueba.'
    )
if set(final_regression_train['row_id']) != set(train_raw['row_id']):
    raise AssertionError(
        'La imputación por regresión modificó indebidamente el universo de entrenamiento.'
    )
if set(final_regression_test['row_id']) != set(test_raw['row_id']):
    raise AssertionError(
        'La imputación por regresión modificó indebidamente el universo de prueba.'
    )

impact_scope_rows = []
strategy_method_description = {
    'casos_completos': (
        f'Eliminación si falta cualquiera de las {N_IMPACT_NUMERIC} variables '
        'numéricas o RainToday_bin'
    ),
    'imputacion_simple': (
        f'Mediana de entrenamiento para las {N_IMPACT_NUMERIC} variables '
        'numéricas y moda de entrenamiento para RainToday_bin'
    ),
    'imputacion_regresion': (
        f'Regresión lineal múltiple explícita para las {N_IMPACT_NUMERIC} '
        'variables numéricas y regresión logística para RainToday_bin'
    )
}
for strategy_name, (train_data_raw, test_data_raw) in final_strategy_data.items():
    missing_train = int(
        train_data_raw[IMPACT_ALL_VARIABLES].isna().sum().sum()
    )
    missing_test = int(
        test_data_raw[IMPACT_ALL_VARIABLES].isna().sum().sum()
    )
    if missing_train != 0 or missing_test != 0:
        raise AssertionError(
            f'La estrategia {strategy_name} conserva faltantes dentro del '
            'alcance simétrico del análisis de impacto.'
        )
    if train_data_raw['Date'].isna().any() or test_data_raw['Date'].isna().any():
        raise AssertionError(
            f'La estrategia {strategy_name} contiene fechas inválidas y no '
            'permite construir Season de forma consistente.'
        )
    impact_scope_rows.append({
        'estrategia': strategy_name,
        'metodo_tratamiento': strategy_method_description[strategy_name],
        'n_variables_numericas_tratadas': len(IMPACT_NUMERIC_VARIABLES),
        'variables_numericas_tratadas': ', '.join(IMPACT_NUMERIC_VARIABLES),
        'incluye_RainToday_bin': True,
        'n_variables_totales_alcance': len(IMPACT_ALL_VARIABLES),
        'faltantes_residuales_train_alcance': missing_train,
        'faltantes_residuales_test_alcance': missing_test,
        'n_train_despues_tratamiento': len(train_data_raw),
        'n_test_despues_tratamiento': len(test_data_raw),
        'mismos_predictores_modelo_final': ', '.join(FINAL_FEATURES),
        'alcance_simetrico_confirmado': True
    })

impact_symmetry_control = pd.DataFrame(impact_scope_rows)
if not (
    impact_symmetry_control['n_variables_numericas_tratadas'].eq(N_IMPACT_NUMERIC).all()
    and impact_symmetry_control['n_variables_totales_alcance'].eq(N_IMPACT_ALL).all()
    and impact_symmetry_control['alcance_simetrico_confirmado'].all()
):
    raise AssertionError(
        f'No se confirmó el alcance simétrico de {N_IMPACT_NUMERIC} variables '
        'numéricas más RainToday_bin en las tres estrategias.'
    )
impact_symmetry_control.to_csv(
    TABLES / '33a_control_simetria_estrategias_imputacion.csv',
    index=False
)
display(impact_symmetry_control)

# Escala común: parámetros del conjunto de entrenamiento de la estrategia principal,
# restringidos exclusivamente a los predictores continuos del modelo final.
final_base_features = []
for feature in FINAL_FEATURES:
    base_feature = feature[:-3] if feature.endswith('_sq') else feature
    if base_feature not in final_base_features:
        final_base_features.append(base_feature)
final_continuous_features = [
    feature for feature in final_base_features if feature not in BINARY_FEATURES
]
reference_train_for_scaling = add_engineered_features(final_strategy_data[SELECTED_IMPUTATION][0])
reference_scaler = StandardScaler().fit(reference_train_for_scaling[final_continuous_features])
common_test_ids = set.intersection(*[
    set(test_data_raw['row_id'])
    for _, test_data_raw in final_strategy_data.values()
])
if not common_test_ids:
    raise AssertionError(
        'La intersección de prueba común entre estrategias está vacía.'
    )

impact_metric_rows = []
impact_coefficient_rows = []
impact_size_rows = []
strategy_models = {}

for strategy_name, (train_data_raw, test_data_raw) in final_strategy_data.items():
    train_data = add_engineered_features(train_data_raw)
    test_data = add_engineered_features(test_data_raw)
    X_strategy_train = train_data[final_base_features].copy().reset_index(drop=True)
    X_strategy_test = test_data[final_base_features].copy().reset_index(drop=True)
    X_strategy_train[final_continuous_features] = reference_scaler.transform(train_data[final_continuous_features])
    X_strategy_test[final_continuous_features] = reference_scaler.transform(test_data[final_continuous_features])
    for variable in nonlinear_variables if KEEP_REFINEMENT else []:
        X_strategy_train[f'{variable}_sq'] = X_strategy_train[variable] ** 2
        X_strategy_test[f'{variable}_sq'] = X_strategy_test[variable] ** 2

    y_strategy_train = train_data['RainTomorrow_bin'].astype(int).reset_index(drop=True)
    y_strategy_test = test_data['RainTomorrow_bin'].astype(int).reset_index(drop=True)
    result = sm.GLM(
        y_strategy_train,
        sm.add_constant(
            X_strategy_train[FINAL_FEATURES],
            has_constant='add'
        ),
        family=sm.families.Binomial()
    ).fit(maxiter=200, disp=0)
    strategy_models[strategy_name] = result

    predictive_model = LogisticRegression(
        penalty=None,
        solver='lbfgs',
        max_iter=2_000,
        class_weight=SELECTED_CLASS_WEIGHT
    ).fit(
        X_strategy_train[FINAL_FEATURES],
        y_strategy_train
    )

    confidence = result.conf_int(alpha=ALPHA)
    for term in result.params.index:
        impact_coefficient_rows.append({
            'estrategia': strategy_name, 'termino': term,
            'beta': result.params[term],
            'IC95_beta_li': confidence.loc[term, 0],
            'IC95_beta_ls': confidence.loc[term, 1],
            'odds_ratio': np.exp(result.params[term]),
            'IC95_OR_li': np.exp(confidence.loc[term, 0]),
            'IC95_OR_ls': np.exp(confidence.loc[term, 1])
        })

    native_probability = (
        predictive_model.predict_proba(
            X_strategy_test[FINAL_FEATURES]
        )[:, 1]
    )
    impact_metric_rows.append({
        'estrategia': strategy_name,
        'tipo_test': 'nativo',
        'estimador_predictivo': (
            f'LogisticRegression; class_weight={SELECTED_CLASS_WEIGHT}'
        ),
        **calculate_metrics(y_strategy_test, native_probability, threshold=BEST_THRESHOLD)
    })

    common_mask = test_data['row_id'].isin(common_test_ids).to_numpy()
    common_probability = (
        predictive_model.predict_proba(
            X_strategy_test.loc[
                common_mask,
                FINAL_FEATURES
            ]
        )[:, 1]
    )
    impact_metric_rows.append({
        'estrategia': strategy_name,
        'tipo_test': 'comun',
        'estimador_predictivo': (
            f'LogisticRegression; class_weight={SELECTED_CLASS_WEIGHT}'
        ),
        **calculate_metrics(y_strategy_test.loc[common_mask], common_probability, threshold=BEST_THRESHOLD)
    })
    impact_size_rows.append({
        'estrategia': strategy_name,
        'n_train_modelo_final': len(train_data),
        'n_test_nativo': len(test_data),
        'n_test_comun': int(common_mask.sum()),
        'n_variables_numericas_tratadas': len(IMPACT_NUMERIC_VARIABLES),
        'incluye_RainToday_bin': True,
        'alcance_simetrico': True
    })

impact_metrics = pd.DataFrame(impact_metric_rows)
impact_coefficients = pd.DataFrame(impact_coefficient_rows)
impact_sizes = pd.DataFrame(impact_size_rows)

reference_coefficients = impact_coefficients.loc[
    impact_coefficients['estrategia'].eq(SELECTED_IMPUTATION), ['termino', 'beta']
].rename(columns={'beta': 'beta_referencia'})
impact_coefficients = impact_coefficients.merge(reference_coefficients, on='termino', how='left')
impact_coefficients['cambio_beta_absoluto'] = (impact_coefficients['beta'] - impact_coefficients['beta_referencia']).abs()
impact_coefficients['cambio_beta_relativo_porcentual'] = 100 * impact_coefficients['cambio_beta_absoluto'] / impact_coefficients['beta_referencia'].abs().replace(0, np.nan)
impact_coefficients['IC_beta_incluye_cero'] = (
    (impact_coefficients['IC95_beta_li'] <= 0) & (impact_coefficients['IC95_beta_ls'] >= 0)
)

impact_sizes.to_csv(TABLES / '33_tamano_muestral_modelo_final_por_imputacion.csv', index=False)
impact_coefficients.to_csv(TABLES / '34_coeficientes_IC_modelo_final_por_imputacion.csv', index=False)
impact_metrics.to_csv(TABLES / '35_desempeno_modelo_final_por_imputacion.csv', index=False)

display(impact_sizes)
display(
    impact_metrics.loc[
        impact_metrics['tipo_test'].eq('comun')
    ].reset_index(drop=True)
)

conclusion_rows = []
for term, group in impact_coefficients.groupby('termino'):
    signs = np.sign(group['beta'].to_numpy())
    conclusion_rows.append({
        'termino': term,
        'signo_consistente_tres_estrategias': bool(np.all(signs == signs[0])),
        'max_cambio_beta_relativo_porcentual': group['cambio_beta_relativo_porcentual'].max(),
        'cambia_conclusion_IC_incluye_cero': group['IC_beta_incluye_cero'].nunique() > 1
    })
impact_conclusions = pd.DataFrame(conclusion_rows)
impact_conclusions.to_csv(TABLES / '36_estabilidad_conclusiones_segun_imputacion.csv', index=False)
display(impact_conclusions)

performance_spread = (
    impact_metrics.loc[impact_metrics['tipo_test'].eq('comun')]
    .agg({
        'ROC_AUC': lambda values: values.max() - values.min(),
        'accuracy': lambda values: values.max() - values.min(),
        'F1': lambda values: values.max() - values.min()
    })
    .rename_axis('metrica')
    .reset_index(name='rango_entre_estrategias')
)
performance_spread.to_csv(TABLES / '37_variacion_desempeno_entre_imputaciones.csv', index=False)

max_auc_spread = performance_spread.set_index('metrica').loc['ROC_AUC', 'rango_entre_estrategias']
max_accuracy_spread = performance_spread.set_index('metrica').loc['accuracy', 'rango_entre_estrategias']
coefficient_conclusion_changes = int(impact_conclusions['cambia_conclusion_IC_incluye_cero'].sum())
if max_auc_spread < 0.01 and max_accuracy_spread < 0.01 and coefficient_conclusion_changes == 0:
    recommendation_text = (
        f'Las conclusiones son robustas frente a la estrategia de faltantes. Se recomienda mantener {SELECTED_IMPUTATION} '
        'porque fue seleccionada exclusivamente con entrenamiento y conserva la muestra, manteniendo signos, intervalos y desempeño comparables.'
    )
else:
    recommendation_text = (
        f'La estrategia de faltantes modifica resultados relevantes. Se recomienda mantener {SELECTED_IMPUTATION} como análisis principal, '
        'reportar obligatoriamente las tres sensibilidades y evitar interpretar como estable cualquier término cuyo intervalo cambie de conclusión.'
    )
imputation_recommendation = pd.DataFrame([{
    'estrategia_recomendada': SELECTED_IMPUTATION,
    'rango_ROC_AUC_test_comun': max_auc_spread,
    'rango_accuracy_test_comun': max_accuracy_spread,
    'n_terminos_con_cambio_conclusion_IC': coefficient_conclusion_changes,
    'recomendacion': recommendation_text
}])
imputation_recommendation.to_csv(TABLES / '37b_recomendacion_impacto_imputacion.csv', index=False)
display(imputation_recommendation)

sensitive_imputation_terms = impact_conclusions.loc[
    impact_conclusions['cambia_conclusion_IC_incluye_cero'],
    'termino'
].tolist()
impact_interpretation = pd.DataFrame([{
    'n_terminos_con_cambio_conclusion_IC': len(sensitive_imputation_terms),
    'terminos_sensibles': (
        ', '.join(sensitive_imputation_terms)
        if sensitive_imputation_terms
        else 'ninguno'
    ),
    'criterio_interpretacion': (
        'Los términos con cambios de conclusión entre estrategias deben '
        'interpretarse como sensibles al tratamiento de faltantes.'
    )
}])
impact_interpretation.to_csv(
    TABLES / '37c_interpretacion_sensibilidad_imputacion.csv',
    index=False
)
display(impact_interpretation)


,estrategia,metodo_tratamiento,n_variables_numericas_tratadas,variables_numericas_tratadas,incluye_RainToday_bin,n_variables_totales_alcance,faltantes_residuales_train_alcance,faltantes_residuales_test_alcance,n_train_despues_tratamiento,n_test_despues_tratamiento,mismos_predictores_modelo_final,alcance_simetrico_confirmado
0,casos_completos,Eliminación si falta cualquiera de las 10 vari...,10,"MinTemp, MaxTemp, Rainfall, WindGustSpeed, Hum...",True,11,0,0,83683,36019,"Humidity3pm, Rainfall_log1p, Pressure3pm, MaxT...",True
1,imputacion_simple,Mediana de entrenamiento para las 10 variables...,10,"MinTemp, MaxTemp, Rainfall, WindGustSpeed, Hum...",True,11,0,0,99535,42658,"Humidity3pm, Rainfall_log1p, Pressure3pm, MaxT...",True
2,imputacion_regresion,Regresión lineal múltiple explícita para las 1...,10,"MinTemp, MaxTemp, Rainfall, WindGustSpeed, Hum...",True,11,0,0,99535,42658,"Humidity3pm, Rainfall_log1p, Pressure3pm, MaxT...",True


,estrategia,n_train_modelo_final,n_test_nativo,n_test_comun,n_variables_numericas_tratadas,incluye_RainToday_bin,alcance_simetrico
0,casos_completos,83683,36019,36019,10,True,True
1,imputacion_simple,99535,42658,36019,10,True,True
2,imputacion_regresion,99535,42658,36019,10,True,True


,estrategia,tipo_test,estimador_predictivo,n,threshold,TN,FP,FN,TP,accuracy,balanced_accuracy,precision,recall,F1,ROC_AUC,PR_AUC,Brier
0,casos_completos,comun,LogisticRegression; class_weight=balanced,36019,0.600000,24411,3750,2582,5276,0.824204,0.769127,0.584534,0.671418,0.624970,0.861182,0.685884,0.147248
1,imputacion_simple,comun,LogisticRegression; class_weight=balanced,36019,0.600000,24383,3778,2568,5290,0.823815,0.769521,0.583370,0.673199,0.625074,0.860624,0.684516,0.149097
2,imputacion_regresion,comun,LogisticRegression; class_weight=balanced,36019,0.600000,24417,3744,2571,5287,0.824676,0.769934,0.585428,0.672818,0.626088,0.860479,0.684861,0.148139


,termino,signo_consistente_tres_estrategias,max_cambio_beta_relativo_porcentual,cambia_conclusion_IC_incluye_cero
0,Humidity3pm,True,4.446414,False
1,Humidity3pm_sq,True,8.649267,False
2,Humidity9am,True,20.636769,False
3,MaxTemp,True,87.875136,True
4,MaxTemp_sq,True,95.177300,True
5,Pressure3pm,True,3.083334,False
6,Rainfall_log1p,True,27.409537,False
7,Rainfall_log1p_sq,True,45.674680,False
8,Season_Autumn,True,5.293611,False
9,Season_Spring,True,8.646605,False


,estrategia_recomendada,rango_ROC_AUC_test_comun,rango_accuracy_test_comun,n_terminos_con_cambio_conclusion_IC,recomendacion
0,imputacion_regresion,0.000703,0.000861,2,La estrategia de faltantes modifica resultados...


,n_terminos_con_cambio_conclusion_IC,terminos_sensibles,criterio_interpretacion
0,2,"MaxTemp, MaxTemp_sq",Los términos con cambios de conclusión entre e...


## 10. Anexo computacional: análisis de sensibilidad

Se evalúan la influencia de outliers, el desbalance de clases y la heterogeneidad por localidad. Estos análisis verifican la robustez del modelo sin alterar la especificación seleccionada.


In [32]:
# Sensibilidad a winsorización 1–99 %, con límites calculados solo en entrenamiento.
winsor_train = selected_train.copy()
winsor_test = selected_test.copy()
winsor_limits = []
for variable in CONTINUOUS_FEATURES:
    lower, upper = winsor_train[variable].quantile([0.01, 0.99])
    winsor_train[variable] = winsor_train[variable].clip(lower, upper)
    winsor_test[variable] = winsor_test[variable].clip(lower, upper)
    winsor_limits.append({'variable': variable, 'q01_train': lower, 'q99_train': upper})

winsor_scaler = StandardScaler().fit(winsor_train[CONTINUOUS_FEATURES])
X_winsor_train = winsor_train[CANDIDATE_FEATURES].copy().reset_index(drop=True)
X_winsor_test = winsor_test[CANDIDATE_FEATURES].copy().reset_index(drop=True)
X_winsor_train[CONTINUOUS_FEATURES] = winsor_scaler.transform(winsor_train[CONTINUOUS_FEATURES])
X_winsor_test[CONTINUOUS_FEATURES] = winsor_scaler.transform(winsor_test[CONTINUOUS_FEATURES])
for variable in nonlinear_variables if KEEP_REFINEMENT else []:
    X_winsor_train[f'{variable}_sq'] = X_winsor_train[variable] ** 2
    X_winsor_test[f'{variable}_sq'] = X_winsor_test[variable] ** 2

winsor_model = LogisticRegression(
    penalty=None, solver='lbfgs', max_iter=2_000,
    class_weight=SELECTED_CLASS_WEIGHT
).fit(X_winsor_train[FINAL_FEATURES], y_train)
winsor_probability = winsor_model.predict_proba(X_winsor_test[FINAL_FEATURES])[:, 1]
outlier_sensitivity = pd.DataFrame([
    {'escenario': 'modelo_principal', **calculate_metrics(y_test, final_probability_test, threshold=BEST_THRESHOLD)},
    {'escenario': 'winsorizacion_1_99', **calculate_metrics(y_test, winsor_probability, threshold=BEST_THRESHOLD)}
])
outlier_sensitivity.to_csv(TABLES / '38_sensibilidad_winsorizacion.csv', index=False)
pd.DataFrame(winsor_limits).to_csv(TABLES / '38b_limites_winsorizacion_train.csv', index=False)
display(outlier_sensitivity)

# Comparación simétrica de ponderación y umbral en el test reservado.
imbalance_sensitivity = pd.DataFrame([
    {
        'escenario': 'sin_ponderacion_umbral_0_50',
        **calculate_metrics(
            y_test,
            final_probability_test_unweighted,
            threshold=0.50
        )
    },
    {
        'escenario': 'sin_ponderacion_umbral_OOF',
        **calculate_metrics(
            y_test,
            final_probability_test_unweighted,
            threshold=best_threshold_lookup['sin_ponderacion']
        )
    },
    {
        'escenario': 'balanceada_umbral_0_50',
        **calculate_metrics(
            y_test,
            final_probability_test_balanced,
            threshold=0.50
        )
    },
    {
        'escenario': 'balanceada_umbral_OOF',
        **calculate_metrics(
            y_test,
            final_probability_test_balanced,
            threshold=best_threshold_lookup['balanceada']
        )
    }
])
imbalance_sensitivity['seleccionada_OOF'] = (
    imbalance_sensitivity['escenario'].eq(
        f'{SELECTED_PREDICTIVE_CONFIGURATION}_umbral_OOF'
    )
)
imbalance_sensitivity.to_csv(
    TABLES / '39_sensibilidad_desbalance_umbral.csv',
    index=False
)
display(imbalance_sensitivity)


,escenario,n,threshold,TN,FP,FN,TP,accuracy,balanced_accuracy,precision,recall,F1,ROC_AUC,PR_AUC,Brier
0,modelo_principal,42658,0.600000,28554,4541,3211,6352,0.818276,0.763508,0.583127,0.664227,0.621040,0.852474,0.672708,0.152491
1,winsorizacion_1_99,42658,0.600000,28535,4560,3219,6344,0.817643,0.762802,0.581805,0.663390,0.619925,0.852565,0.670197,0.152583


,escenario,n,threshold,TN,FP,FN,TP,accuracy,balanced_accuracy,precision,recall,F1,ROC_AUC,PR_AUC,Brier,seleccionada_OOF
0,sin_ponderacion_umbral_0_50,42658,0.500000,31306,1789,5042,4521,0.839866,0.709352,0.716482,0.472760,0.569647,0.852098,0.673892,0.116141,False
1,sin_ponderacion_umbral_OOF,42658,0.300000,28594,4501,3245,6318,0.818416,0.762334,0.583973,0.660671,0.619959,0.852098,0.673892,0.116141,False
2,balanceada_umbral_0_50,42658,0.500000,26313,6782,2455,7108,0.783464,0.769178,0.511735,0.743281,0.606148,0.852474,0.672708,0.152491,False
3,balanceada_umbral_OOF,42658,0.600000,28554,4541,3211,6352,0.818276,0.763508,0.583127,0.664227,0.621040,0.852474,0.672708,0.152491,True


### 10.1 Sensibilidad temporal

La validación temporal se construye dentro del conjunto de entrenamiento y separa fechas completas. Se exige que la fecha máxima del ajuste sea anterior a la fecha mínima de validación.


In [33]:

temporal_source = train_raw.assign(
    Date_parsed=pd.to_datetime(train_raw['Date'])
).copy()
unique_temporal_dates = np.sort(
    temporal_source['Date_parsed'].dropna().unique()
)
if len(unique_temporal_dates) < 10:
    raise ValueError('No existen fechas suficientes para la sensibilidad temporal.')

temporal_date_index = int(np.floor(0.80 * len(unique_temporal_dates)))
temporal_date_index = min(
    max(temporal_date_index, 1),
    len(unique_temporal_dates) - 1
)
temporal_validation_start = pd.Timestamp(
    unique_temporal_dates[temporal_date_index]
)
temporal_fit_raw = temporal_source.loc[
    temporal_source['Date_parsed'] < temporal_validation_start
].drop(columns='Date_parsed').copy()
temporal_validation_raw = temporal_source.loc[
    temporal_source['Date_parsed'] >= temporal_validation_start
].drop(columns='Date_parsed').copy()

if temporal_fit_raw.empty or temporal_validation_raw.empty:
    raise ValueError('El corte temporal produjo un conjunto vacío.')
if not (
    pd.to_datetime(temporal_fit_raw['Date']).max()
    < pd.to_datetime(temporal_validation_raw['Date']).min()
):
    raise AssertionError(
        'La validación temporal comparte fechas entre ajuste y validación.'
    )

if SELECTED_IMPUTATION == 'casos_completos':
    temporal_fit = temporal_fit_raw.dropna(
        subset=COMPARISON_ALL
    ).copy()
    temporal_validation = temporal_validation_raw.dropna(
        subset=COMPARISON_ALL
    ).copy()
elif SELECTED_IMPUTATION == 'imputacion_simple':
    temporal_fit, temporal_validation = apply_simple_imputation(
        temporal_fit_raw,
        temporal_validation_raw,
        COMPARISON_ALL
    )
elif SELECTED_IMPUTATION == 'imputacion_regresion':
    temporal_imputation = chained_regression_imputation(
        temporal_fit_raw,
        temporal_validation_raw,
        IMPUTATION_SPECS,
        seed=SEED + 70_000,
        cycles=N_IMPUTATION_CYCLES,
        diagnostics=False
    )
    temporal_fit = temporal_imputation['train']
    temporal_validation = temporal_imputation['other']
else:
    raise ValueError(
        f'Estrategia desconocida: {SELECTED_IMPUTATION}'
    )

(
    temporal_fit,
    temporal_validation,
    X_temporal_fit,
    X_temporal_validation,
    y_temporal_fit,
    y_temporal_validation,
    temporal_scaler
) = standardize_fold(
    temporal_fit,
    temporal_validation
)

for variable in nonlinear_variables if KEEP_REFINEMENT else []:
    X_temporal_fit[f'{variable}_sq'] = (
        X_temporal_fit[variable] ** 2
    )
    X_temporal_validation[f'{variable}_sq'] = (
        X_temporal_validation[variable] ** 2
    )

temporal_model = LogisticRegression(
    penalty=None,
    solver='lbfgs',
    max_iter=2_000,
    class_weight=SELECTED_CLASS_WEIGHT
)
with warnings.catch_warnings(record=True) as caught_temporal:
    warnings.simplefilter('always')
    temporal_model.fit(
        X_temporal_fit[FINAL_FEATURES],
        y_temporal_fit
    )
temporal_convergence_warnings = [
    item
    for item in caught_temporal
    if issubclass(item.category, ConvergenceWarning)
]
if temporal_convergence_warnings:
    raise RuntimeError(
        'La sensibilidad temporal no convergió: '
        + ' | '.join(
            str(item.message)
            for item in temporal_convergence_warnings
        )
    )

temporal_probability = temporal_model.predict_proba(
    X_temporal_validation[FINAL_FEATURES]
)[:, 1]
temporal_sensitivity = pd.DataFrame([{
    'escenario': 'validacion_temporal_20pct_fechas_mas_recientes_train',
    'fecha_corte_validacion': temporal_validation_start,
    'fecha_inicio_fit': pd.to_datetime(
        temporal_fit['Date']
    ).min(),
    'fecha_fin_fit': pd.to_datetime(
        temporal_fit['Date']
    ).max(),
    'fecha_inicio_validacion': pd.to_datetime(
        temporal_validation['Date']
    ).min(),
    'fecha_fin_validacion': pd.to_datetime(
        temporal_validation['Date']
    ).max(),
    'n_fit': len(temporal_fit),
    'n_validacion': len(temporal_validation),
    'configuracion_predictiva': SELECTED_PREDICTIVE_CONFIGURATION,
    'umbral': BEST_THRESHOLD,
    **calculate_metrics(
        y_temporal_validation,
        temporal_probability,
        threshold=BEST_THRESHOLD
    )
}])
temporal_sensitivity.to_csv(
    TABLES / '40b_sensibilidad_validacion_temporal.csv',
    index=False
)
display(temporal_sensitivity)


,escenario,fecha_corte_validacion,fecha_inicio_fit,fecha_fin_fit,fecha_inicio_validacion,fecha_fin_validacion,n_fit,n_validacion,configuracion_predictiva,umbral,n,threshold,TN,FP,FN,TP,accuracy,balanced_accuracy,precision,recall,F1,ROC_AUC,PR_AUC,Brier
0,validacion_temporal_20pct_fechas_mas_recientes...,2015-08-16,2007-11-01,2015-08-15,2015-08-16,2017-06-25,76706,22829,balanceada,0.600000,22829,0.600000,15544,2242,1824,3219,0.821893,0.756128,0.589452,0.638311,0.612909,0.845340,0.659482,0.149840


In [34]:
# Sensibilidad con Location: misma especificación final y misma regularización.
location_encoder = OneHotEncoder(handle_unknown='ignore', drop='first', sparse_output=True)
location_train_encoded = location_encoder.fit_transform(selected_train[['Location']])
location_test_encoded = location_encoder.transform(selected_test[['Location']])

from scipy import sparse
X_location_train = sparse.hstack([
    sparse.csr_matrix(X_final_train.to_numpy(dtype=float)),
    location_train_encoded
], format='csr')
X_location_test = sparse.hstack([
    sparse.csr_matrix(X_final_test.to_numpy(dtype=float)),
    location_test_encoded
], format='csr')

base_sklearn = LogisticRegression(
    penalty=None, solver='lbfgs', max_iter=2_000,
    class_weight=SELECTED_CLASS_WEIGHT
).fit(X_final_train, y_train)
base_sklearn_probability = base_sklearn.predict_proba(X_final_test)[:, 1]
location_model = LogisticRegression(
    penalty=None, solver='lbfgs', max_iter=2_000,
    class_weight=SELECTED_CLASS_WEIGHT
).fit(X_location_train, y_train)
location_probability = location_model.predict_proba(X_location_test)[:, 1]

location_comparison = pd.DataFrame([
    {'escenario': 'sin_Location', **calculate_metrics(y_test, base_sklearn_probability, threshold=BEST_THRESHOLD)},
    {'escenario': 'con_Location', **calculate_metrics(y_test, location_probability, threshold=BEST_THRESHOLD)}
])
location_comparison.to_csv(TABLES / '40_sensibilidad_incorporacion_Location.csv', index=False)
display(location_comparison)

location_rows = []
test_location_frame = selected_test[['Location']].copy().reset_index(drop=True)
test_location_frame['y_true'] = y_test.to_numpy()
test_location_frame['probability'] = final_probability_test
for location, group in test_location_frame.groupby('Location'):
    row = {'Location': location, 'n': len(group), 'prevalencia': group['y_true'].mean()}
    prediction = (group['probability'] >= BEST_THRESHOLD).astype(int)
    row['accuracy'] = accuracy_score(group['y_true'], prediction)
    row['recall'] = recall_score(group['y_true'], prediction, zero_division=0)
    row['precision'] = precision_score(group['y_true'], prediction, zero_division=0)
    row['F1'] = f1_score(group['y_true'], prediction, zero_division=0)
    row['ROC_AUC'] = roc_auc_score(group['y_true'], group['probability']) if group['y_true'].nunique() == 2 else np.nan
    location_rows.append(row)
performance_by_location = pd.DataFrame(location_rows).sort_values('n', ascending=False)
performance_by_location.to_csv(TABLES / '41_desempeno_modelo_final_por_localidad.csv', index=False)
display(performance_by_location)

location_delta = (
    location_comparison
    .set_index('escenario')
    .loc['con_Location', ['ROC_AUC', 'accuracy', 'recall', 'F1']]
    - location_comparison
    .set_index('escenario')
    .loc['sin_Location', ['ROC_AUC', 'accuracy', 'recall', 'F1']]
)
location_gain_material = bool(
    abs(float(location_delta['ROC_AUC'])) >= 0.01
    or abs(float(location_delta['F1'])) >= 0.01
)
if location_gain_material:
    location_decision = (
        'Mantener el modelo principal parsimonioso y reportar una alternativa '
        'con Location por su ganancia predictiva material'
    )
    location_foundation = (
        'Location mejora materialmente al menos ROC-AUC o F1. No se incorpora '
        'retrospectivamente al modelo principal porque fue evaluada como '
        'sensibilidad después de fijar la especificación; debe reportarse como '
        'modelo alternativo de menor transferibilidad geográfica.'
    )
else:
    location_decision = 'Mantener Location fuera del modelo principal'
    location_foundation = (
        'La ganancia predictiva no alcanza 0,01 en ROC-AUC ni F1; se priorizan '
        'parsimonia, interpretación y transferibilidad entre localidades.'
    )

location_interpretation = pd.DataFrame([{
    'decision_modelo_principal': location_decision,
    'ganancia_material_umbral_0_01': location_gain_material,
    'delta_ROC_AUC': float(location_delta['ROC_AUC']),
    'delta_accuracy': float(location_delta['accuracy']),
    'delta_recall': float(location_delta['recall']),
    'delta_F1': float(location_delta['F1']),
    'fundamento': location_foundation
}])
location_interpretation.to_csv(
    TABLES / '41b_interpretacion_sensibilidad_Location.csv',
    index=False
)
display(location_interpretation)


,escenario,n,threshold,TN,FP,FN,TP,accuracy,balanced_accuracy,precision,recall,F1,ROC_AUC,PR_AUC,Brier
0,sin_Location,42658,0.600000,28554,4541,3211,6352,0.818276,0.763508,0.583127,0.664227,0.621040,0.852474,0.672708,0.152491
1,con_Location,42658,0.600000,28661,4434,3182,6381,0.821464,0.766641,0.590014,0.667259,0.626264,0.859773,0.686385,0.148518


,Location,n,prevalencia,accuracy,recall,precision,F1,ROC_AUC
37,Sydney,1011,0.241345,0.806133,0.581967,0.601695,0.591667,0.818758
9,Canberra,1000,0.176000,0.843000,0.522727,0.557576,0.539589,0.839413
15,Hobart,987,0.231003,0.771023,0.614035,0.503597,0.553360,0.813050
7,Brisbane,970,0.246392,0.837113,0.447699,0.804511,0.575269,0.837352
32,PerthAirport,959,0.176225,0.896767,0.603550,0.761194,0.673267,0.901828
13,Darwin,949,0.270811,0.830348,0.856031,0.639535,0.732113,0.914571
35,Sale,939,0.216187,0.814696,0.758621,0.551971,0.639004,0.865395
20,Mildura,933,0.110397,0.903537,0.533981,0.567010,0.550000,0.891321
0,Adelaide,932,0.204936,0.873391,0.549738,0.766423,0.640244,0.888583
39,Townsville,932,0.165236,0.810086,0.805195,0.457565,0.583529,0.885546


,decision_modelo_principal,ganancia_material_umbral_0_01,delta_ROC_AUC,delta_accuracy,delta_recall,delta_F1,fundamento
0,Mantener Location fuera del modelo principal,False,0.007299,0.003188,0.003033,0.005223,"La ganancia predictiva no alcanza 0,01 en ROC-..."


## 11. Síntesis computacional y verificación final

La síntesis se genera a partir de los resultados de la ejecución y separa estrategia de imputación, modelo inferencial, configuración predictiva, calibración, estabilidad bootstrap y análisis de sensibilidad.


In [35]:
selected_test_result = final_test_metrics.iloc[0].to_dict()
unstable_terms = bootstrap_summary.loc[
    bootstrap_summary['parametro_inestable'],
    'termino'
].tolist()
cluster_changes = int(
    cluster_robust_sensitivity[
        'cambia_conclusion_IC_incluye_cero'
    ].sum()
)
imputation_conclusion_changes = int(
    impact_conclusions[
        'cambia_conclusion_IC_incluye_cero'
    ].sum()
)

summary_rows = [
    {
        'dimension': 'Cierre de retroalimentación S2',
        'resultado': (
            f'Observaciones verificadas: {int(correction_control["cumple"].sum())}/'
            f'{len(correction_control)}. Todas las evidencias se generan desde el '
            'notebook maestro de S3.'
        )
    },
    {
        'dimension': 'Profundización bootstrap de Rainfall',
        'resultado': (
            f'Asimetría={rainfall_skewness:.4f}; n_boot={N_BOOT_RAINFALL:,}; '
            f'IC BCa 95 %=[{rainfall_bca_li:.4f}, {rainfall_bca_ls:.4f}].'
        )
    },
    {
        'dimension': 'Sensibilidad Monte Carlo heredada de S2',
        'resultado': (
            f'{N_MC_S2_CLOSURE:,} iteraciones; nueve escenarios; '
            f'VPP base con Humidity3pm ≥ 60 %='
            f'{base_mc_row["vpp_umbral_60"]:.4f}; '
            f'alertas esperadas por 1.000='
            f'{base_mc_row["alertas_esperadas_por_1000"]:.1f}.'
        )
    },
    {
        'dimension': 'Multicolinealidad recalculada',
        'resultado': (
            'Pressure9am–Pressure3pm: '
            f'r={recalculated_pair_correlation("Pressure9am", "Pressure3pm"):.4f}; '
            'Temp9am–Temp3pm: '
            f'r={recalculated_pair_correlation("Temp9am", "Temp3pm"):.4f}.'
        )
    },
    {
        'dimension': 'Imputación',
        'resultado': (
            f'Estrategia seleccionada con entrenamiento: {SELECTED_IMPUTATION}. '
            f'Modo de regresión: {IMPUTATION_REGRESSION_MODE}.'
        )
    },
    {
        'dimension': 'Modelo inferencial',
        'resultado': (
            f'Procedimiento seleccionado: {BEST_MODEL_NAME}. '
            f'Especificación definitiva: {MULTICOLLINEARITY_SPECIFICATION}. '
            f'VIF máximo: {FINAL_MAX_VIF:.4f}.'
        )
    },
    {
        'dimension': 'Suficiencia muestral y separación',
        'resultado': (
            f'Eventos por parámetro: {epv_events:.2f}. '
            f'Señal numérica de separación: {separation_signal}.'
        )
    },
    {
        'dimension': 'Configuración predictiva',
        'resultado': (
            f'Configuración seleccionada por OOF: '
            f'{SELECTED_PREDICTIVE_CONFIGURATION}; '
            f'umbral: {BEST_THRESHOLD:.4f}.'
        )
    },
    {
        'dimension': 'Evaluación final en test',
        'resultado': (
            f'Accuracy={selected_test_result["accuracy"]:.4f}; '
            f'precision={selected_test_result["precision"]:.4f}; '
            f'recall={selected_test_result["recall"]:.4f}; '
            f'F1={selected_test_result["F1"]:.4f}; '
            f'ROC-AUC={selected_test_result["ROC_AUC"]:.4f}; '
            f'PR-AUC={selected_test_result["PR_AUC"]:.4f}; '
            f'Brier={selected_test_result["Brier"]:.4f}.'
        )
    },
    {
        'dimension': 'Bootstrap',
        'resultado': (
            f'Réplicas solicitadas: {N_BOOT:,}; '
            f'tasa de éxito: {bootstrap_success_rate:.2%}; '
            f'parámetros inestables: {len(unstable_terms)}.'
        )
    },
    {
        'dimension': 'Dependencia por Location',
        'resultado': (
            f'Términos cuya conclusión cambia con errores robustos por cluster: '
            f'{cluster_changes}.'
        )
    },
    {
        'dimension': 'Impacto de la imputación',
        'resultado': (
            f'Términos cuya conclusión cambia entre estrategias: '
            f'{imputation_conclusion_changes}.'
        )
    },
    {
        'dimension': 'Alcance',
        'resultado': (
            'Los resultados son predictivos y asociativos. La estructura '
            'temporal y espacial se evalúa mediante diagnósticos y sensibilidades; '
            'no se realizan afirmaciones causales.'
        )
    }
]
technical_summary = pd.DataFrame(summary_rows)
technical_summary.to_csv(
    TABLES / '42_sintesis_tecnica_resultados.csv',
    index=False
)
display(technical_summary)


,dimension,resultado
0,Cierre de retroalimentación S2,Observaciones verificadas: 5/5. Todas las evid...
1,Profundización bootstrap de Rainfall,"Asimetría=9.8362; n_boot=10,000; IC BCa 95 %=[..."
2,Sensibilidad Monte Carlo heredada de S2,"100,000 iteraciones; nueve escenarios; VPP bas..."
3,Multicolinealidad recalculada,Pressure9am–Pressure3pm: r=0.9613; Temp9am–Tem...
4,Imputación,Estrategia seleccionada con entrenamiento: imp...
5,Modelo inferencial,Procedimiento seleccionado: M3_BIC_parsimonios...
6,Suficiencia muestral y separación,Eventos por parámetro: 1593.86. Señal numérica...
7,Configuración predictiva,Configuración seleccionada por OOF: balanceada...
8,Evaluación final en test,Accuracy=0.8183; precision=0.5831; recall=0.66...
9,Bootstrap,"Réplicas solicitadas: 10,000; tasa de éxito: 1..."


In [36]:
humidity_feedback_evidence = candidate_evidence.loc[
    candidate_evidence['variable_origen'].eq('Humidity3pm')
]
raintoday_feedback_evidence = candidate_evidence.loc[
    candidate_evidence['variable_origen'].eq('RainToday_bin')
]
permutation_traceability_present = traceability['tipo_insumo_s3'].astype(str).str.contains(
    'Prueba de hipótesis validada',
    regex=False
).any()
chi_square_traceability_present = traceability['tipo_insumo_s3'].astype(str).str.contains(
    'Prueba complementaria documentada',
    regex=False
).any()
required_predictive_metrics = {
    'balanced_accuracy', 'precision', 'recall', 'F1',
    'ROC_AUC', 'PR_AUC', 'Brier'
}
predictive_metrics_present = required_predictive_metrics.issubset(
    set(predictive_configuration_oof.columns)
)
feedback_core_files = [
    TABLES / '02a_bootstrap_Rainfall_asimetria.csv',
    FIGURES / 'fig_00a_bootstrap_media_Rainfall.png',
    TABLES / '02b0_parametros_base_montecarlo_S1.csv',
    TABLES / '02b_sensibilidad_montecarlo_regenerada.csv',
    TABLES / '02c_consistencia_sensibilidad_montecarlo_S2_S3.csv',
    TABLES / '02d_convergencia_montecarlo_regenerada.csv',
    TABLES / '02e_diagnostico_outliers_IQR_regenerado.csv',
    TABLES / '02f_consistencia_diagnostico_IQR_S2_S3.csv',
    TABLES / '02g_recalculo_multicolinealidad_base_completa.csv',
    TABLES / '02h_control_cierre_retroalimentacion_S2.csv',
    TABLES / '13c_trazabilidad_variables_candidatas_S1_S2.csv',
    TABLES / '13e_delimitacion_predictoras_modelos_logisticos.csv',
    TABLES / '23_sensibilidad_umbral_out_of_fold.csv',
    TABLES / '23b_seleccion_configuracion_predictiva_OOF.csv',
    TABLES / '27c_VIF_especificacion_definitiva.csv',
    TABLES / '27f_sensibilidad_errores_robustos_cluster_Location.csv',
    TABLES / '40b_sensibilidad_validacion_temporal.csv',
    TABLES / '41b_interpretacion_sensibilidad_Location.csv'
]
feedback_core_files_complete = all(
    path.exists() and path.stat().st_size > 0
    for path in feedback_core_files
)

feedback_response_matrix = pd.DataFrame([
    {
        'criterio_o_observacion_S2': 'Bootstrap: incorporar un parámetro con asimetría marcada',
        'respuesta_implementada_en_S3': (
            'Se analiza la media de Rainfall mediante 10.000 remuestras y se comparan '
            'los intervalos clásico, percentil y BCa.'
        ),
        'evidencia_computacional': '02a_bootstrap_Rainfall_asimetria.csv; fig_00a_bootstrap_media_Rainfall.png',
        'verificacion': (
            f'n_boot={N_BOOT_RAINFALL}; asimetría={rainfall_skewness:.4f}; '
            f'IC_BCa=[{rainfall_bca_li:.4f}, {rainfall_bca_ls:.4f}]'
        ),
        'cumple': bool(
            N_BOOT_RAINFALL >= 10_000
            and abs(rainfall_skewness) > 1
            and (TABLES / '02a_bootstrap_Rainfall_asimetria.csv').exists()
            and (FIGURES / 'fig_00a_bootstrap_media_Rainfall.png').exists()
            and np.isfinite([
                rainfall_classic_li, rainfall_classic_ls,
                rainfall_percentile_li, rainfall_percentile_ls,
                rainfall_bca_li, rainfall_bca_ls
            ]).all()
        )
    },
    {
        'criterio_o_observacion_S2': 'Permutación: conservar la relación validada y la distinción respecto de chi-cuadrado',
        'respuesta_implementada_en_S3': (
            'La prueba de permutación se incorpora como antecedente inferencial validado '
            'para Humidity3pm y el chi-cuadrado se conserva como evidencia categórica '
            'complementaria para RainToday_bin.'
        ),
        'evidencia_computacional': '01_trazabilidad_S1_S2_hacia_S3.csv; 13c_trazabilidad_variables_candidatas_S1_S2.csv',
        'verificacion': 'Antecedentes de S2 cargados y vinculados con decisiones de modelamiento.',
        'cumple': bool(
            permutation_traceability_present
            and chi_square_traceability_present
            and not humidity_feedback_evidence.empty
            and not raintoday_feedback_evidence.empty
            and humidity_feedback_evidence['validada_estable_en_S2'].all()
            and raintoday_feedback_evidence['validada_estable_en_S2'].all()
        )
    },
    {
        'criterio_o_observacion_S2': 'Correlaciones: recalcular los pares 0,96 y 0,86 sobre la base completa',
        'respuesta_implementada_en_S3': (
            'Se recalculan los pares Pressure9am–Pressure3pm y Temp9am–Temp3pm, '
            'se registra el número de casos válidos y se actualiza la delimitación '
            'de predictoras antes de la selección automática.'
        ),
        'evidencia_computacional': '02g_recalculo_multicolinealidad_base_completa.csv; 13e_delimitacion_predictoras_modelos_logisticos.csv',
        'verificacion': (
            f'Pressure={recalculated_pair_correlation("Pressure9am", "Pressure3pm"):.4f}; '
            f'Temperatura={recalculated_pair_correlation("Temp9am", "Temp3pm"):.4f}'
        ),
        'cumple': bool(
            multicollinearity_recalculated['n_pares_validos_base_completa'].gt(0).all()
            and multicollinearity_recalculated.loc[
                multicollinearity_recalculated['correlacion_referencia_S2'].notna(),
                'diferencia_absoluta_vs_referencia_S2'
            ].le(0.01).all()
        )
    },
    {
        'criterio_o_observacion_S2': 'Monte Carlo: incorporar al notebook maestro la sensibilidad de nueve escenarios',
        'respuesta_implementada_en_S3': (
            'Se regenera el escenario base y ocho perturbaciones relativas con '
            '100.000 iteraciones, semilla fija, números aleatorios comunes y control '
            'de convergencia.'
        ),
        'evidencia_computacional': '02b_sensibilidad_montecarlo_regenerada.csv; 02d_convergencia_montecarlo_regenerada.csv',
        'verificacion': (
            f'n_iteraciones={N_MC_S2_CLOSURE}; '
            f'n_escenarios={len(montecarlo_sensitivity_regenerated)}'
        ),
        'cumple': bool(
            N_MC_S2_CLOSURE >= 10_000
            and len(montecarlo_sensitivity_regenerated) == 9
            and mc_parameter_traceability['coincide_referencia_S2'].all()
            and np.isclose(
                mc_parameters_base['proporcion_yes'],
                0.22418121848473554,
                rtol=0,
                atol=1e-12
            )
            and mc_consistency['consistencia_numerica'].all()
        )
    },
    {
        'criterio_o_observacion_S2': 'Monte Carlo: complementar el delta individual con un estimador agregado',
        'respuesta_implementada_en_S3': (
            'Se incorporan tasas operacionales, resultados esperados por cada 1.000 '
            'observaciones y distribuciones empíricas de VPP y sensibilidad en '
            'carteras de tamaño 1.000.'
        ),
        'evidencia_computacional': '02b_sensibilidad_montecarlo_regenerada.csv',
        'verificacion': (
            f'n_carteras={N_MC_PORTFOLIOS}; tamaño_cartera={MC_PORTFOLIO_SIZE}; '
            f'VPP_base={base_mc_row["vpp_umbral_60"]:.4f}'
        ),
        'cumple': bool(
            N_MC_PORTFOLIOS >= 10_000
            and montecarlo_sensitivity_regenerated[[
                'alertas_esperadas_por_1000',
                'verdaderos_positivos_esperados_por_1000',
                'falsos_positivos_esperados_por_1000',
                'eventos_no_detectados_esperados_por_1000',
                'vpp_cartera_media',
                'vpp_cartera_ic_empirico_95_li',
                'vpp_cartera_ic_empirico_95_ls',
                'sensibilidad_cartera_media',
                'sensibilidad_cartera_ic_empirico_95_li',
                'sensibilidad_cartera_ic_empirico_95_ls'
            ]].notna().all().all()
        )
    },
    {
        'criterio_o_observacion_S2': 'Robustez: regenerar el diagnóstico IQR citado en el informe',
        'respuesta_implementada_en_S3': (
            'El diagnóstico IQR se reproduce desde la base original por grupo de '
            'RainTomorrow y se contrasta con la evidencia heredada de S2.'
        ),
        'evidencia_computacional': '02e_diagnostico_outliers_IQR_regenerado.csv; 02f_consistencia_diagnostico_IQR_S2_S3.csv',
        'verificacion': (
            f'eliminados_total={int(iqr_diagnostic_regenerated["datos_eliminados"].sum())}'
        ),
        'cumple': bool(
            iqr_comparison['reproduccion_exacta_conteos'].all()
            and iqr_comparison['reproduccion_porcentaje_tolerancia'].all()
        )
    },
    {
        'criterio_o_observacion_S2': 'Modelamiento: priorizar Humidity3pm con fundamento en S1 y S2',
        'respuesta_implementada_en_S3': (
            'Humidity3pm se incluye en el modelo basado en S1/S2, en el universo '
            'de selección y en la especificación definitiva, con evaluación de '
            'no linealidad.'
        ),
        'evidencia_computacional': '13c_trazabilidad_variables_candidatas_S1_S2.csv; 16_variables_tres_modelos.csv; 27d_decision_especificacion_definitiva.csv',
        'verificacion': 'Predictor incorporado desde la formulación inicial hasta el modelo definitivo.',
        'cumple': bool(
            not humidity_feedback_evidence.empty
            and humidity_feedback_evidence['validada_estable_en_S2'].all()
            and np.isclose(
                humidity_feedback_evidence[
                    'correlacion_S1_con_RainTomorrow'
                ].iloc[0],
                0.4462,
                atol=5e-4
            )
            and 'Humidity3pm' in M1_FEATURES
            and 'Humidity3pm' in CANDIDATE_FEATURES
            and 'Humidity3pm' in FINAL_FEATURES
        )
    },
    {
        'criterio_o_observacion_S2': 'Modelamiento: incorporar RainToday_bin por su asociación categórica',
        'respuesta_implementada_en_S3': (
            'RainToday_bin se incorpora al modelo S1/S2 y al universo candidato; '
            'su permanencia final se decide mediante BIC y VIF, sin imponerla '
            'contra la evidencia multivariable.'
        ),
        'evidencia_computacional': '13c_trazabilidad_variables_candidatas_S1_S2.csv; 16_variables_tres_modelos.csv; 27d_decision_especificacion_definitiva.csv',
        'verificacion': (
            'Variable evaluada explícitamente; decisión definitiva documentada '
            f'en la especificación {MULTICOLLINEARITY_SPECIFICATION}.'
        ),
        'cumple': bool(
            not raintoday_feedback_evidence.empty
            and raintoday_feedback_evidence['validada_estable_en_S2'].all()
            and np.isclose(
                raintoday_feedback_evidence[
                    'correlacion_S1_con_RainTomorrow'
                ].iloc[0],
                0.3131,
                atol=5e-4
            )
            and 'RainToday_bin' in M1_FEATURES
            and 'RainToday_bin' in CANDIDATE_FEATURES
            and MULTICOLLINEARITY_SPECIFICATION in set(
                multicollinearity_comparison['especificacion']
            )
        )
    },
    {
        'criterio_o_observacion_S2': 'Modelamiento: controlar redundancia mediante selección y VIF',
        'respuesta_implementada_en_S3': (
            'Los pares altamente correlacionados se excluyen del universo automático '
            'y la especificación definitiva se confirma mediante VIF calculado '
            'exclusivamente con entrenamiento.'
        ),
        'evidencia_computacional': '13e_delimitacion_predictoras_modelos_logisticos.csv; 27b_resolucion_multicolinealidad_modelo_final.csv; 27c_VIF_especificacion_definitiva.csv',
        'verificacion': f'VIF_máximo_final={FINAL_MAX_VIF:.4f}',
        'cumple': bool(
            FINAL_MAX_VIF <= 10
            and multicollinearity_recalculated[
                'n_pares_validos_base_completa'
            ].gt(0).all()
            and {'Pressure9am', 'Temp9am', 'Temp3pm'}.issubset(
                set(
                    predictor_scope.loc[
                        predictor_scope[
                            'decision_universo_logistico'
                        ].str.startswith('Excluir'),
                        'variable_S1'
                    ]
                )
            )
        )
    },
    {
        'criterio_o_observacion_S2': 'Desbalance: comparar ponderación de clase y no depender de accuracy',
        'respuesta_implementada_en_S3': (
            'Se comparan configuraciones ponderadas y no ponderadas con umbrales '
            'fijos y optimizados, utilizando balanced accuracy, precision, recall, '
            'F1, ROC-AUC, PR-AUC y Brier.'
        ),
        'evidencia_computacional': '20b_comparacion_inferencia_y_prediccion_balanceada.csv; 23b_seleccion_configuracion_predictiva_OOF.csv; 26d_calibracion_configuraciones_predictivas.csv',
        'verificacion': (
            f'configuraciones_OOF={len(predictive_configuration_oof)}; '
            f'seleccionadas={int(predictive_configuration_oof["seleccionada"].sum())}'
        ),
        'cumple': bool(
            len(predictive_configuration_oof) == 4
            and predictive_configuration_oof['seleccionada'].sum() == 1
            and predictive_metrics_present
            and {'sin_ponderacion', 'balanceada'}.issubset(
                set(predictive_configuration_oof['configuracion'])
            )
        )
    },
    {
        'criterio_o_observacion_S2': 'Umbral: usar la sensibilidad previa como puente hacia el punto de operación',
        'respuesta_implementada_en_S3': (
            'El umbral probabilístico se selecciona mediante predicciones OOF y se '
            'distingue expresamente del umbral meteorológico Humidity3pm ≥ 60 % '
            'utilizado en la simulación de S2.'
        ),
        'evidencia_computacional': '02b_sensibilidad_montecarlo_regenerada.csv; 23_sensibilidad_umbral_out_of_fold.csv; 23b_seleccion_configuracion_predictiva_OOF.csv',
        'verificacion': f'umbral_probabilístico_seleccionado={BEST_THRESHOLD:.4f}',
        'cumple': bool(
            0 < BEST_THRESHOLD < 1
            and not threshold_table.empty
            and predictive_configuration_oof['seleccionada'].sum() == 1
            and predictive_configuration_oof.loc[
                predictive_configuration_oof['seleccionada'],
                'tipo_umbral'
            ].eq('OOF_optimizado').all()
        )
    },
    {
        'criterio_o_observacion_S2': 'Robustez: controlar Location y la dependencia espacial o temporal',
        'respuesta_implementada_en_S3': (
            'Se incorporan errores robustos agrupados por Location, sensibilidad '
            'con variables indicadoras de localidad y una validación temporal '
            'sobre el tramo más reciente del entrenamiento.'
        ),
        'evidencia_computacional': '27f_sensibilidad_errores_robustos_cluster_Location.csv; 40b_sensibilidad_validacion_temporal.csv; 41b_interpretacion_sensibilidad_Location.csv',
        'verificacion': (
            f'cambios_inferenciales_cluster={cluster_changes}; '
            f'filas_sensibilidad_temporal={len(temporal_sensitivity)}'
        ),
        'cumple': bool(
            not cluster_robust_sensitivity.empty
            and not temporal_sensitivity.empty
            and not location_interpretation.empty
        )
    },
    {
        'criterio_o_observacion_S2': 'Documentación: cerrar la trazabilidad entre comentario, código y archivo generado',
        'respuesta_implementada_en_S3': (
            'Se generan controles específicos para cada observación, una matriz '
            'exhaustiva de respuesta, un control metodológico y un manifiesto '
            'de integridad con SHA-256.'
        ),
        'evidencia_computacional': '02h_control_cierre_retroalimentacion_S2.csv; 43a_matriz_respuesta_retroalimentacion_S2.csv; 43_control_metodologico_rubrica.csv; 44_control_integridad_salidas_sumativa3.csv',
        'verificacion': 'La ejecución se detiene si alguna observación o evidencia obligatoria queda incumplida.',
        'cumple': bool(
            feedback_core_files_complete
            and correction_control['cumple'].all()
            and mc_parameter_traceability['coincide_referencia_S2'].all()
            and iqr_comparison['reproduccion_exacta_conteos'].all()
            and multicollinearity_recalculated[
                'n_pares_validos_base_completa'
            ].gt(0).all()
        )
    }
])
feedback_response_matrix['estado'] = np.where(
    feedback_response_matrix['cumple'],
    'Cumplido',
    'Pendiente'
)
feedback_response_matrix.to_csv(
    TABLES / '43a_matriz_respuesta_retroalimentacion_S2.csv',
    index=False
)
if not feedback_response_matrix['cumple'].all():
    pending_feedback = feedback_response_matrix.loc[
        ~feedback_response_matrix['cumple'],
        'criterio_o_observacion_S2'
    ].tolist()
    raise AssertionError(
        'La matriz de respuesta mantiene observaciones pendientes: '
        f'{pending_feedback}'
    )

display(feedback_response_matrix)

required_evidence = {
    'bootstrap_Rainfall_asimetria': TABLES / '02a_bootstrap_Rainfall_asimetria.csv',
    'parametros_base_montecarlo_S1': TABLES / '02b0_parametros_base_montecarlo_S1.csv',
    'sensibilidad_montecarlo_regenerada': TABLES / '02b_sensibilidad_montecarlo_regenerada.csv',
    'consistencia_montecarlo_S2_S3': TABLES / '02c_consistencia_sensibilidad_montecarlo_S2_S3.csv',
    'convergencia_montecarlo_regenerada': TABLES / '02d_convergencia_montecarlo_regenerada.csv',
    'diagnostico_IQR_regenerado': TABLES / '02e_diagnostico_outliers_IQR_regenerado.csv',
    'consistencia_IQR_S2_S3': TABLES / '02f_consistencia_diagnostico_IQR_S2_S3.csv',
    'multicolinealidad_recalculada': TABLES / '02g_recalculo_multicolinealidad_base_completa.csv',
    'cierre_retroalimentacion_S2': TABLES / '02h_control_cierre_retroalimentacion_S2.csv',
    'faltantes_S1': TABLES / '03a_faltantes_S1_y_alcance_S3.csv',
    'inventario_numerico': TABLES / '03b_inventario_variables_numericas_y_decision.csv',
    'mecanismo_faltantes': TABLES / '03c_diagnostico_complementario_mecanismo_faltantes.csv',
    'cobertura_total_faltantes': TABLES / '03f_control_cobertura_total_variables_faltantes.csv',
    'predictores_imputacion': TABLES / '05_predictores_regresiones_imputacion.csv',
    'trazabilidad_matriz_S1': TABLES / '05a_control_trazabilidad_matriz_correlacion_S1.csv',
    'regresiones_imputacion': TABLES / '06a_desempeno_y_supuestos_regresiones_imputacion.csv',
    'independencia_imputacion': TABLES / '06c_independencia_residuos_regresiones_imputacion.csv',
    'evaluacion_explicita_supuestos': TABLES / '06d_evaluacion_explicita_supuestos_regresiones_imputacion.csv',
    'cantidad_imputada': TABLES / '08a_cantidad_valores_imputados.csv',
    'postprocesamiento_imputacion': TABLES / '08e_postprocesamiento_predicciones_imputacion.csv',
    'comparacion_imputaciones': TABLES / '10b_comparacion_integrada_imputacion.csv',
    'preservacion_correlaciones_S1': TABLES / '11_preservacion_correlaciones_train.csv',
    'seleccion_imputacion': TABLES / '12_sintesis_seleccion_estrategia_imputacion.csv',
    'sensibilidad_ponderaciones_imputacion': TABLES / '12b_sensibilidad_ponderaciones_seleccion_imputacion.csv',
    'trazabilidad_outliers': TABLES / '13a_trazabilidad_outliers_S1_S2.csv',
    'delimitacion_predictoras_logisticas': TABLES / '13e_delimitacion_predictoras_modelos_logisticos.csv',
    'stepwise_AIC': TABLES / '14_historial_stepwise_AIC.csv',
    'BIC_exhaustivo': TABLES / '15_busqueda_exhaustiva_BIC.csv',
    'coeficientes_tres_modelos': TABLES / '17_coeficientes_odds_ratios_tres_modelos.csv',
    'validacion_anidada': TABLES / '19a_metricas_validacion_anidada_por_fold.csv',
    'seleccion_variables_anidada': TABLES / '19d_variables_seleccionadas_en_cada_fold.csv',
    'metricas_tres_modelos': TABLES / '20_metricas_train_test_tres_modelos_umbral_050.csv',
    'comparacion_inferencia_prediccion': TABLES / '20b_comparacion_inferencia_y_prediccion_balanceada.csv',
    'seleccion_predictiva_OOF': TABLES / '23b_seleccion_configuracion_predictiva_OOF.csv',
    'calibracion_predictiva': TABLES / '26d_calibracion_configuraciones_predictivas.csv',
    'resumen_calibracion_predictiva': TABLES / '26f_resumen_calibracion_predictiva.csv',
    'decision_inferencial_predictiva': TABLES / '26e_decision_modelo_inferencial_y_predictivo.csv',
    'matrices_confusion': TABLES / '21_matrices_confusion_train_test_tres_modelos.csv',
    'VIF': TABLES / '27_VIF_modelo_final.csv',
    'resolucion_multicolinealidad': TABLES / '27b_resolucion_multicolinealidad_modelo_final.csv',
    'decision_especificacion_definitiva': TABLES / '27d_decision_especificacion_definitiva.csv',
    'suficiencia_y_separacion': TABLES / '27e_suficiencia_muestral_y_separacion.csv',
    'robusto_cluster_Location': TABLES / '27f_sensibilidad_errores_robustos_cluster_Location.csv',
    'residuos_influencia': TABLES / '29_resumen_residuos_influencia.csv',
    'control_bootstrap': TABLES / '31c_control_convergencia_bootstrap.csv',
    'estabilidad_acumulada_bootstrap': TABLES / '31d_estabilidad_acumulada_bootstrap.csv',
    'bootstrap': TABLES / '32_estabilidad_coeficientes_odds_ratios_bootstrap.csv',
    'interpretacion_efectos': TABLES / '32b_interpretacion_final_efectos_y_estabilidad.csv',
    'simetria_impacto_imputacion': TABLES / '33a_control_simetria_estrategias_imputacion.csv',
    'impacto_imputacion': TABLES / '35_desempeno_modelo_final_por_imputacion.csv',
    'interpretacion_impacto_imputacion': TABLES / '37c_interpretacion_sensibilidad_imputacion.csv',
    'sensibilidad_temporal': TABLES / '40b_sensibilidad_validacion_temporal.csv',
    'interpretacion_Location': TABLES / '41b_interpretacion_sensibilidad_Location.csv',
    'sintesis': TABLES / '42_sintesis_tecnica_resultados.csv',
    'matriz_respuesta_retroalimentacion_S2': TABLES / '43a_matriz_respuesta_retroalimentacion_S2.csv',
}

control_rows = []
for criterion, path in required_evidence.items():
    control_rows.append({
        'criterio': criterion,
        'archivo': str(path.relative_to(ROOT)),
        'existe': path.exists(),
        'tamano_bytes': path.stat().st_size if path.exists() else 0
    })
methodological_control = pd.DataFrame(control_rows)
methodological_control.to_csv(
    TABLES / '43_control_metodologico_rubrica.csv',
    index=False
)
if not methodological_control['existe'].all():
    missing_evidence = methodological_control.loc[
        ~methodological_control['existe'],
        'archivo'
    ].tolist()
    raise AssertionError(
        f'El control metodológico detectó evidencias faltantes: {missing_evidence}'
    )
if not (methodological_control['tamano_bytes'] > 0).all():
    empty_evidence = methodological_control.loc[
        methodological_control['tamano_bytes'].le(0),
        'archivo'
    ].tolist()
    raise AssertionError(
        f'El control metodológico detectó evidencias vacías: {empty_evidence}'
    )

if not feedback_response_matrix['cumple'].all():
    raise AssertionError(
        'La matriz exhaustiva de retroalimentación S2 contiene observaciones pendientes.'
    )

if not correction_control['cumple'].all():
    raise AssertionError(
        'El control final detectó observaciones de S2 sin cierre reproducible.'
    )
if N_BOOT_RAINFALL < 10_000:
    raise AssertionError(
        'La profundización bootstrap de Rainfall requiere al menos 10.000 remuestras.'
    )
if N_MC_S2_CLOSURE < 10_000:
    raise AssertionError(
        'La sensibilidad Monte Carlo requiere al menos 10.000 iteraciones.'
    )
if not mc_parameter_traceability['coincide_referencia_S2'].all():
    raise AssertionError(
        'Los parámetros base de Monte Carlo no reproducen la evidencia validada de S2.'
    )
if not np.isclose(
    mc_parameters_base['proporcion_yes'],
    0.22418121848473554,
    rtol=0,
    atol=1e-12
):
    raise AssertionError(
        'La prevalencia Monte Carlo no reproduce la proporción validada en S1.'
    )
if N_MC_PORTFOLIOS < 10_000:
    raise AssertionError(
        'La distribución agregada requiere al menos 10.000 carteras simuladas.'
    )
if strategy_summary['seleccion_robusta_en_todos_los_escenarios'].sum() != 1:
    raise AssertionError(
        'La síntesis de imputación debe marcar como robusta únicamente la estrategia seleccionada.'
    )
if not strategy_summary.loc[
    strategy_summary['seleccionada'],
    'seleccion_robusta_en_todos_los_escenarios'
].all():
    raise AssertionError(
        'La estrategia seleccionada no quedó confirmada en todos los escenarios de ponderación.'
    )
if len(montecarlo_sensitivity_regenerated) != 9:
    raise AssertionError(
        'La sensibilidad Monte Carlo debe contener el escenario base y ocho perturbaciones.'
    )
if not mc_consistency['consistencia_numerica'].all():
    raise AssertionError(
        'La sensibilidad Monte Carlo regenerada no es consistente con la evidencia heredada de S2.'
    )
if not (
    iqr_comparison['reproduccion_exacta_conteos'].all()
    and iqr_comparison['reproduccion_porcentaje_tolerancia'].all()
):
    raise AssertionError(
        'El diagnóstico IQR regenerado no reproduce la evidencia heredada de S2.'
    )
if multicollinearity_recalculated[
    'n_pares_validos_base_completa'
].le(0).any():
    raise AssertionError(
        'El recálculo de multicolinealidad contiene pares sin observaciones válidas.'
    )

expected_nested_rows = N_NESTED_CV * 3
if len(nested_cv_metrics) != expected_nested_rows:
    raise AssertionError(
        f'La validación anidada generó {len(nested_cv_metrics)} filas; '
        f'se esperaban {expected_nested_rows}.'
    )
if set(nested_selections['modelo']) != {
    'M1_S1_S2',
    'M2_stepwise_AIC',
    'M3_BIC_parsimonioso'
}:
    raise AssertionError(
        'La validación anidada no evaluó los tres procedimientos.'
    )
if len(impact_symmetry_control) != 3:
    raise AssertionError(
        'El control de simetría no contiene las tres estrategias.'
    )
if not impact_symmetry_control['alcance_simetrico_confirmado'].all():
    raise AssertionError(
        'El análisis de impacto no mantiene un alcance simétrico.'
    )
if not impact_symmetry_control[
    'n_variables_numericas_tratadas'
].eq(N_IMPACT_NUMERIC).all():
    raise AssertionError(
        f'Las tres estrategias no trataron las mismas {N_IMPACT_NUMERIC} variables numéricas.'
    )
if not impact_symmetry_control[
    'n_variables_totales_alcance'
].eq(N_IMPACT_ALL).all():
    raise AssertionError(
        f'El alcance debe contener {N_IMPACT_NUMERIC} variables numéricas y RainToday_bin.'
    )
if impact_symmetry_control[
    [
        'faltantes_residuales_train_alcance',
        'faltantes_residuales_test_alcance'
    ]
].to_numpy().sum() != 0:
    raise AssertionError(
        'Existen faltantes residuales en el análisis simétrico de imputación.'
    )
if len(impact_sizes) != 3:
    raise AssertionError(
        'La comparación de impacto no contiene las tres estrategias.'
    )

if int(scope_coverage_control.loc[
    scope_coverage_control['dimension'].eq(
        'Variables numéricas faltantes sin decisión'
    ),
    'cantidad'
].iloc[0]) != 0:
    raise AssertionError(
        'El control de alcance detectó variables faltantes sin decisión.'
    )
if len(predictor_scope) != len(S1_CORE_NUMERIC):
    raise AssertionError(
        'La delimitación logística no contabiliza las diez variables de S1.'
    )
if set(predictor_scope['variable_S1']) != set(S1_CORE_NUMERIC):
    raise AssertionError(
        'La tabla de delimitación logística no coincide con la matriz de S1.'
    )

if len(performance_three_models) != 6:
    raise AssertionError(
        'No se generaron métricas train/test para los tres modelos.'
    )
if len(confusion_three_models) != 6:
    raise AssertionError('No se generaron seis matrices de confusión.')
if final_test_metrics.empty or len(final_test_metrics) != 1:
    raise AssertionError(
        'La evaluación final no contiene exactamente la configuración seleccionada.'
    )
if N_BOOT != 10_000:
    raise AssertionError(
        f'El bootstrap debe utilizar 10.000 réplicas; se configuraron {N_BOOT}.'
    )
if len(predictive_configuration_oof) != 4:
    raise AssertionError(
        'La comparación OOF no contiene las cuatro configuraciones esperadas.'
    )
if predictive_configuration_oof['seleccionada'].sum() != 1:
    raise AssertionError(
        'La regla OOF no seleccionó exactamente una configuración predictiva.'
    )
if sample_separation_control['eventos_por_parametro'].iloc[0] < 10:
    raise AssertionError(
        'La razón de eventos por parámetro es inferior a 10.'
    )
if not bootstrap_control['cumple_tasa_minima'].all():
    raise AssertionError(
        'El bootstrap no alcanzó la tasa mínima de éxito.'
    )
if not information_criteria['convergio'].all():
    raise AssertionError(
        'Al menos uno de los tres modelos no convergió.'
    )
if final_effect_interpretation.empty:
    raise AssertionError(
        'No se generó la interpretación final de efectos.'
    )
if FINAL_MAX_VIF > 10:
    raise AssertionError(
        f'La especificación definitiva mantiene VIF máximo={FINAL_MAX_VIF:.3f}, '
        'superior a 10.'
    )
if set(imputation_result['counts']['variable']) != set(ANALYTIC_VARIABLES):
    raise AssertionError(
        'El control de imputación no contiene todas las variables analíticas.'
    )

# Exportación de la base procesada conservando la identificación del split.
processed_train = selected_train.copy()
processed_train['split'] = 'train'
processed_test = selected_test.copy()
processed_test['split'] = 'test'
processed_data = pd.concat(
    [processed_train, processed_test],
    ignore_index=True
)
if processed_data[ANALYTIC_VARIABLES].isna().any().any():
    missing_processed = processed_data[ANALYTIC_VARIABLES].isna().sum()
    missing_processed = missing_processed[missing_processed.gt(0)].to_dict()
    raise AssertionError(
        f'La base procesada conserva faltantes analíticos: {missing_processed}'
    )
processed_path = (
    PROCESSED / 'weatherAUS_sumativa3_modelamiento.csv'
)
processed_data.to_csv(processed_path, index=False)

# Control de integridad completo: todas las tablas y figuras generadas,
# más la base procesada. El propio manifiesto 44 se excluye para evitar
# autorreferencia criptográfica.
generated_tables = sorted(
    path
    for path in TABLES.glob('*.csv')
    if path.name != '44_control_integridad_salidas_sumativa3.csv'
)
generated_figures = sorted(FIGURES.glob('*.png'))
files_for_integrity = [
    *generated_tables,
    *generated_figures,
    processed_path
]
if not generated_tables:
    raise AssertionError('No se generaron tablas de Semana 3.')
if not generated_figures:
    raise AssertionError('No se generaron figuras de Semana 3.')

integrity_rows = []
for path in files_for_integrity:
    if not path.exists() or path.stat().st_size <= 0:
        raise AssertionError(
            f'El artefacto no existe o está vacío: {path}'
        )
    if path.parent == TABLES:
        category = 'tabla'
    elif path.parent == FIGURES:
        category = 'figura'
    else:
        category = 'base_procesada'
    digest = hashlib.sha256(path.read_bytes()).hexdigest()
    integrity_rows.append({
        'categoria': category,
        'archivo': str(path.relative_to(ROOT)),
        'tamano_bytes': path.stat().st_size,
        'sha256': digest
    })

integrity = pd.DataFrame(integrity_rows)
if integrity['archivo'].duplicated().any():
    raise AssertionError(
        'El manifiesto de integridad contiene rutas duplicadas.'
    )
integrity.to_csv(
    TABLES / '44_control_integridad_salidas_sumativa3.csv',
    index=False
)

execution_status = pd.DataFrame([{
    'estado': 'Ejecución completa',
    'evidencias_metodologicas_verificadas': int(
        methodological_control['existe'].sum()
    ),
    'evidencias_metodologicas_requeridas': len(methodological_control),
    'tablas_auditadas_integridad': len(generated_tables),
    'figuras_auditadas_integridad': len(generated_figures),
    'bases_procesadas_auditadas': 1,
    'archivos_totales_auditados_integridad': len(integrity),
    'bootstrap_exitoso': bool(
        bootstrap_control['cumple_tasa_minima'].all()
    ),
    'VIF_maximo_final': FINAL_MAX_VIF,
    'base_procesada_generada': processed_path.exists(),
    'faltantes_analiticos_base_procesada': int(
        processed_data[ANALYTIC_VARIABLES].isna().sum().sum()
    )
}])
display(execution_status)

print(
    'Ejecución completa: controles metodológicos e integridad verificados; '
    'las observaciones heredadas de S2 fueron cerradas mediante evidencias '
    'reproducibles; todas las variables numéricas del conjunto analítico S1/S2 '
    'fueron imputadas; base procesada generada correctamente.'
)


,criterio_o_observacion_S2,respuesta_implementada_en_S3,evidencia_computacional,verificacion,cumple,estado
0,Bootstrap: incorporar un parámetro con asimetr...,Se analiza la media de Rainfall mediante 10.00...,02a_bootstrap_Rainfall_asimetria.csv; fig_00a_...,n_boot=10000; asimetría=9.8362; IC_BCa=[2.3177...,True,Cumplido
1,Permutación: conservar la relación validada y ...,La prueba de permutación se incorpora como ant...,01_trazabilidad_S1_S2_hacia_S3.csv; 13c_trazab...,Antecedentes de S2 cargados y vinculados con d...,True,Cumplido
2,"Correlaciones: recalcular los pares 0,96 y 0,8...",Se recalculan los pares Pressure9am–Pressure3p...,02g_recalculo_multicolinealidad_base_completa....,Pressure=0.9613; Temperatura=0.8606,True,Cumplido
3,Monte Carlo: incorporar al notebook maestro la...,Se regenera el escenario base y ocho perturbac...,02b_sensibilidad_montecarlo_regenerada.csv; 02...,n_iteraciones=100000; n_escenarios=9,True,Cumplido
4,Monte Carlo: complementar el delta individual ...,"Se incorporan tasas operacionales, resultados ...",02b_sensibilidad_montecarlo_regenerada.csv,n_carteras=10000; tamaño_cartera=1000; VPP_bas...,True,Cumplido
5,Robustez: regenerar el diagnóstico IQR citado ...,El diagnóstico IQR se reproduce desde la base ...,02e_diagnostico_outliers_IQR_regenerado.csv; 0...,eliminados_total=219,True,Cumplido
6,Modelamiento: priorizar Humidity3pm con fundam...,Humidity3pm se incluye en el modelo basado en ...,13c_trazabilidad_variables_candidatas_S1_S2.cs...,Predictor incorporado desde la formulación ini...,True,Cumplido
7,Modelamiento: incorporar RainToday_bin por su ...,RainToday_bin se incorpora al modelo S1/S2 y a...,13c_trazabilidad_variables_candidatas_S1_S2.cs...,Variable evaluada explícitamente; decisión def...,True,Cumplido
8,Modelamiento: controlar redundancia mediante s...,Los pares altamente correlacionados se excluye...,13e_delimitacion_predictoras_modelos_logistico...,VIF_máximo_final=4.7897,True,Cumplido
9,Desbalance: comparar ponderación de clase y no...,Se comparan configuraciones ponderadas y no po...,20b_comparacion_inferencia_y_prediccion_balanc...,configuraciones_OOF=4; seleccionadas=1,True,Cumplido


,estado,evidencias_metodologicas_verificadas,evidencias_metodologicas_requeridas,tablas_auditadas_integridad,figuras_auditadas_integridad,bases_procesadas_auditadas,archivos_totales_auditados_integridad,bootstrap_exitoso,VIF_maximo_final,base_procesada_generada,faltantes_analiticos_base_procesada
0,Ejecución completa,55,55,106,24,1,131,True,4.789744,True,0


Ejecución completa: controles metodológicos e integridad verificados; las observaciones heredadas de S2 fueron cerradas mediante evidencias reproducibles; todas las variables numéricas del conjunto analítico S1/S2 fueron imputadas; base procesada generada correctamente.
